In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

# Your files will then be accessible under '/content/drive/MyDrive/'

Mounted at /content/drive/


In [ ]:
# ============================================================
# HOSPITALIZATION FORECASTING MODEL
# Step H0 - Configuration / Pre-Setup
# TRAIN = SynPUF Sets 1 + 2
# TEST  = SynPUF Set 3
# ============================================================

print("\nINITIALIZING HOSPITALIZATION FORECASTING MODEL")

# ============================================================
# Core Libraries
# ============================================================

import os
import gc
import warnings
import random

import numpy as np
import pandas as pd

!pip install openpyxl -q

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")

# ============================================================
# Random Seeds
# ============================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ============================================================
# Folder Structure
# ============================================================

DATA_FOLDER = "/content/drive/MyDrive/Colab Notebooks/"

INPUT_FOLDER = os.path.join(
    DATA_FOLDER,
    "Medicare Synthetic Model 2"
)

BASE_FOLDER = os.path.join(
    DATA_FOLDER,
    "Hospitalization_Forecasting_Model"
)

OUTPUT_FOLDER = os.path.join(
    BASE_FOLDER,
    "Output"
)

TEMP_FOLDER = os.path.join(
    BASE_FOLDER,
    "Temp"
)

os.makedirs(BASE_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(TEMP_FOLDER, exist_ok=True)

print("\nFolder structure initialized.")
print("INPUT_FOLDER:", INPUT_FOLDER)
print("OUTPUT_FOLDER:", OUTPUT_FOLDER)
print("TEMP_FOLDER:", TEMP_FOLDER)

# ============================================================
# External Validation Controls
# ============================================================

USE_EXTERNAL_VALIDATION = True

TRAIN_SAMPLES = [1, 2]
TEST_SAMPLE = 3

TRAIN_YEARS = [2008, 2009]
TEST_YEAR = 2010

# ============================================================
# Timeline Controls
# ============================================================

TIMELINE_FREQUENCY = "MS"
TIMELINE_START_DATE = "2008-01-01"
TIMELINE_END_DATE = "2010-12-01"

LOOKBACK_WINDOWS_DAYS = [30, 90, 180, 365]
PREDICTION_WINDOWS_DAYS = [30, 90, 180]

PRIMARY_PREDICTION_WINDOW_DAYS = 90

TARGET_LABEL_NAME = (
    f"hospitalized_next_{PRIMARY_PREDICTION_WINDOW_DAYS}d"
)

PREDICTION_DATE_COL = "prediction_month"

# ============================================================
# File Builders
# ============================================================

def pde_file(sample):
    return os.path.join(
        INPUT_FOLDER,
        f"DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_{sample}.csv"
    )


def inpatient_file(sample):
    return os.path.join(
        INPUT_FOLDER,
        f"DE1_0_2008_to_2010_Inpatient_Claims_Sample_{sample}.csv"
    )


def outpatient_file(sample):
    return os.path.join(
        INPUT_FOLDER,
        f"DE1_0_2008_to_2010_Outpatient_Claims_Sample_{sample}.csv"
    )


def beneficiary_file(sample, year):
    return os.path.join(
        INPUT_FOLDER,
        f"DE1_0_{year}_Beneficiary_Summary_File_Sample_{sample}.csv"
    )


def carrier_files(sample):
    return [
        os.path.join(
            INPUT_FOLDER,
            f"DE1_0_2008_to_2010_Carrier_Claims_Sample_{sample}A.csv"
        ),
        os.path.join(
            INPUT_FOLDER,
            f"DE1_0_2008_to_2010_Carrier_Claims_Sample_{sample}B.csv"
        )
    ]


# Optional professional patient profile created by professional Step 32
PROFESSIONAL_MODEL_RISK_FILE = os.path.join(
    INPUT_FOLDER,
    "professional_model_patient_risk_scores.csv"
)

REQUIRE_PROFESSIONAL_MODEL_FILE = False

# ============================================================
# File Validation Printout
# ============================================================

print("\nSource file existence check:")

for sample in TRAIN_SAMPLES:
    print(f"TRAIN PDE Sample {sample}:", os.path.exists(pde_file(sample)))
    print(f"TRAIN Inpatient Sample {sample}:", os.path.exists(inpatient_file(sample)))
    print(f"TRAIN Outpatient Sample {sample}:", os.path.exists(outpatient_file(sample)))

    for f in carrier_files(sample):
        print(f"TRAIN Carrier Sample {sample}:", f, os.path.exists(f))

    for yr in [2008, 2009, 2010]:
        print(
            f"TRAIN Beneficiary Sample {sample} Year {yr}:",
            os.path.exists(beneficiary_file(sample, yr))
        )

print(f"TEST PDE Sample {TEST_SAMPLE}:", os.path.exists(pde_file(TEST_SAMPLE)))
print(f"TEST Inpatient Sample {TEST_SAMPLE}:", os.path.exists(inpatient_file(TEST_SAMPLE)))
print(f"TEST Outpatient Sample {TEST_SAMPLE}:", os.path.exists(outpatient_file(TEST_SAMPLE)))

for f in carrier_files(TEST_SAMPLE):
    print(f"TEST Carrier Sample {TEST_SAMPLE}:", f, os.path.exists(f))

for yr in [2008, 2009, 2010]:
    print(
        f"TEST Beneficiary Sample {TEST_SAMPLE} Year {yr}:",
        os.path.exists(beneficiary_file(TEST_SAMPLE, yr))
    )

print("PROFESSIONAL_MODEL_RISK_FILE:", os.path.exists(PROFESSIONAL_MODEL_RISK_FILE))

# ============================================================
# Memory / Processing Controls
# ============================================================

LOW_MEMORY_MODE = True
CHUNK_SIZE = 250000
MAX_TIMELINE_ROWS = 3000000

ENABLE_MEMBER_SAMPLING = False
SAMPLE_MEMBER_COUNT = 100000

# ============================================================
# Rx / PDE Field Mapping
# ============================================================

PDE_FIELD_MAP = {
    "member_id": "DESYNPUF_ID",
    "pde_id": "PDE_ID",
    "rx_service_date": "SRVC_DT",
    "drug_code": "PROD_SRVC_ID",
    "quantity_dispensed": "QTY_DSPNSD_NUM",
    "days_supply": "DAYS_SUPLY_NUM",
    "patient_pay_amount": "PTNT_PAY_AMT",
    "total_rx_cost": "TOT_RX_CST_AMT"
}

# ============================================================
# Inpatient Field Mapping
# ============================================================

INPATIENT_FIELD_MAP = {
    "member_id": "DESYNPUF_ID",
    "claim_id": "CLM_ID",
    "claim_from_date": "CLM_FROM_DT",
    "claim_thru_date": "CLM_THRU_DT",
    "facility_id": "PRVDR_NUM",
    "paid_amount": "CLM_PMT_AMT",
    "admission_date": "CLM_ADMSN_DT",
    "admitting_diagnosis": "ADMTNG_ICD9_DGNS_CD",
    "utilization_days": "CLM_UTLZTN_DAY_CNT",
    "discharge_date": "NCH_BENE_DSCHRG_DT",
    "drg_code": "CLM_DRG_CD"
}

# ============================================================
# Beneficiary Field Controls
# ============================================================

CHRONIC_CONDITION_COLUMNS = [
    "SP_ALZHDMTA",
    "SP_CHF",
    "SP_CHRNKIDN",
    "SP_CNCR",
    "SP_COPD",
    "SP_DEPRESSN",
    "SP_DIABETES",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_STRKETIA"
]

# ============================================================
# Feature Toggles
# ============================================================

USE_RX_FEATURES = True
USE_INPATIENT_HISTORY_FEATURES = True
USE_OUTPATIENT_HISTORY_FEATURES = True
USE_PROFESSIONAL_HISTORY_FEATURES = True
USE_BENEFICIARY_FEATURES = True

USE_RX_COST_FEATURES = True
USE_RX_DRUG_DIVERSITY_FEATURES = True
USE_RX_GROWTH_FEATURES = True

# ============================================================
# Hospitalization Forecasting Model Parameters
# ============================================================

HOSPITALIZATION_MODEL_PARAMS = {
    "learning_rate": 0.05,
    "max_depth": 6,
    "max_iter": 250,
    "min_samples_leaf": 75,
    "l2_regularization": 1.0,
    "random_state": RANDOM_STATE
}

# ============================================================
# Operational Thresholds
# ============================================================

THRESHOLD_GRID = np.round(
    np.arange(0.05, 1.00, 0.05),
    2
)

HIGH_RISK_THRESHOLD = 0.70
CARE_MANAGEMENT_THRESHOLD = 0.50
WATCHLIST_THRESHOLD = 0.30

FALSE_NEGATIVE_RATE_LIMIT = 0.10
CARE_MANAGEMENT_WORKLOAD_LIMIT = 0.30

# ============================================================
# Leakage Governance
# ============================================================

FEATURE_CUTOFF_DATE_ENFORCED = True

LEAKAGE_COLUMNS = [
    TARGET_LABEL_NAME,
    "future_hospitalization_flag",
    "future_ip_claim_count",
    "future_admission_date",
    "future_discharge_date",
    "future_paid_amount",
    "future_los",
    "post_prediction_claims",
    "post_prediction_rx"
]

# ============================================================
# Output Files
# ============================================================

TRAIN_TIMELINE_FILE = os.path.join(
    OUTPUT_FOLDER,
    "train_patient_month_timeline_sets1_2.csv"
)

TEST_TIMELINE_FILE = os.path.join(
    OUTPUT_FOLDER,
    "test_patient_month_timeline_set3.csv"
)

TRAIN_MODEL_READY_FILE = os.path.join(
    OUTPUT_FOLDER,
    "train_hospitalization_model_ready_sets1_2.csv"
)

TEST_MODEL_READY_FILE = os.path.join(
    OUTPUT_FOLDER,
    "test_hospitalization_model_ready_set3.csv"
)

# ============================================================
# Helper Functions
# ============================================================

def clean_id(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if value.endswith(".0"):
        value = value[:-2]
    try:
        if "e+" in value.lower() or "e-" in value.lower():
            value = str(int(float(value)))
    except Exception:
        pass
    return value


def safe_numeric(series, fill_value=0):
    return (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(fill_value)
    )


def safe_date(series):
    return pd.to_datetime(
        series,
        format="%Y%m%d",
        errors="coerce"
    )


def file_exists_and_not_empty(path):
    return os.path.exists(path) and os.path.getsize(path) > 0


def print_h0_summary():

    print("\n================================================")
    print("HOSPITALIZATION FORECASTING MODEL CONFIG LOADED")
    print("================================================")

    print("\nPrediction Target:")
    print("TARGET_LABEL_NAME:", TARGET_LABEL_NAME)
    print("PRIMARY_PREDICTION_WINDOW_DAYS:", PRIMARY_PREDICTION_WINDOW_DAYS)

    print("\nArchitecture:")
    print("1. Patient-month timeline")
    print("2. Rx/PDE progression features")
    print("3. Professional/outpatient history features")
    print("4. Prior inpatient history")
    print("5. Future hospitalization label")

    print("\nExternal Validation:")
    print("TRAIN_SAMPLES:", TRAIN_SAMPLES)
    print("TEST_SAMPLE:", TEST_SAMPLE)

    print("\nLookback Windows:")
    print(LOOKBACK_WINDOWS_DAYS)

    print("\nPrediction Windows:")
    print(PREDICTION_WINDOWS_DAYS)

    print("\nFeature Toggles:")
    print("USE_RX_FEATURES:", USE_RX_FEATURES)
    print("USE_INPATIENT_HISTORY_FEATURES:", USE_INPATIENT_HISTORY_FEATURES)
    print("USE_OUTPATIENT_HISTORY_FEATURES:", USE_OUTPATIENT_HISTORY_FEATURES)
    print("USE_PROFESSIONAL_HISTORY_FEATURES:", USE_PROFESSIONAL_HISTORY_FEATURES)


print_h0_summary()

print("\nStep H0 completed successfully.")


INITIALIZING HOSPITALIZATION FORECASTING MODEL

Folder structure initialized.
INPUT_FOLDER: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2
OUTPUT_FOLDER: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Output
TEMP_FOLDER: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Temp

Source file existence check:
TRAIN PDE Sample 1: True
TRAIN Inpatient Sample 1: True
TRAIN Outpatient Sample 1: True
TRAIN Carrier Sample 1: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Carrier_Claims_Sample_1A.csv True
TRAIN Carrier Sample 1: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Carrier_Claims_Sample_1B.csv True
TRAIN Beneficiary Sample 1 Year 2008: True
TRAIN Beneficiary Sample 1 Year 2009: True
TRAIN Beneficiary Sample 1 Year 2010: True
TRAIN PDE Sample 2: True
TRAIN Inpatient Sample 2: True
TRAIN Outpatient Sample 2: True
TRAIN Carrier Sample 2: /content/d

In [ ]:
# ============================================================
# Step H1 - FAST Rx/PDE Timeline Features
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H1 - FAST Rx / PDE timeline features")

def pde_file(sample):
    return os.path.join(
        INPUT_FOLDER,
        f"DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_{sample}.csv"
    )

def load_pde_files(samples, dataset_label):
    parts = []

    for sample in samples:
        path = pde_file(sample)
        print(f"\nLoading PDE {dataset_label} Sample {sample}: {path}")

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        df = pd.read_csv(path, dtype=str, low_memory=False)
        df["source_sample"] = sample

        rename_map = {v: k for k, v in PDE_FIELD_MAP.items()}
        df = df.rename(columns=rename_map)

        df["member_id"] = df["member_id"].apply(clean_id)
        df["pde_id"] = df["pde_id"].apply(clean_id)
        df["drug_code"] = df["drug_code"].astype(str).str.strip()
        df["rx_service_date"] = safe_date(df["rx_service_date"])

        for col in ["quantity_dispensed", "days_supply", "patient_pay_amount", "total_rx_cost"]:
            df[col] = safe_numeric(df[col], fill_value=0)

        df = df.dropna(subset=["rx_service_date"])
        df = df[df["member_id"] != ""]

        df["rx_month_start"] = df["rx_service_date"].values.astype("datetime64[M]")

        parts.append(df)

    out = pd.concat(parts, ignore_index=True)

    print(f"\n{dataset_label} PDE shape:", out.shape)
    print(out["source_sample"].value_counts())

    return out


def build_fast_rx_timeline(rx_df, dataset_label):

    print(f"\nBuilding FAST Rx timeline for {dataset_label}...")

    rx_monthly = (
        rx_df
        .groupby(["member_id", "rx_month_start"])
        .agg(
            rx_claims_month=("pde_id", "nunique"),
            unique_drugs_month=("drug_code", "nunique"),
            rx_days_supply_month=("days_supply", "sum"),
            rx_quantity_month=("quantity_dispensed", "sum"),
            rx_total_cost_month=("total_rx_cost", "sum"),
            rx_patient_pay_month=("patient_pay_amount", "sum")
        )
        .reset_index()
    )

    members = rx_df[["member_id"]].drop_duplicates()

    months = pd.DataFrame({
        PREDICTION_DATE_COL: pd.date_range(
            start=TIMELINE_START_DATE,
            end=TIMELINE_END_DATE,
            freq=TIMELINE_FREQUENCY
        )
    })

    members["key"] = 1
    months["key"] = 1

    timeline = (
        members
        .merge(months, on="key")
        .drop(columns="key")
    )

    timeline = timeline.merge(
        rx_monthly,
        left_on=["member_id", PREDICTION_DATE_COL],
        right_on=["member_id", "rx_month_start"],
        how="left"
    )

    timeline = timeline.drop(columns=["rx_month_start"], errors="ignore")

    monthly_cols = [
        "rx_claims_month",
        "unique_drugs_month",
        "rx_days_supply_month",
        "rx_quantity_month",
        "rx_total_cost_month",
        "rx_patient_pay_month"
    ]

    for col in monthly_cols:
        timeline[col] = safe_numeric(timeline[col], fill_value=0)

    timeline = timeline.sort_values(["member_id", PREDICTION_DATE_COL])

    # Approximate day windows using month windows:
    # 30d = 1 month, 90d = 3 months, 180d = 6 months, 365d = 12 months
    window_map = {
        30: 1,
        90: 3,
        180: 6,
        365: 12
    }

    rolling_base_cols = {
        "rx_claims": "rx_claims_month",
        "unique_drugs": "unique_drugs_month",
        "rx_days_supply": "rx_days_supply_month",
        "rx_quantity": "rx_quantity_month",
        "rx_total_cost": "rx_total_cost_month",
        "rx_patient_pay": "rx_patient_pay_month"
    }

    for days, months_back in window_map.items():

        for prefix, base_col in rolling_base_cols.items():

            new_col = f"{prefix}_{days}d"

            timeline[new_col] = (
                timeline
                .groupby("member_id")[base_col]
                .transform(
                    lambda x: x.shift(1).rolling(
                        window=months_back,
                        min_periods=1
                    ).sum()
                )
            )

    # Derived features
    timeline["rx_claim_growth_90_vs_180"] = (
        timeline["rx_claims_90d"] /
        timeline["rx_claims_180d"].replace(0, np.nan)
    )

    timeline["rx_claim_growth_30_vs_90"] = (
        timeline["rx_claims_30d"] /
        timeline["rx_claims_90d"].replace(0, np.nan)
    )

    timeline["drug_growth_90_vs_180"] = (
        timeline["unique_drugs_90d"] /
        timeline["unique_drugs_180d"].replace(0, np.nan)
    )

    timeline["avg_days_supply_per_rx_90d"] = (
        timeline["rx_days_supply_90d"] /
        timeline["rx_claims_90d"].replace(0, np.nan)
    )

    timeline["avg_rx_cost_per_rx_90d"] = (
        timeline["rx_total_cost_90d"] /
        timeline["rx_claims_90d"].replace(0, np.nan)
    )

    timeline["polypharmacy_score_90d"] = timeline["unique_drugs_90d"]

    timeline["high_polypharmacy_flag_90d"] = (
        timeline["unique_drugs_90d"] >= 5
    ).astype(int)

    rx_feature_cols = [
        c for c in timeline.columns
        if c.startswith("rx_")
        or c.startswith("unique_drugs_")
        or c.startswith("drug_growth_")
        or c.startswith("avg_")
        or c.startswith("polypharmacy_")
        or c.startswith("high_polypharmacy_")
    ]

    for col in rx_feature_cols:
        timeline[col] = (
            safe_numeric(timeline[col], fill_value=0)
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
        )

    print(f"{dataset_label} FAST Rx timeline created.")
    print("Shape:", timeline.shape)
    print("Rx feature count:", len(rx_feature_cols))

    return timeline, rx_feature_cols


rx_train = load_pde_files(TRAIN_SAMPLES, "TRAIN")
rx_test = load_pde_files([TEST_SAMPLE], "TEST")

df_train_timeline, rx_feature_cols = build_fast_rx_timeline(
    rx_train,
    "TRAIN Sets 1+2"
)

df_test_timeline, _ = build_fast_rx_timeline(
    rx_test,
    "TEST Set 3"
)

df_train_timeline.to_csv(
    os.path.join(OUTPUT_FOLDER, "stepH1_train_sets1_2_rx_timeline_features.csv"),
    index=False
)

df_test_timeline.to_csv(
    os.path.join(OUTPUT_FOLDER, "stepH1_test_set3_rx_timeline_features.csv"),
    index=False
)

print("\nStep H1 completed successfully.")
print("df_train_timeline:", df_train_timeline.shape)
print("df_test_timeline:", df_test_timeline.shape)


STEP H1 - FAST Rx / PDE timeline features

Loading PDE TRAIN Sample 1: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv

Loading PDE TRAIN Sample 2: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_2.csv

TRAIN PDE shape: (11113575, 10)
source_sample
2    5561154
1    5552421
Name: count, dtype: int64

Loading PDE TEST Sample 3: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_3.csv

TEST PDE shape: (5557147, 10)
source_sample
3    5557147
Name: count, dtype: int64

Building FAST Rx timeline for TRAIN Sets 1+2...
TRAIN Sets 1+2 FAST Rx timeline created.
Shape: (7174224, 39)
Rx feature count: 37

Building FAST Rx timeline for TEST Set 3...
TEST Set 3 FAST Rx timeline created.
Shape: (3587652, 39)
Rx feature count: 37

Step H1 completed successfully.
df_train_timeline: (7174224, 3

In [ ]:
# ============================================================
# Step H2 - FAST Patient Timeline + Hospitalization Labels
# TRAIN Sets 1+2, TEST Set 3
# Uses monthly aggregation + rolling windows
# ============================================================

print("\nSTEP H2 - FAST patient timeline and hospitalization labels")

# ------------------------------------------------------------
# Load inpatient events
# ------------------------------------------------------------

def load_inpatient_events(samples, dataset_label):

    all_parts = []

    for sample in samples:

        path = inpatient_file(sample)

        print(f"\nLoading inpatient {dataset_label} Sample {sample}: {path}")

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        ip = pd.read_csv(path, dtype=str, low_memory=False)
        ip["source_sample"] = sample

        rename_map = {v: k for k, v in INPATIENT_FIELD_MAP.items()}
        ip = ip.rename(columns=rename_map)

        ip["member_id"] = ip["member_id"].apply(clean_id)
        ip["claim_id"] = ip["claim_id"].apply(clean_id)

        ip["admission_date"] = safe_date(ip["admission_date"])
        ip["discharge_date"] = safe_date(ip["discharge_date"])
        ip["claim_from_date"] = safe_date(ip["claim_from_date"])
        ip["claim_thru_date"] = safe_date(ip["claim_thru_date"])

        ip["length_of_stay"] = safe_numeric(
            ip["utilization_days"],
            fill_value=0
        )

        ip["drg_code"] = (
            ip["drg_code"]
            .astype(str)
            .str.strip()
            .replace(["", "nan", "None", "NaN"], "UNKNOWN")
        )

        ip["admitting_diagnosis"] = (
            ip["admitting_diagnosis"]
            .astype(str)
            .str.strip()
            .replace(["", "nan", "None", "NaN"], "UNKNOWN")
        )

        ip = ip.dropna(subset=["admission_date"])
        ip = ip[ip["member_id"] != ""]

        ip["admission_month_start"] = (
            ip["admission_date"]
            .values
            .astype("datetime64[M]")
        )

        ip["ip_event_key"] = (
            ip["source_sample"].astype(str)
            + "_"
            + ip["member_id"].astype(str)
            + "_"
            + ip["claim_id"].astype(str)
        )

        all_parts.append(ip)

    combined = pd.concat(all_parts, ignore_index=True)

    print(f"\n{dataset_label} combined inpatient shape:", combined.shape)
    print(
        f"{dataset_label} admission range:",
        combined["admission_date"].min(),
        "to",
        combined["admission_date"].max()
    )
    print(f"{dataset_label} source samples:")
    print(combined["source_sample"].value_counts())

    return combined


ip_train_events = load_inpatient_events(
    TRAIN_SAMPLES,
    "TRAIN"
)

ip_test_events = load_inpatient_events(
    [TEST_SAMPLE],
    "TEST"
)

# ------------------------------------------------------------
# Expand timeline to include members with IP but no Rx
# ------------------------------------------------------------

def expand_timeline_members_fast(timeline, ip_events, dataset_label):

    print(f"\nExpanding timeline members for {dataset_label}...")

    existing_members = timeline[["member_id"]].drop_duplicates()
    ip_members = ip_events[["member_id"]].drop_duplicates()

    all_members = (
        pd.concat([existing_members, ip_members], ignore_index=True)
        .drop_duplicates()
    )

    months = pd.DataFrame({
        PREDICTION_DATE_COL: pd.date_range(
            start=TIMELINE_START_DATE,
            end=TIMELINE_END_DATE,
            freq=TIMELINE_FREQUENCY
        )
    })

    all_members["key"] = 1
    months["key"] = 1

    expanded = (
        all_members
        .merge(months, on="key")
        .drop(columns=["key"])
    )

    expanded = expanded.merge(
        timeline,
        on=["member_id", PREDICTION_DATE_COL],
        how="left"
    )

    fill_cols = [
        c for c in expanded.columns
        if c.startswith("rx_")
        or c.startswith("unique_drugs_")
        or c.startswith("drug_growth_")
        or c.startswith("avg_")
        or c.startswith("polypharmacy_")
        or c.startswith("high_polypharmacy_")
    ]

    for col in fill_cols:
        expanded[col] = safe_numeric(expanded[col], fill_value=0)

    print(f"{dataset_label} expanded timeline shape:", expanded.shape)

    return expanded


df_train_timeline = expand_timeline_members_fast(
    df_train_timeline,
    ip_train_events,
    "TRAIN Sets 1+2"
)

df_test_timeline = expand_timeline_members_fast(
    df_test_timeline,
    ip_test_events,
    "TEST Set 3"
)

# ------------------------------------------------------------
# Build monthly IP feature table
# ------------------------------------------------------------

def build_ip_monthly(ip_events, dataset_label):

    print(f"\nBuilding monthly IP table for {dataset_label}...")

    ip_monthly = (
        ip_events
        .groupby(["member_id", "admission_month_start"])
        .agg(
            ip_claims_month=("ip_event_key", "nunique"),
            ip_los_month=("length_of_stay", "sum"),
            ip_unique_drg_month=("drg_code", "nunique")
        )
        .reset_index()
    )

    print(f"{dataset_label} monthly IP shape:", ip_monthly.shape)

    return ip_monthly


ip_train_monthly = build_ip_monthly(
    ip_train_events,
    "TRAIN Sets 1+2"
)

ip_test_monthly = build_ip_monthly(
    ip_test_events,
    "TEST Set 3"
)

# ------------------------------------------------------------
# Add prior IP history and future hospitalization labels fast
# ------------------------------------------------------------

def add_hospitalization_timeline_features_fast(timeline, ip_monthly, dataset_label):

    print(f"\nAdding FAST hospitalization timeline features for {dataset_label}...")

    timeline = timeline.copy()

    timeline[PREDICTION_DATE_COL] = pd.to_datetime(
        timeline[PREDICTION_DATE_COL],
        errors="coerce"
    )

    ip_monthly = ip_monthly.copy()

    ip_monthly["admission_month_start"] = pd.to_datetime(
        ip_monthly["admission_month_start"],
        errors="coerce"
    )

    timeline = timeline.merge(
        ip_monthly,
        left_on=["member_id", PREDICTION_DATE_COL],
        right_on=["member_id", "admission_month_start"],
        how="left"
    )

    timeline = timeline.drop(
        columns=["admission_month_start"],
        errors="ignore"
    )

    monthly_ip_cols = [
        "ip_claims_month",
        "ip_los_month",
        "ip_unique_drg_month"
    ]

    for col in monthly_ip_cols:
        timeline[col] = safe_numeric(timeline[col], fill_value=0)

    timeline = timeline.sort_values(
        ["member_id", PREDICTION_DATE_COL]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Prior IP lookback features
    # 30d = prior 1 month, 90d = prior 3 months,
    # 180d = prior 6 months, 365d = prior 12 months
    # --------------------------------------------------------

    window_map = {
        30: 1,
        90: 3,
        180: 6,
        365: 12
    }

    for days, months_back in window_map.items():

        timeline[f"prior_ip_claims_{days}d"] = (
            timeline
            .groupby("member_id")["ip_claims_month"]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=months_back,
                    min_periods=1
                ).sum()
            )
        )

        timeline[f"prior_ip_los_{days}d"] = (
            timeline
            .groupby("member_id")["ip_los_month"]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=months_back,
                    min_periods=1
                ).sum()
            )
        )

        timeline[f"prior_ip_unique_drg_{days}d"] = (
            timeline
            .groupby("member_id")["ip_unique_drg_month"]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=months_back,
                    min_periods=1
                ).sum()
            )
        )

    # --------------------------------------------------------
    # Future hospitalization labels
    # 30d = current month
    # 90d = current + next 2 months
    # 180d = current + next 5 months
    # --------------------------------------------------------

    future_window_map = {
        30: 1,
        90: 3,
        180: 6
    }

    for days, months_forward in future_window_map.items():

        future_claims = (
            timeline
            .groupby("member_id")["ip_claims_month"]
            .transform(
                lambda x: x.iloc[::-1]
                .rolling(
                    window=months_forward,
                    min_periods=1
                )
                .sum()
                .iloc[::-1]
            )
        )

        timeline[f"future_ip_claims_{days}d"] = future_claims

        timeline[f"hospitalized_next_{days}d"] = (
            timeline[f"future_ip_claims_{days}d"] > 0
        ).astype(int)

    # --------------------------------------------------------
    # Days since last IP approximation - SAFE VERSION
    # --------------------------------------------------------

    timeline["ever_hospitalized_before"] = (
        timeline.groupby("member_id")["ip_claims_month"]
        .transform(lambda x: x.shift(1).cumsum())
        .fillna(0)
        .gt(0)
        .astype(int)
    )

    timeline["ip_month_observed"] = pd.NaT

    timeline.loc[
        timeline["ip_claims_month"] > 0,
        "ip_month_observed"
    ] = timeline.loc[
        timeline["ip_claims_month"] > 0,
        PREDICTION_DATE_COL
    ]

    timeline["ip_month_observed"] = pd.to_datetime(
        timeline["ip_month_observed"],
        errors="coerce"
    )

    timeline["last_ip_month_prior"] = (
        timeline
        .groupby("member_id")["ip_month_observed"]
        .ffill()
    )

    timeline["last_ip_month_prior"] = (
        timeline
        .groupby("member_id")["last_ip_month_prior"]
        .shift(1)
    )

    timeline["last_ip_month_prior"] = pd.to_datetime(
        timeline["last_ip_month_prior"],
        errors="coerce"
    )

    timeline["days_since_last_ip"] = (
        timeline[PREDICTION_DATE_COL] -
        timeline["last_ip_month_prior"]
    ).dt.days

    timeline["days_since_last_ip"] = (
        timeline["days_since_last_ip"]
        .fillna(9999)
        .clip(lower=0)
    )

    timeline = timeline.drop(
        columns=[
            "ip_month_observed",
            "last_ip_month_prior"
        ],
        errors="ignore"
    )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    ip_feature_cols = [
        c for c in timeline.columns
        if c.startswith("prior_ip_")
        or c.startswith("future_ip_")
        or c.startswith("hospitalized_next_")
    ] + [
        "ip_claims_month",
        "ip_los_month",
        "ip_unique_drg_month",
        "days_since_last_ip",
        "ever_hospitalized_before"
    ]

    ip_feature_cols = list(dict.fromkeys(ip_feature_cols))

    for col in ip_feature_cols:
        timeline[col] = (
            safe_numeric(timeline[col], fill_value=0)
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
        )

    label_cols = [
        f"hospitalized_next_{w}d"
        for w in PREDICTION_WINDOWS_DAYS
        if f"hospitalized_next_{w}d" in timeline.columns
    ]

    print(f"\n{dataset_label} future hospitalization label rates:")
    print(timeline[label_cols].mean())

    print(f"\n{dataset_label} prior hospitalization summary:")
    prior_summary_cols = [
        "prior_ip_claims_30d",
        "prior_ip_claims_90d",
        "prior_ip_claims_180d",
        "prior_ip_claims_365d",
        "prior_ip_los_365d",
        "prior_ip_unique_drg_365d",
        "days_since_last_ip",
        "ever_hospitalized_before"
    ]

    prior_summary_cols = [
        c for c in prior_summary_cols
        if c in timeline.columns
    ]

    print(timeline[prior_summary_cols].describe())

    return timeline


df_train_timeline = add_hospitalization_timeline_features_fast(
    df_train_timeline,
    ip_train_monthly,
    "TRAIN Sets 1+2"
)

df_test_timeline = add_hospitalization_timeline_features_fast(
    df_test_timeline,
    ip_test_monthly,
    "TEST Set 3"
)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

df_train_timeline.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH2_train_sets1_2_timeline_with_labels.csv"
    ),
    index=False
)

df_test_timeline.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH2_test_set3_timeline_with_labels.csv"
    ),
    index=False
)

print("\nStep H2 completed successfully.")
print("df_train_timeline:", df_train_timeline.shape)
print("df_test_timeline:", df_test_timeline.shape)
print("Primary target:", TARGET_LABEL_NAME)


STEP H2 - FAST patient timeline and hospitalization labels

Loading inpatient TRAIN Sample 1: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv

Loading inpatient TRAIN Sample 2: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Inpatient_Claims_Sample_2.csv

TRAIN combined inpatient shape: (133267, 85)
TRAIN admission range: 2007-11-27 00:00:00 to 2010-12-30 00:00:00
TRAIN source samples:
source_sample
1    66773
2    66494
Name: count, dtype: int64

Loading inpatient TEST Sample 3: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Inpatient_Claims_Sample_3.csv

TEST combined inpatient shape: (66672, 85)
TEST admission range: 2007-11-29 00:00:00 to 2010-12-30 00:00:00
TEST source samples:
source_sample
3    66672
Name: count, dtype: int64

Expanding timeline members for TRAIN Sets 1+2...
TRAIN Sets 1+2 expanded timeline shape: (7265664, 39)

Expanding

In [ ]:
prof_cols = [
    c for c in df_train_timeline.columns
    if "risk" in c.lower()
    or "professional" in c.lower()
    or "provider" in c.lower()
    or "suspicious" in c.lower()
]

print(prof_cols)

[]


In [ ]:
possible_score_cols = [
    "patient_risk_score",
    "professional_patient_risk_score",
    "provider_patient_risk_score",
    "professional_risk_score"
]

existing_score_cols = [
    c for c in possible_score_cols
    if c in df_train_timeline.columns
]

print("Existing professional score columns:", existing_score_cols)

for c in existing_score_cols:
    print("\nTRAIN:", c)
    print(df_train_timeline[c].describe())
    print(df_train_timeline[c].value_counts(dropna=False).head(10))

    print("\nTEST:", c)
    print(df_test_timeline[c].describe())
    print(df_test_timeline[c].value_counts(dropna=False).head(10))

Existing professional score columns: []


In [ ]:
import os

PROFESSIONAL_MODEL_RISK_FILE = "/content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/professional_model_patient_risk_scores.csv"

print("Professional file exists:", os.path.exists(PROFESSIONAL_MODEL_RISK_FILE))

Professional file exists: True


In [ ]:
print("Train columns containing sample:")
print([c for c in df_train_timeline.columns if "sample" in c.lower()])

print("Test columns containing sample:")
print([c for c in df_test_timeline.columns if "sample" in c.lower()])

print("First 20 train columns:")
print(df_train_timeline.columns[:20].tolist())

Train columns containing sample:
[]
Test columns containing sample:
[]
First 20 train columns:
['member_id', 'prediction_month', 'rx_claims_month', 'unique_drugs_month', 'rx_days_supply_month', 'rx_quantity_month', 'rx_total_cost_month', 'rx_patient_pay_month', 'rx_claims_30d', 'unique_drugs_30d', 'rx_days_supply_30d', 'rx_quantity_30d', 'rx_total_cost_30d', 'rx_patient_pay_30d', 'rx_claims_90d', 'unique_drugs_90d', 'rx_days_supply_90d', 'rx_quantity_90d', 'rx_total_cost_90d', 'rx_patient_pay_90d']


In [ ]:
# ============================================================
# Step H2.5 - Merge Professional Claims Model Patient Risk Bridge
# Phase 1B: Hospitalization Model + Professional Claims Intelligence
# ============================================================

print("\nSTEP H2.5 - Merging professional claims model patient-risk bridge")

import os
import gc
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Professional model bridge files
# ------------------------------------------------------------

PROFESSIONAL_MODEL_TRAIN_FILE = os.path.join(
    INPUT_FOLDER,
    "professional_model_patient_risk_scores_train.csv"
)

PROFESSIONAL_MODEL_TEST_FILE = os.path.join(
    INPUT_FOLDER,
    "professional_model_patient_risk_scores_test.csv"
)

print("Professional train file exists:", os.path.exists(PROFESSIONAL_MODEL_TRAIN_FILE))
print("Professional test file exists:", os.path.exists(PROFESSIONAL_MODEL_TEST_FILE))

if not os.path.exists(PROFESSIONAL_MODEL_TRAIN_FILE):
    raise FileNotFoundError(PROFESSIONAL_MODEL_TRAIN_FILE)

if not os.path.exists(PROFESSIONAL_MODEL_TEST_FILE):
    raise FileNotFoundError(PROFESSIONAL_MODEL_TEST_FILE)

# ------------------------------------------------------------
# Load professional model outputs
# ------------------------------------------------------------

prof_train = pd.read_csv(
    PROFESSIONAL_MODEL_TRAIN_FILE,
    dtype={"member_id": str},
    low_memory=False
)

prof_test = pd.read_csv(
    PROFESSIONAL_MODEL_TEST_FILE,
    dtype={"member_id": str},
    low_memory=False
)

prof_train = prof_train.loc[:, ~prof_train.columns.duplicated()].copy()
prof_test = prof_test.loc[:, ~prof_test.columns.duplicated()].copy()

# ------------------------------------------------------------
# Standardize keys
# ------------------------------------------------------------

for df in [prof_train, prof_test, df_train_timeline, df_test_timeline]:
    df["member_id"] = (
        df["member_id"]
        .astype(str)
        .str.strip()
        .replace(["", "nan", "None", "NULL"], "UNKNOWN")
    )

    if "source_sample" in df.columns:
        df["source_sample"] = pd.to_numeric(
            df["source_sample"],
            errors="coerce"
        ).astype("Int64")

# ------------------------------------------------------------
# Select professional bridge columns
# ------------------------------------------------------------

prof_bridge_candidate_cols = [
    "source_sample",
    "member_id",

    # Core professional model bridge
    "patient_risk_score",
    "patient_risk_segment",
    "chronic_count",
    "high_risk_patient",

    # Professional utilization
    "professional_claim_count",
    "professional_line_count",
    "professional_provider_count",
    "professional_procedure_count",
    "professional_diagnosis_count",

    # Historical professional suspicious behavior
    "historical_suspicious_claim_rate",
    "historical_suspicious_claim_count",

    # Professional density / frequency
    "diagnosis_density",
    "procedure_density",
    "member_claim_volume",
    "member_procedure_frequency",
    "member_procedure_pct",

    # Optional demographic bridge fields
    "age",
    "is_female",
    "birth_year"
]

prof_train_cols = [
    c for c in prof_bridge_candidate_cols
    if c in prof_train.columns
]

prof_test_cols = [
    c for c in prof_bridge_candidate_cols
    if c in prof_test.columns
]

common_prof_cols = [
    c for c in prof_bridge_candidate_cols
    if c in prof_train_cols and c in prof_test_cols
]

if "member_id" not in common_prof_cols:
    raise KeyError("member_id missing from professional model files.")

if "source_sample" not in common_prof_cols:
    raise KeyError("source_sample missing from professional model files.")

print("\nProfessional bridge columns being merged:")
print(common_prof_cols)

prof_train_bridge = prof_train[common_prof_cols].copy()
prof_test_bridge = prof_test[common_prof_cols].copy()

# ------------------------------------------------------------
# Deduplicate one row per source_sample/member_id
# ------------------------------------------------------------

prof_train_bridge = (
    prof_train_bridge
    .drop_duplicates(subset=["source_sample", "member_id"])
    .copy()
)

prof_test_bridge = (
    prof_test_bridge
    .drop_duplicates(subset=["source_sample", "member_id"])
    .copy()
)

# ------------------------------------------------------------
# Rename columns to avoid conflicts with hospitalization native features
# ------------------------------------------------------------

rename_map = {
    c: f"prof_{c}"
    for c in common_prof_cols
    if c not in ["source_sample", "member_id"]
}

prof_train_bridge = prof_train_bridge.rename(columns=rename_map)
prof_test_bridge = prof_test_bridge.rename(columns=rename_map)

# ------------------------------------------------------------
# Drop existing prof_ columns before rerun
# ------------------------------------------------------------

prof_feature_cols = list(rename_map.values())

df_train_timeline = df_train_timeline.drop(
    columns=[c for c in prof_feature_cols if c in df_train_timeline.columns],
    errors="ignore"
)

df_test_timeline = df_test_timeline.drop(
    columns=[c for c in prof_feature_cols if c in df_test_timeline.columns],
    errors="ignore"
)

# ------------------------------------------------------------
# Ensure source_sample exists in timelines
# ------------------------------------------------------------

if "source_sample" not in df_train_timeline.columns:
    print("source_sample missing in df_train_timeline. Reconstructing from train samples.")
    df_train_timeline["source_sample"] = np.where(
        df_train_timeline["member_id"].astype(str).isin(
            prof_train.loc[prof_train["source_sample"].eq(1), "member_id"].astype(str)
        ),
        1,
        2
    )

if "source_sample" not in df_test_timeline.columns:
    print("source_sample missing in df_test_timeline. Assigning TEST_SAMPLE = 3.")
    df_test_timeline["source_sample"] = 3

df_train_timeline["source_sample"] = pd.to_numeric(
    df_train_timeline["source_sample"],
    errors="coerce"
).astype("Int64")

df_test_timeline["source_sample"] = pd.to_numeric(
    df_test_timeline["source_sample"],
    errors="coerce"
).astype("Int64")

# ------------------------------------------------------------
# Merge
# ------------------------------------------------------------

before_train_rows = len(df_train_timeline)
before_test_rows = len(df_test_timeline)

df_train_timeline = df_train_timeline.merge(
    prof_train_bridge.drop(columns=["source_sample"], errors="ignore"),
    on="member_id",
    how="left",
    copy=False
)

df_test_timeline = df_test_timeline.merge(
    prof_test_bridge.drop(columns=["source_sample"], errors="ignore"),
    on="member_id",
    how="left",
    copy=False
)

if len(df_train_timeline) != before_train_rows:
    raise ValueError("Train row count changed during professional bridge merge.")

if len(df_test_timeline) != before_test_rows:
    raise ValueError("Test row count changed during professional bridge merge.")

# ------------------------------------------------------------
# Fill professional features
# ------------------------------------------------------------

for col in prof_feature_cols:
    if col in df_train_timeline.columns:
        if df_train_timeline[col].dtype == "object":
            df_train_timeline[col] = (
                df_train_timeline[col]
                .fillna("UNKNOWN")
                .astype(str)
            )
            df_test_timeline[col] = (
                df_test_timeline[col]
                .fillna("UNKNOWN")
                .astype(str)
            )
        else:
            df_train_timeline[col] = (
                pd.to_numeric(df_train_timeline[col], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0)
            )
            df_test_timeline[col] = (
                pd.to_numeric(df_test_timeline[col], errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0)
            )

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\nProfessional model features merged successfully.")
print("Train shape:", df_train_timeline.shape)
print("Test shape:", df_test_timeline.shape)

print("\nProfessional feature columns now in train timeline:")
print([c for c in df_train_timeline.columns if c.startswith("prof_")])

print("\nProfessional bridge missing rate - train:")
print(
    df_train_timeline[[c for c in df_train_timeline.columns if c.startswith("prof_")]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(20)
)

print("\nProfessional bridge missing rate - test:")
print(
    df_test_timeline[[c for c in df_test_timeline.columns if c.startswith("prof_")]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(20)
)

gc.collect()

print("\nStep H2.5 completed successfully.")


STEP H2.5 - Merging professional claims model patient-risk bridge
Professional train file exists: True
Professional test file exists: True

Professional bridge columns being merged:
['source_sample', 'member_id', 'patient_risk_score', 'patient_risk_segment', 'chronic_count', 'high_risk_patient', 'diagnosis_density', 'procedure_density', 'member_claim_volume', 'member_procedure_frequency', 'member_procedure_pct', 'age', 'is_female', 'birth_year']
source_sample missing in df_train_timeline. Reconstructing from train samples.
source_sample missing in df_test_timeline. Assigning TEST_SAMPLE = 3.

Professional model features merged successfully.
Train shape: (7265664, 102)
Test shape: (3634740, 102)

Professional feature columns now in train timeline:
['prof_patient_risk_score', 'prof_patient_risk_segment', 'prof_chronic_count', 'prof_high_risk_patient', 'prof_diagnosis_density', 'prof_procedure_density', 'prof_member_claim_volume', 'prof_member_procedure_frequency', 'prof_member_procedure

In [ ]:
import pandas as pd

# ============================================================
# Step H3 - FAST / Memory-Safe Beneficiary + Professional Merge
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H3 - Adding beneficiary and professional history features")

# ------------------------------------------------------------
# Load beneficiary features
# ------------------------------------------------------------

def load_beneficiary_features(samples, dataset_label):

    print(f"\nLoading beneficiary features for {dataset_label}...")

    bene_parts = []

    for sample in samples:
        for yr in [2008, 2009, 2010]:

            path = beneficiary_file(sample, yr)

            print(
                f"Sample {sample}, Year {yr}:",
                os.path.exists(path)
            )

            if not os.path.exists(path):
                raise FileNotFoundError(path)

            usecols = [
                "DESYNPUF_ID",
                "BENE_BIRTH_DT",
                "BENE_SEX_IDENT_CD"
            ] + CHRONIC_CONDITION_COLUMNS

            temp = pd.read_csv(
                path,
                dtype=str,
                usecols=lambda c: c in usecols,
                low_memory=False
            )

            temp["bene_year"] = yr
            temp["source_sample"] = sample

            bene_parts.append(temp)

    bene = pd.concat(bene_parts, ignore_index=True)

    bene["member_id"] = bene["DESYNPUF_ID"].apply(clean_id)

    bene["BENE_BIRTH_DT"] = safe_date(
        bene["BENE_BIRTH_DT"]
    )

    bene["birth_year"] = bene["BENE_BIRTH_DT"].dt.year

    bene["is_female"] = (
        bene["BENE_SEX_IDENT_CD"]
        .astype(str)
        .str.strip()
        .eq("2")
        .astype("int8")
    )

    for col in CHRONIC_CONDITION_COLUMNS:

        if col in bene.columns:
            bene[col] = safe_numeric(
                bene[col],
                fill_value=0
            )
            bene[col] = (bene[col] == 1).astype("int8")
        else:
            bene[col] = 0

    bene["chronic_count"] = (
        bene[CHRONIC_CONDITION_COLUMNS]
        .sum(axis=1)
        .astype("int8")
    )

    bene["high_risk_patient"] = (
        bene["chronic_count"] >= 3
    ).astype("int8")

    keep_cols = [
        "member_id",
        "bene_year",
        "birth_year",
        "is_female",
        "chronic_count",
        "high_risk_patient"
    ] + CHRONIC_CONDITION_COLUMNS

    bene = (
        bene[keep_cols]
        .drop_duplicates(["member_id", "bene_year"])
        .reset_index(drop=True)
    )

    print(f"{dataset_label} beneficiary feature shape:", bene.shape)

    return bene


# ------------------------------------------------------------
# Merge beneficiary features into timeline
# ------------------------------------------------------------

def merge_beneficiary_to_timeline(timeline, bene, dataset_label):

    print(f"\nMerging beneficiary features into {dataset_label}...")

    timeline = timeline.copy()

    timeline[PREDICTION_DATE_COL] = pd.to_datetime(
        timeline[PREDICTION_DATE_COL],
        errors="coerce"
    )

    timeline["timeline_year"] = (
        timeline[PREDICTION_DATE_COL]
        .dt.year
        .astype("int16")
    )

    # Drop prior beneficiary columns if rerunning
    beneficiary_cols_to_drop = [
        "bene_year",
        "birth_year",
        "is_female",
        "chronic_count",
        "high_risk_patient",
        "age"
    ] + CHRONIC_CONDITION_COLUMNS

    duplicate_cols = []

    for col in beneficiary_cols_to_drop:
        duplicate_cols.extend([
            col,
            f"{col}_x",
            f"{col}_y"
        ])

    timeline = timeline.drop(
        columns=[
            c for c in duplicate_cols
            if c in timeline.columns
        ],
        errors="ignore"
    )

    timeline = timeline.merge(
        bene,
        left_on=["member_id", "timeline_year"],
        right_on=["member_id", "bene_year"],
        how="left"
    )

    timeline["age"] = (
        timeline["timeline_year"] -
        timeline["birth_year"]
    )

    timeline["age"] = (
        safe_numeric(timeline["age"], fill_value=0)
        .clip(lower=0, upper=100)
        .astype("int16")
    )

    numeric_fill_cols = [
        "is_female",
        "chronic_count",
        "high_risk_patient"
    ] + CHRONIC_CONDITION_COLUMNS

    for col in numeric_fill_cols:

        if col in timeline.columns:
            timeline[col] = (
                safe_numeric(timeline[col], fill_value=0)
                .astype("int16")
            )

    timeline = timeline.drop(
        columns=["bene_year", "birth_year"],
        errors="ignore"
    )

    print(f"{dataset_label} after beneficiary merge:", timeline.shape)

    return timeline


# ------------------------------------------------------------
# Merge professional model profile
# ------------------------------------------------------------

def merge_professional_profile(timeline, dataset_label):

    print(f"\nMerging professional profile into {dataset_label}...")

    timeline = timeline.copy()

    # Drop old professional columns if rerunning
    old_prof_cols = [
        c for c in timeline.columns
        if c.startswith("professional_")
    ]

    if old_prof_cols:
        timeline = timeline.drop(
            columns=old_prof_cols,
            errors="ignore"
        )

    if not USE_PROFESSIONAL_HISTORY_FEATURES:

        print("Professional history disabled.")

        timeline["professional_patient_risk_score"] = 0
        timeline["professional_patient_risk_segment"] = 0
        timeline["professional_professional_claim_count"] = 0
        timeline["professional_historical_suspicious_claim_rate"] = 0

        return timeline

    if not os.path.exists(PROFESSIONAL_MODEL_RISK_FILE):

        print("Professional profile file not found. Creating placeholders.")
        print("Expected:", PROFESSIONAL_MODEL_RISK_FILE)

        timeline["professional_patient_risk_score"] = 0
        timeline["professional_patient_risk_segment"] = 0
        timeline["professional_professional_claim_count"] = 0
        timeline["professional_historical_suspicious_claim_rate"] = 0

        if REQUIRE_PROFESSIONAL_MODEL_FILE:
            raise FileNotFoundError(PROFESSIONAL_MODEL_RISK_FILE)

        return timeline

    prof = pd.read_csv(
        PROFESSIONAL_MODEL_RISK_FILE,
        dtype=str,
        low_memory=False
    )

    if "member_id" not in prof.columns:
        raise ValueError(
            "Professional profile file must contain member_id."
        )

    prof["member_id"] = prof["member_id"].apply(clean_id)

    rename_map = {
        c: f"professional_{c}"
        for c in prof.columns
        if c != "member_id"
        and not c.startswith("professional_")
    }

    prof = prof.rename(columns=rename_map)

    prof_cols = [
        c for c in prof.columns
        if c != "member_id"
    ]

    for col in prof_cols:
        prof[col] = safe_numeric(
            prof[col],
            fill_value=0
        )

    prof = (
        prof[["member_id"] + prof_cols]
        .drop_duplicates("member_id")
        .reset_index(drop=True)
    )

    timeline = timeline.merge(
        prof,
        on="member_id",
        how="left"
    )

    for col in prof_cols:
        timeline[col] = (
            safe_numeric(timeline[col], fill_value=0)
        )

    print(
        f"{dataset_label} professional features merged:",
        len(prof_cols)
    )

    return timeline

# ------------------------------------------------------------
# Execute H3
# ------------------------------------------------------------

bene_train = load_beneficiary_features(
    TRAIN_SAMPLES,
    "TRAIN Sets 1+2"
)

bene_test = load_beneficiary_features(
    [TEST_SAMPLE],
    "TEST Set 3"
)

df_train_timeline = merge_beneficiary_to_timeline(
    df_train_timeline,
    bene_train,
    "TRAIN Sets 1+2"
)

df_test_timeline = merge_beneficiary_to_timeline(
    df_test_timeline,
    bene_test,
    "TEST Set 3"
)

df_train_timeline = merge_professional_profile(
    df_train_timeline,
    "TRAIN Sets 1+2"
)

df_test_timeline = merge_professional_profile(
    df_test_timeline,
    "TEST Set 3"
)

print("\nProfessional Match Rate")

print(
    df_train_timeline["professional_patient_risk_score"] # Changed 'patient_risk_score' to 'professional_patient_risk_score'
    .notna()
    .mean()
)

print(
    df_test_timeline["professional_patient_risk_score"] # Changed 'patient_risk_score' to 'professional_patient_risk_score'
    .notna()
    .mean()
)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

df_train_timeline.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH3_train_sets1_2_timeline_with_patient_context.csv"
    ),
    index=False
)

df_test_timeline.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH3_test_set3_timeline_with_patient_context.csv"
    ),
    index=False
)

print("\nStep H3 completed successfully.")
print("df_train_timeline:", df_train_timeline.shape)
print("df_test_timeline:", df_test_timeline.shape)


STEP H3 - Adding beneficiary and professional history features

Loading beneficiary features for TRAIN Sets 1+2...
Sample 1, Year 2008: True
Sample 1, Year 2009: True
Sample 1, Year 2010: True
Sample 2, Year 2008: True
Sample 2, Year 2009: True
Sample 2, Year 2010: True
TRAIN Sets 1+2 beneficiary feature shape: (687502, 17)

Loading beneficiary features for TEST Set 3...
Sample 3, Year 2008: True
Sample 3, Year 2009: True
Sample 3, Year 2010: True
TEST Set 3 beneficiary feature shape: (343846, 17)

Merging beneficiary features into TRAIN Sets 1+2...
TRAIN Sets 1+2 after beneficiary merge: (7265664, 102)

Merging beneficiary features into TEST Set 3...
TEST Set 3 after beneficiary merge: (3634740, 102)

Merging professional profile into TRAIN Sets 1+2...
TRAIN Sets 1+2 professional features merged: 11

Merging professional profile into TEST Set 3...
TEST Set 3 professional features merged: 11

Professional Match Rate
1.0
1.0

Step H3 completed successfully.
df_train_timeline: (7265664,

In [ ]:
print(df_train_timeline["professional_patient_risk_segment"].value_counts(dropna=False))

print(df_test_timeline["professional_patient_risk_segment"].value_counts(dropna=False))

professional_patient_risk_segment
0.0    5568768
1.0     937224
2.0     759672
Name: count, dtype: int64
professional_patient_risk_segment
0.0    3634740
Name: count, dtype: int64


In [ ]:
prof_cols = [c for c in df_train_timeline.columns if c.startswith("professional_")]

print(prof_cols)

print("\nTRAIN non-zero rates:")
print((df_train_timeline[prof_cols] != 0).mean().sort_values(ascending=False))

print("\nTEST non-zero rates:")
print((df_test_timeline[prof_cols] != 0).mean().sort_values(ascending=False))

['professional_patient_risk_score', 'professional_patient_risk_segment', 'professional_chronic_count', 'professional_high_risk_patient', 'professional_procedure_density', 'professional_diagnosis_density', 'professional_provider_claim_count', 'professional_claim_count', 'professional_line_count', 'professional_historical_suspicious_claim_rate', 'professional_historical_suspicious_claim_count']

TRAIN non-zero rates:
professional_patient_risk_score                   0.473933
professional_diagnosis_density                    0.473933
professional_procedure_density                    0.473933
professional_line_count                           0.473933
professional_claim_count                          0.473933
professional_provider_claim_count                 0.473908
professional_historical_suspicious_claim_rate     0.446310
professional_historical_suspicious_claim_count    0.446310
professional_chronic_count                        0.416214
professional_high_risk_patient                    

In [ ]:
# ============================================================
# Step H4 - Leakage-Safe Feature Set
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H4 - Building leakage-safe hospitalization forecasting feature set")

# ------------------------------------------------------------
# Validate target
# ------------------------------------------------------------

if TARGET_LABEL_NAME not in df_train_timeline.columns:
    raise ValueError(f"Target label {TARGET_LABEL_NAME} not found. Run H2 first.")

if TARGET_LABEL_NAME not in df_test_timeline.columns:
    raise ValueError(f"Target label {TARGET_LABEL_NAME} not found in test timeline. Run H2 first.")

# ------------------------------------------------------------
# Candidate safe features
# ------------------------------------------------------------

candidate_features = []

safe_prefixes = [
    "rx_",
    "unique_drugs_",
    "drug_growth_",
    "avg_",
    "polypharmacy_",
    "high_polypharmacy_",
    "prior_ip_",
    "professional_",
    "SP_"
]

safe_direct_features = [
    "days_since_last_ip",
    "ever_hospitalized_before",
    "age",
    "is_female",
    "chronic_count",
    "high_risk_patient",
    "timeline_year"
]

for col in df_train_timeline.columns:
    if col in safe_direct_features or any(col.startswith(prefix) for prefix in safe_prefixes):
        candidate_features.append(col)

# ------------------------------------------------------------
# Explicit leakage exclusions
# ------------------------------------------------------------

explicit_leakage_cols = [
    TARGET_LABEL_NAME,

    # Future labels / future outcomes
    "hospitalized_next_30d",
    "hospitalized_next_90d",
    "hospitalized_next_180d",
    "future_ip_claims_30d",
    "future_ip_claims_90d",
    "future_ip_claims_180d",

    # Same-month inpatient fields — these leak the target
    "ip_claims_month",
    "ip_los_month",
    "ip_unique_drg_month",

    # Dates / identifiers
    "member_id",
    PREDICTION_DATE_COL
]

if "LEAKAGE_COLUMNS" in globals():
    explicit_leakage_cols = list(dict.fromkeys(
        explicit_leakage_cols + LEAKAGE_COLUMNS
    ))

leakage_terms = [
    "future_",
    "hospitalized_next_",
    "admission_date",
    "discharge_date",
    "post_prediction"
]

# Optional: exclude paid/payment terms from predictors
leakage_terms += [
    "paid",
    "payment"
]

# ------------------------------------------------------------
# Final safe feature list
# ------------------------------------------------------------

safe_features_h = [
    c for c in candidate_features
    if c in df_train_timeline.columns
    and c in df_test_timeline.columns
    and c not in explicit_leakage_cols
    and not any(term in c.lower() for term in leakage_terms)
]

safe_features_h = list(dict.fromkeys(safe_features_h))

# ------------------------------------------------------------
# Build train/test matrices
# ------------------------------------------------------------

X_train_h = df_train_timeline[safe_features_h].copy()
X_test_h = df_test_timeline[safe_features_h].copy()

y_train_h = df_train_timeline[TARGET_LABEL_NAME].astype(int)
y_test_h = df_test_timeline[TARGET_LABEL_NAME].astype(int)

for col in safe_features_h:
    X_train_h[col] = safe_numeric(X_train_h[col], fill_value=0)
    X_test_h[col] = safe_numeric(X_test_h[col], fill_value=0)

X_train_h = X_train_h.replace([np.inf, -np.inf], 0).fillna(0)
X_test_h = X_test_h.replace([np.inf, -np.inf], 0).fillna(0)

# ------------------------------------------------------------
# Leakage audit
# ------------------------------------------------------------

leakage_found_h = [
    c for c in X_train_h.columns
    if c in explicit_leakage_cols
    or any(term in c.lower() for term in leakage_terms)
]

print("Train matrix:", X_train_h.shape)
print("Test matrix:", X_test_h.shape)

print("\nTarget distribution - TRAIN:")
print(y_train_h.value_counts(normalize=True).sort_index())

print("\nTarget distribution - TEST:")
print(y_test_h.value_counts(normalize=True).sort_index())

print("\nFeature count:", len(safe_features_h))

print("\nLeakage audit:")
print(leakage_found_h)

if len(leakage_found_h) > 0:
    raise ValueError("Leakage columns found: " + str(leakage_found_h))

# ------------------------------------------------------------
# Save feature list and model-ready files
# ------------------------------------------------------------

pd.DataFrame({
    "safe_feature": safe_features_h
}).to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH4_safe_feature_list_train_sets1_2_test_set3.csv"
    ),
    index=False
)

df_train_timeline.to_csv(
    TRAIN_MODEL_READY_FILE,
    index=False
)

df_test_timeline.to_csv(
    TEST_MODEL_READY_FILE,
    index=False
)

print("\nStep H4 completed successfully.")
print("Leakage-safe hospitalization forecasting feature set is ready.")


STEP H4 - Building leakage-safe hospitalization forecasting feature set
Train matrix: (7265664, 78)
Test matrix: (3634740, 78)

Target distribution - TRAIN:
hospitalized_next_90d
0    0.953875
1    0.046125
Name: proportion, dtype: float64

Target distribution - TEST:
hospitalized_next_90d
0    0.953829
1    0.046171
Name: proportion, dtype: float64

Feature count: 78

Leakage audit:
[]

Step H4 completed successfully.
Leakage-safe hospitalization forecasting feature set is ready.


In [ ]:
# ============================================================
# Step H5 - First Hospitalization Forecasting Model
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H5 - Training first hospitalization forecasting model")

# ------------------------------------------------------------
# Validate required objects
# ------------------------------------------------------------

required_objects = [
    "X_train_h",
    "X_test_h",
    "y_train_h",
    "y_test_h",
    "df_train_timeline",
    "df_test_timeline"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"{obj} not found. Run Step H4 first.")

print("Train matrix:", X_train_h.shape)
print("Test matrix:", X_test_h.shape)

print("\nTRAIN target distribution:")
print(y_train_h.value_counts(normalize=True).sort_index())

print("\nTEST target distribution:")
print(y_test_h.value_counts(normalize=True).sort_index())

# ------------------------------------------------------------
# Train first model
# ------------------------------------------------------------

hospitalization_model = HistGradientBoostingClassifier(
    **HOSPITALIZATION_MODEL_PARAMS
)

hospitalization_model.fit(X_train_h, y_train_h)

# ------------------------------------------------------------
# Predict probabilities
# ------------------------------------------------------------

train_proba_h = hospitalization_model.predict_proba(X_train_h)[:, 1]
test_proba_h = hospitalization_model.predict_proba(X_test_h)[:, 1]

train_auc_h = roc_auc_score(y_train_h, train_proba_h)
test_auc_h = roc_auc_score(y_test_h, test_proba_h)
auc_gap_h = train_auc_h - test_auc_h

print("\nHOSPITALIZATION FORECASTING RESULTS")
print("TRAIN AUC:", round(train_auc_h, 4))
print("TEST AUC:", round(test_auc_h, 4))
print("AUC GAP:", round(auc_gap_h, 4))

# ------------------------------------------------------------
# Default threshold evaluation
# ------------------------------------------------------------

threshold_h = CARE_MANAGEMENT_THRESHOLD

test_pred_h = (
    test_proba_h >= threshold_h
).astype(int)

precision_h = precision_score(
    y_test_h,
    test_pred_h,
    zero_division=0
)

recall_h = recall_score(
    y_test_h,
    test_pred_h,
    zero_division=0
)

f1_h = f1_score(
    y_test_h,
    test_pred_h,
    zero_division=0
)

cm_h = confusion_matrix(
    y_test_h,
    test_pred_h,
    labels=[0, 1]
)

tn_h, fp_h, fn_h, tp_h = cm_h.ravel()

false_negative_rate_h = (
    fn_h / (fn_h + tp_h)
    if (fn_h + tp_h) > 0
    else 0
)

care_management_workload_rate_h = (
    (tp_h + fp_h) / len(y_test_h)
    if len(y_test_h) > 0
    else 0
)

print(f"\nTEST METRICS AT {threshold_h} THRESHOLD")
print("Precision:", round(precision_h, 4))
print("Recall:", round(recall_h, 4))
print("F1:", round(f1_h, 4))
print("False Negative Rate:", round(false_negative_rate_h, 4))
print("Care Management Workload Rate:", round(care_management_workload_rate_h, 4))
print("\nConfusion Matrix:")
print(cm_h)

# ------------------------------------------------------------
# Save prediction outputs
# ------------------------------------------------------------

hospitalization_predictions_train = pd.DataFrame({
    "member_id": df_train_timeline["member_id"].values,
    PREDICTION_DATE_COL: df_train_timeline[PREDICTION_DATE_COL].values,
    "actual": y_train_h.values,
    "predicted_probability": train_proba_h
})

hospitalization_predictions_test = pd.DataFrame({
    "member_id": df_test_timeline["member_id"].values,
    PREDICTION_DATE_COL: df_test_timeline[PREDICTION_DATE_COL].values,
    "actual": y_test_h.values,
    "predicted_probability": test_proba_h
})

hospitalization_predictions_train.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH5_train_sets1_2_hospitalization_predictions.csv"
    ),
    index=False
)

hospitalization_predictions_test.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH5_test_set3_hospitalization_predictions.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save model results summary
# ------------------------------------------------------------

stepH5_results = pd.DataFrame({
    "metric": [
        "train_auc",
        "test_auc",
        "auc_gap",
        "threshold",
        "precision",
        "recall",
        "f1",
        "false_negative_rate",
        "care_management_workload_rate",
        "tn",
        "fp",
        "fn",
        "tp",
        "train_rows",
        "test_rows",
        "feature_count"
    ],
    "value": [
        train_auc_h,
        test_auc_h,
        auc_gap_h,
        threshold_h,
        precision_h,
        recall_h,
        f1_h,
        false_negative_rate_h,
        care_management_workload_rate_h,
        tn_h,
        fp_h,
        fn_h,
        tp_h,
        X_train_h.shape[0],
        X_test_h.shape[0],
        X_train_h.shape[1]
    ]
})

stepH5_results.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH5_model_results_train_sets1_2_test_set3.csv"
    ),
    index=False
)

print("\nStep H5 completed successfully.")
print("Saved:")
print(os.path.join(OUTPUT_FOLDER, "stepH5_train_sets1_2_hospitalization_predictions.csv"))
print(os.path.join(OUTPUT_FOLDER, "stepH5_test_set3_hospitalization_predictions.csv"))
print(os.path.join(OUTPUT_FOLDER, "stepH5_model_results_train_sets1_2_test_set3.csv"))


STEP H5 - Training first hospitalization forecasting model
Train matrix: (7265664, 78)
Test matrix: (3634740, 78)

TRAIN target distribution:
hospitalized_next_90d
0    0.953875
1    0.046125
Name: proportion, dtype: float64

TEST target distribution:
hospitalized_next_90d
0    0.953829
1    0.046171
Name: proportion, dtype: float64

HOSPITALIZATION FORECASTING RESULTS
TRAIN AUC: 0.8437
TEST AUC: 0.839
AUC GAP: 0.0048

TEST METRICS AT 0.5 THRESHOLD
Precision: 0.5075
Recall: 0.0026
F1: 0.0052
False Negative Rate: 0.9974
Care Management Workload Rate: 0.0002

Confusion Matrix:
[[3466496     425]
 [ 167381     438]]

Step H5 completed successfully.
Saved:
/content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Output/stepH5_train_sets1_2_hospitalization_predictions.csv
/content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Output/stepH5_test_set3_hospitalization_predictions.csv
/content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Out

In [ ]:
# ============================================================
# Step H6 - Threshold Analysis / Care Management Scorecard
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H6 - Building hospitalization threshold scorecard")

required_objects = [
    "test_proba_h",
    "y_test_h",
    "df_test_timeline"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"{obj} not found. Run Step H5 first.")

threshold_rows_h = []

for threshold in THRESHOLD_GRID:

    pred = (test_proba_h >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_h,
        pred,
        labels=[0, 1]
    ).ravel()

    total = tn + fp + fn + tp

    precision = precision_score(y_test_h, pred, zero_division=0)
    recall = recall_score(y_test_h, pred, zero_division=0)
    f1 = f1_score(y_test_h, pred, zero_division=0)

    false_negative_rate = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0
    )

    care_management_workload_rate = (
        (tp + fp) / total
        if total > 0
        else 0
    )

    non_intervention_rate = (
        (tn + fn) / total
        if total > 0
        else 0
    )

    threshold_rows_h.append({
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_negative_rate": false_negative_rate,
        "care_management_workload_rate": care_management_workload_rate,
        "non_intervention_rate": non_intervention_rate
    })

threshold_scorecard_h = pd.DataFrame(threshold_rows_h)

threshold_scorecard_h["meets_false_negative_limit"] = (
    threshold_scorecard_h["false_negative_rate"] <= FALSE_NEGATIVE_RATE_LIMIT
)

threshold_scorecard_h["meets_workload_limit"] = (
    threshold_scorecard_h["care_management_workload_rate"] <= CARE_MANAGEMENT_WORKLOAD_LIMIT
)

threshold_scorecard_h["operationally_viable"] = (
    threshold_scorecard_h["meets_false_negative_limit"]
    &
    threshold_scorecard_h["meets_workload_limit"]
)

viable_h = threshold_scorecard_h[
    threshold_scorecard_h["operationally_viable"]
].copy()

if len(viable_h) > 0:
    recommended_threshold_h = (
        viable_h
        .sort_values(
            by=["f1", "precision"],
            ascending=[False, False]
        )
        .iloc[0]["threshold"]
    )
else:
    recommended_threshold_h = (
        threshold_scorecard_h
        .sort_values(
            by=["false_negative_rate", "care_management_workload_rate"],
            ascending=[True, True]
        )
        .iloc[0]["threshold"]
    )

recommended_threshold_h = float(recommended_threshold_h)

print("\nHospitalization threshold scorecard:")
print(threshold_scorecard_h)

print("\nRecommended hospitalization threshold:", recommended_threshold_h)

threshold_scorecard_h.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH6_threshold_scorecard_train_sets1_2_test_set3.csv"
    ),
    index=False
)

print("\nStep H6 completed successfully.")


STEP H6 - Building hospitalization threshold scorecard

Hospitalization threshold scorecard:
    threshold       tn      fp      fn      tp  precision    recall        f1  \
0        0.05  2575504  891417   37876  129943   0.127225  0.774304  0.218542   
1        0.10  3040140  426781   73928   93891   0.180327  0.559478  0.272744   
2        0.15  3267323  199598  105493   62326   0.237955  0.371388  0.290062   
3        0.20  3363604  103317  125964   41855   0.288313  0.249406  0.267452   
4        0.25  3409946   56975  139195   28624   0.334396  0.170565  0.225903   
5        0.30  3436302   30619  149160   18659   0.378648  0.111185  0.171896   
6        0.35  3451570   15351  156680   11139   0.420498  0.066375  0.114652   
7        0.40  3460030    6891  162221    5598   0.448234  0.033357  0.062094   
8        0.45  3464832    2089  165887    1932   0.480477  0.011512  0.022486   
9        0.50  3466496     425  167381     438   0.507532  0.002610  0.005193   
10       0.55  

In [ ]:
# ============================================================
# Step H7 - Hospitalization Risk Tier Assignment
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H7 - Assigning hospitalization risk tiers")

required_objects = [
    "df_test_timeline",
    "test_proba_h",
    "y_test_h",
    "recommended_threshold_h"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"{obj} not found. Run H5 and H6 first.")

hospitalization_scored_test = df_test_timeline.copy()

hospitalization_scored_test["actual"] = y_test_h.values
hospitalization_scored_test["predicted_probability"] = test_proba_h

hospitalization_scored_test["predicted_flag_recommended"] = (
    hospitalization_scored_test["predicted_probability"] >= recommended_threshold_h
).astype(int)

hospitalization_scored_test["hospitalization_risk_tier"] = pd.cut(
    hospitalization_scored_test["predicted_probability"],
    bins=[
        -0.001,
        WATCHLIST_THRESHOLD,
        CARE_MANAGEMENT_THRESHOLD,
        HIGH_RISK_THRESHOLD,
        0.90,
        1.001
    ],
    labels=[
        "LOW_RISK",
        "WATCHLIST",
        "CARE_MANAGEMENT",
        "HIGH_RISK",
        "CRITICAL_RISK"
    ]
)

hospitalization_risk_tier_summary = (
    hospitalization_scored_test
    .groupby("hospitalization_risk_tier", observed=True)
    .agg(
        patient_month_count=("member_id", "count"),
        unique_members=("member_id", "nunique"),
        actual_hospitalization_rate=("actual", "mean"),
        avg_predicted_probability=("predicted_probability", "mean")
    )
    .reset_index()
)

hospitalization_risk_tier_summary["patient_month_pct"] = (
    hospitalization_risk_tier_summary["patient_month_count"] /
    hospitalization_risk_tier_summary["patient_month_count"].sum()
)

print("\nHospitalization risk tier summary:")
print(hospitalization_risk_tier_summary)

hospitalization_scored_test[
    [
        "member_id",
        PREDICTION_DATE_COL,
        "actual",
        "predicted_probability",
        "predicted_flag_recommended",
        "hospitalization_risk_tier"
    ]
].to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH7_test_set3_scored_hospitalization_risk_tiers.csv"
    ),
    index=False
)

hospitalization_risk_tier_summary.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH7_hospitalization_risk_tier_summary_train_sets1_2_test_set3.csv"
    ),
    index=False
)

print("\nStep H7 completed successfully.")


STEP H7 - Assigning hospitalization risk tiers

Hospitalization risk tier summary:
  hospitalization_risk_tier  patient_month_count  unique_members  \
0                  LOW_RISK              3585462          100965   
1                 WATCHLIST                48415            8495   
2           CARE_MANAGEMENT                  863             591   

   actual_hospitalization_rate  avg_predicted_probability  patient_month_pct  
0                     0.041601                   0.041644           0.986442  
1                     0.376350                   0.364548           0.013320  
2                     0.507532                   0.530378           0.000237  

Step H7 completed successfully.


In [ ]:
# ============================================================
# Step H8 - Feature Importance / Explainability
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H8 - Building hospitalization explainability layer")

required_objects = [
    "hospitalization_model",
    "X_test_h",
    "y_test_h"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"{obj} not found. Run H5 first.")

print("\nUsing permutation importance...")

perm_h = permutation_importance(
    hospitalization_model,
    X_test_h,
    y_test_h,
    n_repeats=3,
    random_state=RANDOM_STATE,
    scoring="roc_auc"
)

importance_h = pd.DataFrame({
    "feature": X_test_h.columns,
    "importance": perm_h.importances_mean,
    "importance_std": perm_h.importances_std
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)


def classify_hospitalization_feature(feature_name):

    f = feature_name.lower()

    if f.startswith("rx_") or "drug" in f or "polypharmacy" in f:
        return "RX_TRAJECTORY"

    if f.startswith("prior_ip") or "hospitalized_before" in f or "days_since_last_ip" in f:
        return "INPATIENT_HISTORY"

    if f.startswith("professional_"):
        return "PROFESSIONAL_HISTORY"

    if f.startswith("sp_") or "chronic" in f or "risk_patient" in f:
        return "CHRONIC_CONDITION"

    if "age" in f or "female" in f:
        return "DEMOGRAPHIC"

    return "OTHER"


importance_h["feature_category"] = importance_h["feature"].apply(
    classify_hospitalization_feature
)

importance_h["possible_leakage"] = (
    importance_h["feature"]
    .str.lower()
    .str.contains(
        "future|hospitalized_next|post_prediction|admission_date|discharge_date",
        regex=True
    )
)

leakage_h = importance_h[importance_h["possible_leakage"]].copy()

print("\nPotential leakage hits:")
print(leakage_h)

print("\nTop 50 hospitalization forecasting features:")
print(importance_h.head(50))

category_summary_h = (
    importance_h
    .groupby("feature_category")
    .agg(
        feature_count=("feature", "count"),
        total_importance=("importance", "sum"),
        avg_importance=("importance", "mean")
    )
    .reset_index()
    .sort_values("total_importance", ascending=False)
)

print("\nHospitalization feature category summary:")
print(category_summary_h)

importance_h.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH8_feature_importance_full_train_sets1_2_test_set3.csv"
    ),
    index=False
)

category_summary_h.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH8_feature_category_summary_train_sets1_2_test_set3.csv"
    ),
    index=False
)

if len(leakage_h) > 0:
    raise ValueError("Possible leakage features detected in H8.")

print("\nStep H8 completed successfully.")


STEP H8 - Building hospitalization explainability layer

Using permutation importance...

Potential leakage hits:
Empty DataFrame
Columns: [feature, importance, importance_std, feature_category, possible_leakage]
Index: []

Top 50 hospitalization forecasting features:
                       feature  importance  importance_std   feature_category  \
0                chronic_count    0.225122    1.209973e-04  CHRONIC_CONDITION   
1                  SP_CHRNKIDN    0.012990    3.233326e-05  CHRONIC_CONDITION   
2           days_since_last_ip    0.009044    4.633898e-05  INPATIENT_HISTORY   
3     ever_hospitalized_before    0.008265    3.981731e-05  INPATIENT_HISTORY   
4                timeline_year    0.007980    7.977190e-05              OTHER   
5                      SP_COPD    0.007919    7.373326e-05  CHRONIC_CONDITION   
6                  SP_STRKETIA    0.002460    2.479128e-05  CHRONIC_CONDITION   
7               rx_claims_365d    0.001851    2.959366e-05      RX_TRAJECTORY   
8

In [ ]:
# ============================================================
# Step H9 - Segment Validation
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H9 - Building hospitalization segment validation")

required_objects = [
    "hospitalization_scored_test",
    "recommended_threshold_h"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"{obj} not found. Run H7 first.")

segment_h = hospitalization_scored_test.copy()

segment_h["predicted_flag"] = (
    segment_h["predicted_probability"] >= recommended_threshold_h
).astype(int)


def hospitalization_segment_performance(df, segment_col, min_rows=100):

    rows = []

    for segment_value, g in df.groupby(segment_col, dropna=False):

        if len(g) < min_rows:
            continue

        y_true = g["actual"].astype(int)
        y_pred = g["predicted_flag"].astype(int)
        y_prob = g["predicted_probability"]

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        ).ravel()

        auc = np.nan

        if y_true.nunique() > 1:
            auc = roc_auc_score(y_true, y_prob)

        false_negative_rate = (
            fn / (fn + tp)
            if (fn + tp) > 0
            else 0
        )

        workload_rate = (
            (tp + fp) / len(g)
            if len(g) > 0
            else 0
        )

        rows.append({
            "segment_column": segment_col,
            "segment_value": segment_value,
            "patient_month_count": len(g),
            "unique_members": g["member_id"].nunique(),
            "actual_hospitalization_rate": y_true.mean(),
            "avg_predicted_probability": y_prob.mean(),
            "auc": auc,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
            "false_negative_rate": false_negative_rate,
            "workload_rate": workload_rate
        })

    return pd.DataFrame(rows)


segment_cols_h = [
    "hospitalization_risk_tier",
    "timeline_year",
    "patient_risk_segment",
    "professional_patient_risk_segment",
    "high_risk_patient",
    "high_polypharmacy_flag_90d",
    "ever_hospitalized_before"
]

available_segment_cols_h = [
    c for c in segment_cols_h
    if c in segment_h.columns
]

segment_results_h = []

for col in available_segment_cols_h:
    print("Validating segment:", col)

    temp = hospitalization_segment_performance(
        segment_h,
        col,
        min_rows=100
    )

    segment_results_h.append(temp)

segment_performance_h = pd.concat(
    segment_results_h,
    ignore_index=True
)

segment_performance_h["false_negative_exceeds_limit"] = (
    segment_performance_h["false_negative_rate"] > FALSE_NEGATIVE_RATE_LIMIT
)

segment_performance_h["low_auc_flag"] = (
    segment_performance_h["auc"].notna()
    &
    (segment_performance_h["auc"] < 0.70)
)

segment_performance_h["needs_review"] = (
    segment_performance_h["false_negative_exceeds_limit"]
    |
    segment_performance_h["low_auc_flag"]
)

print("\nHospitalization segment performance:")
print(segment_performance_h.head(50))

print("\nSegments needing review:")
print(
    segment_performance_h[
        segment_performance_h["needs_review"]
    ].head(50)
)

segment_performance_h.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH9_segment_performance_validation_train_sets1_2_test_set3.csv"
    ),
    index=False
)

segment_performance_h[
    segment_performance_h["needs_review"]
].to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH9_segments_needing_review_train_sets1_2_test_set3.csv"
    ),
    index=False
)

print("\nStep H9 completed successfully.")


STEP H9 - Building hospitalization segment validation
Validating segment: hospitalization_risk_tier
Validating segment: timeline_year
Validating segment: professional_patient_risk_segment
Validating segment: high_risk_patient
Validating segment: high_polypharmacy_flag_90d
Validating segment: ever_hospitalized_before

Hospitalization segment performance:
                       segment_column    segment_value  patient_month_count  \
0           hospitalization_risk_tier         LOW_RISK              3585462   
1           hospitalization_risk_tier        WATCHLIST                48415   
2           hospitalization_risk_tier  CARE_MANAGEMENT                  863   
3                       timeline_year             2008              1211580   
4                       timeline_year             2009              1211580   
5                       timeline_year             2010              1211580   
6   professional_patient_risk_segment              0.0              3634740   
7          

In [ ]:
# ============================================================
# Step H10 - Final Governance Summary
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H10 - Building hospitalization forecasting governance summary")

required_objects = [
    "train_auc_h",
    "test_auc_h",
    "threshold_scorecard_h",
    "hospitalization_risk_tier_summary",
    "segment_performance_h",
    "recommended_threshold_h",
    "X_train_h",
    "X_test_h",
    "y_train_h",
    "y_test_h"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"{obj} not found. Run H5-H9 first.")

rec_row_h = threshold_scorecard_h[
    threshold_scorecard_h["threshold"] == recommended_threshold_h
]

if rec_row_h.empty:
    rec_row_h = threshold_scorecard_h.iloc[[0]]

rec_row_h = rec_row_h.iloc[0]

auc_gap_h = train_auc_h - test_auc_h

passes_auc_h = test_auc_h >= 0.70
passes_gap_h = auc_gap_h <= 0.10
passes_false_negative_h = (
    rec_row_h["false_negative_rate"] <= FALSE_NEGATIVE_RATE_LIMIT
)
passes_workload_h = (
    rec_row_h["care_management_workload_rate"] <= CARE_MANAGEMENT_WORKLOAD_LIMIT
)

segments_needing_review_h = (
    segment_performance_h["needs_review"].sum()
    if "needs_review" in segment_performance_h.columns
    else 0
)

hospitalization_governance_status = (
    "PASS_FOR_PILOT"
    if (
        passes_auc_h
        and passes_gap_h
        and passes_false_negative_h
        and passes_workload_h
    )
    else "REVIEW_BEFORE_PILOT"
)

hospitalization_validation_summary = pd.DataFrame({
    "metric": [
        "train_auc",
        "test_auc",
        "auc_gap",
        "recommended_threshold",
        "precision",
        "recall",
        "f1",
        "false_negative_rate",
        "care_management_workload_rate",
        "non_intervention_rate",
        "train_rows",
        "test_rows",
        "feature_count",
        "segments_needing_review",
        "governance_status"
    ],
    "value": [
        train_auc_h,
        test_auc_h,
        auc_gap_h,
        recommended_threshold_h,
        rec_row_h["precision"],
        rec_row_h["recall"],
        rec_row_h["f1"],
        rec_row_h["false_negative_rate"],
        rec_row_h["care_management_workload_rate"],
        rec_row_h["non_intervention_rate"],
        X_train_h.shape[0],
        X_test_h.shape[0],
        X_train_h.shape[1],
        segments_needing_review_h,
        hospitalization_governance_status
    ]
})

hospitalization_governance_checklist = pd.DataFrame({
    "control": [
        "External validation used",
        "Patient-month timeline created",
        "Rx/PDE features included",
        "Future hospitalization label created",
        "Leakage-safe feature set created",
        "Test AUC >= 0.70",
        "AUC gap <= 0.10",
        "False negative rate <= limit",
        "Care management workload <= limit",
        "Segment validation completed"
    ],
    "status": [
        "PASS" if USE_EXTERNAL_VALIDATION else "REVIEW",
        "PASS",
        "PASS" if USE_RX_FEATURES else "REVIEW",
        "PASS",
        "PASS",
        "PASS" if passes_auc_h else "REVIEW",
        "PASS" if passes_gap_h else "REVIEW",
        "PASS" if passes_false_negative_h else "REVIEW",
        "PASS" if passes_workload_h else "REVIEW",
        "PASS"
    ],
    "notes": [
        f"Train Samples {TRAIN_SAMPLES}; Test Sample {TEST_SAMPLE}",
        "Unit of prediction is member-month.",
        "PDE/Rx lookback features created in H1.",
        f"Target: {TARGET_LABEL_NAME}",
        f"Feature count: {X_train_h.shape[1]}",
        f"Test AUC: {test_auc_h:.4f}",
        f"AUC gap: {auc_gap_h:.4f}",
        f"False negative rate: {rec_row_h['false_negative_rate']:.4f}",
        f"Workload rate: {rec_row_h['care_management_workload_rate']:.4f}",
        f"Segments needing review: {segments_needing_review_h}"
    ]
})

hospitalization_executive_summary = pd.DataFrame({
    "domain": [
        "Model purpose",
        "Prediction window",
        "Training population",
        "Testing population",
        "Recommended threshold",
        "Care management workload",
        "Missed hospitalization control",
        "Risk tiering",
        "Pilot recommendation"
    ],
    "summary": [
        "Predict likelihood of hospitalization using patient timeline, Rx, prior inpatient, beneficiary, and professional-history features.",
        f"{PRIMARY_PREDICTION_WINDOW_DAYS} days",
        f"SynPUF Samples {TRAIN_SAMPLES}",
        f"SynPUF Sample {TEST_SAMPLE}",
        f"{recommended_threshold_h:.2f}",
        f"{rec_row_h['care_management_workload_rate']:.2%}",
        f"False negative rate is {rec_row_h['false_negative_rate']:.2%} versus limit of {FALSE_NEGATIVE_RATE_LIMIT:.2%}.",
        "LOW_RISK, WATCHLIST, CARE_MANAGEMENT, HIGH_RISK, CRITICAL_RISK",
        hospitalization_governance_status
    ]
})

print("\nHospitalization validation summary:")
print(hospitalization_validation_summary)

print("\nHospitalization governance checklist:")
print(hospitalization_governance_checklist)

print("\nHospitalization executive summary:")
print(hospitalization_executive_summary)

hospitalization_validation_summary.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH10_validation_summary_train_sets1_2_test_set3.csv"
    ),
    index=False
)

hospitalization_governance_checklist.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH10_governance_checklist_train_sets1_2_test_set3.csv"
    ),
    index=False
)

hospitalization_executive_summary.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH10_executive_summary_train_sets1_2_test_set3.csv"
    ),
    index=False
)

stepH10_excel = os.path.join(
    OUTPUT_FOLDER,
    "stepH10_hospitalization_forecasting_report_train_sets1_2_test_set3.xlsx"
)

with pd.ExcelWriter(stepH10_excel) as writer:
    hospitalization_validation_summary.to_excel(
        writer,
        sheet_name="Validation Summary",
        index=False
    )
    hospitalization_governance_checklist.to_excel(
        writer,
        sheet_name="Governance Checklist",
        index=False
    )
    hospitalization_executive_summary.to_excel(
        writer,
        sheet_name="Executive Summary",
        index=False
    )
    threshold_scorecard_h.to_excel(
        writer,
        sheet_name="Threshold Scorecard",
        index=False
    )
    hospitalization_risk_tier_summary.to_excel(
        writer,
        sheet_name="Risk Tier Summary",
        index=False
    )
    segment_performance_h.to_excel(
        writer,
        sheet_name="Segment Validation",
        index=False
    )

print("\nStep H10 completed successfully.")
print("Governance status:", hospitalization_governance_status)
print("Saved:", stepH10_excel)


STEP H10 - Building hospitalization forecasting governance summary

Hospitalization validation summary:
                           metric                value
0                       train_auc             0.843747
1                        test_auc             0.838972
2                         auc_gap             0.004775
3           recommended_threshold                 0.05
4                       precision             0.127225
5                          recall             0.774304
6                              f1             0.218542
7             false_negative_rate             0.225696
8   care_management_workload_rate             0.280999
9           non_intervention_rate             0.719001
10                     train_rows              7265664
11                      test_rows              3634740
12                  feature_count                   78
13        segments_needing_review                   13
14              governance_status  REVIEW_BEFORE_PILOT

Hospitalizatio

In [ ]:
# ============================================================
# Step H11 - Multi-Model and Ensemble Comparison
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H11 - Multi-model and ensemble comparison")

candidate_models_h = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        **HOSPITALIZATION_MODEL_PARAMS
    )
}

trained_models_h = {}
model_results_h = []

for model_name, model in candidate_models_h.items():

    print(f"\nTraining {model_name}...")

    model.fit(X_train_h, y_train_h)

    proba = model.predict_proba(X_test_h)[:, 1]
    auc = roc_auc_score(y_test_h, proba)

    pred = (proba >= recommended_threshold_h).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test_h, pred, labels=[0, 1]).ravel()

    model_results_h.append({
        "model_name": model_name,
        "test_auc": auc,
        "precision": precision_score(y_test_h, pred, zero_division=0),
        "recall": recall_score(y_test_h, pred, zero_division=0),
        "f1": f1_score(y_test_h, pred, zero_division=0),
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else 0,
        "workload_rate": (tp + fp) / len(y_test_h)
    })

    trained_models_h[model_name] = model

soft_voting_h = VotingClassifier(
    estimators=[
        ("lr", trained_models_h["logistic_regression"]),
        ("rf", trained_models_h["random_forest"]),
        ("hgb", trained_models_h["hist_gradient_boosting"])
    ],
    voting="soft",
    n_jobs=-1
)

print("\nTraining soft voting ensemble...")
soft_voting_h.fit(X_train_h, y_train_h)

proba = soft_voting_h.predict_proba(X_test_h)[:, 1]
auc = roc_auc_score(y_test_h, proba)
pred = (proba >= recommended_threshold_h).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test_h, pred, labels=[0, 1]).ravel()

model_results_h.append({
    "model_name": "soft_voting_ensemble",
    "test_auc": auc,
    "precision": precision_score(y_test_h, pred, zero_division=0),
    "recall": recall_score(y_test_h, pred, zero_division=0),
    "f1": f1_score(y_test_h, pred, zero_division=0),
    "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else 0,
    "workload_rate": (tp + fp) / len(y_test_h)
})

trained_models_h["soft_voting_ensemble"] = soft_voting_h

model_comparison_h = pd.DataFrame(model_results_h).sort_values(
    by=["test_auc", "f1"],
    ascending=[False, False]
)

best_model_name_h = model_comparison_h.iloc[0]["model_name"]
best_model_h = trained_models_h[best_model_name_h]
best_model_test_proba_h = best_model_h.predict_proba(X_test_h)[:, 1]

print("\nHospitalization model comparison:")
print(model_comparison_h)

print("\nBest model:", best_model_name_h)

model_comparison_h.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH11_model_comparison_train_sets1_2_test_set3.csv"
    ),
    index=False
)

print("\nStep H11 completed successfully.")


STEP H11 - Multi-model and ensemble comparison

Training logistic_regression...

Training random_forest...

Training hist_gradient_boosting...

Training soft voting ensemble...

Hospitalization model comparison:
               model_name  test_auc  precision    recall        f1  \
2  hist_gradient_boosting  0.838972   0.127225  0.774304  0.218542   
1           random_forest  0.835804   0.119025  0.799403  0.207199   
3    soft_voting_ensemble  0.835664   0.125841  0.773470  0.216464   
0     logistic_regression  0.795706   0.136246  0.596351  0.221814   

   false_negative_rate  workload_rate  
2             0.225696       0.280999  
1             0.200597       0.310096  
3             0.226530       0.283785  
0             0.403649       0.202091  

Best model: hist_gradient_boosting

Step H11 completed successfully.


In [ ]:
# ============================================================
# Step H12 - Probability Calibration
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H12 - Probability calibration")

if "best_model_h" not in globals():
    raise ValueError("best_model_h not found. Run H11 first.")

calibrated_model_h = CalibratedClassifierCV(
    estimator=best_model_h,
    method="isotonic",
    cv=3
)

calibrated_model_h.fit(X_train_h, y_train_h)

calibrated_test_proba_h = calibrated_model_h.predict_proba(X_test_h)[:, 1]

calibrated_auc_h = roc_auc_score(y_test_h, calibrated_test_proba_h)
calibrated_brier_h = brier_score_loss(y_test_h, calibrated_test_proba_h)

calibration_h = pd.DataFrame({
    "actual": y_test_h.values,
    "predicted_probability": calibrated_test_proba_h
})

calibration_h["probability_bin"] = pd.cut(
    calibration_h["predicted_probability"],
    bins=np.arange(0, 1.05, 0.05),
    include_lowest=True
)

calibration_summary_h = (
    calibration_h
    .groupby("probability_bin", observed=True)
    .agg(
        patient_month_count=("actual", "count"),
        actual_hospitalization_rate=("actual", "mean"),
        avg_predicted_probability=("predicted_probability", "mean")
    )
    .reset_index()
)

calibration_summary_h["calibration_gap"] = (
    calibration_summary_h["actual_hospitalization_rate"]
    - calibration_summary_h["avg_predicted_probability"]
)

print("Calibrated AUC:", round(calibrated_auc_h, 4))
print("Calibrated Brier:", round(calibrated_brier_h, 6))
print(calibration_summary_h)

calibration_summary_h.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH12_calibration_summary_train_sets1_2_test_set3.csv"
    ),
    index=False
)

print("\nStep H12 completed successfully.")


STEP H12 - Probability calibration
Calibrated AUC: 0.8389
Calibrated Brier: 0.039517
   probability_bin  patient_month_count  actual_hospitalization_rate  \
0   (-0.001, 0.05]              2605270                     0.014413   
1      (0.05, 0.1]               497279                     0.070858   
2      (0.1, 0.15]               260625                     0.120226   
3      (0.15, 0.2]               123795                     0.171695   
4      (0.2, 0.25]                62725                     0.221475   
5      (0.25, 0.3]                32734                     0.272286   
6      (0.3, 0.35]                24712                     0.328707   
7      (0.35, 0.4]                14292                     0.389938   
8      (0.4, 0.45]                 8394                     0.433167   
9      (0.45, 0.5]                 4065                     0.463961   
10     (0.5, 0.55]                  756                     0.492063   
11     (0.55, 0.6]                   80           

In [ ]:
# ============================================================
# Step H13 - Temporal Validation
# TRAIN Sets 1+2 internal temporal validation
# Train months: 2008-2009
# Validate months: 2010
# ============================================================

print("\nSTEP H13 - Temporal validation")

df_temporal = df_train_timeline.copy()
df_temporal["timeline_year"] = df_temporal[PREDICTION_DATE_COL].dt.year

train_mask_h13 = df_temporal["timeline_year"].isin(TRAIN_YEARS)
valid_mask_h13 = df_temporal["timeline_year"].eq(TEST_YEAR)

X_temporal_train = X_train_h.loc[train_mask_h13].copy()
y_temporal_train = y_train_h.loc[train_mask_h13].copy()

X_temporal_valid = X_train_h.loc[valid_mask_h13].copy()
y_temporal_valid = y_train_h.loc[valid_mask_h13].copy()

temporal_model_h = HistGradientBoostingClassifier(
    **HOSPITALIZATION_MODEL_PARAMS
)

temporal_model_h.fit(X_temporal_train, y_temporal_train)

temporal_valid_proba_h = temporal_model_h.predict_proba(X_temporal_valid)[:, 1]

temporal_auc_h = roc_auc_score(y_temporal_valid, temporal_valid_proba_h)

temporal_pred_h = (temporal_valid_proba_h >= recommended_threshold_h).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_temporal_valid,
    temporal_pred_h,
    labels=[0, 1]
).ravel()

temporal_summary_h = pd.DataFrame({
    "metric": [
        "temporal_train_rows",
        "temporal_validation_rows",
        "temporal_validation_auc",
        "precision",
        "recall",
        "f1",
        "false_negative_rate",
        "workload_rate"
    ],
    "value": [
        len(X_temporal_train),
        len(X_temporal_valid),
        temporal_auc_h,
        precision_score(y_temporal_valid, temporal_pred_h, zero_division=0),
        recall_score(y_temporal_valid, temporal_pred_h, zero_division=0),
        f1_score(y_temporal_valid, temporal_pred_h, zero_division=0),
        fn / (fn + tp) if (fn + tp) > 0 else 0,
        (tp + fp) / len(y_temporal_valid)
    ]
})

print(temporal_summary_h)

temporal_summary_h.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH13_temporal_validation_summary_train_sets1_2.csv"
    ),
    index=False
)

print("\nStep H13 completed successfully.")


STEP H13 - Temporal validation
                     metric         value
0       temporal_train_rows  4.843776e+06
1  temporal_validation_rows  2.421888e+06
2   temporal_validation_auc  8.127761e-01
3                 precision  8.269860e-02
4                    recall  6.505791e-01
5                        f1  1.467438e-01
6       false_negative_rate  3.494209e-01
7             workload_rate  2.167842e-01

Step H13 completed successfully.


In [ ]:
# ============================================================
# Step H14 - Intervention Opportunity Analysis
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H14 - Intervention opportunity analysis")

intervention_df_h = df_test_timeline.copy()
intervention_df_h["actual"] = y_test_h.values

if "calibrated_test_proba_h" in globals():
    intervention_df_h["predicted_probability"] = calibrated_test_proba_h
else:
    intervention_df_h["predicted_probability"] = test_proba_h

intervention_df_h["intervention_tier"] = pd.cut(
    intervention_df_h["predicted_probability"],
    bins=[
        -0.001,
        WATCHLIST_THRESHOLD,
        CARE_MANAGEMENT_THRESHOLD,
        HIGH_RISK_THRESHOLD,
        0.90,
        1.001
    ],
    labels=[
        "LOW_RISK",
        "WATCHLIST",
        "CARE_MANAGEMENT",
        "HIGH_RISK",
        "CRITICAL_RISK"
    ]
)

intervention_signal_cols = [
    "rx_claims_30d",
    "rx_claims_90d",
    "unique_drugs_90d",
    "rx_days_supply_90d",
    "rx_claim_growth_30_vs_90",
    "high_polypharmacy_flag_90d",
    "prior_ip_claims_90d",
    "prior_ip_claims_365d",
    "days_since_last_ip",
    "ever_hospitalized_before",
    "chronic_count",
    "high_risk_patient"
]

intervention_signal_cols = [
    c for c in intervention_signal_cols
    if c in intervention_df_h.columns
]

intervention_summary_h = (
    intervention_df_h
    .groupby("intervention_tier", observed=True)
    .agg(
        patient_month_count=("member_id", "count"),
        unique_members=("member_id", "nunique"),
        actual_hospitalization_rate=("actual", "mean"),
        avg_predicted_probability=("predicted_probability", "mean")
    )
    .reset_index()
)

for col in intervention_signal_cols:
    temp = (
        intervention_df_h
        .groupby("intervention_tier", observed=True)[col]
        .mean()
        .reset_index()
        .rename(columns={col: f"avg_{col}"})
    )

    intervention_summary_h = intervention_summary_h.merge(
        temp,
        on="intervention_tier",
        how="left"
    )

print("\nIntervention opportunity summary:")
print(intervention_summary_h)

intervention_summary_h.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH14_intervention_opportunity_summary_train_sets1_2_test_set3.csv"
    ),
    index=False
)

print("\nStep H14 completed successfully.")


STEP H14 - Intervention opportunity analysis

Intervention opportunity summary:
  intervention_tier  patient_month_count  unique_members  \
0          LOW_RISK              3582428          100965   
1         WATCHLIST                51463            8656   
2   CARE_MANAGEMENT                  848             576   
3         HIGH_RISK                    1               1   

   actual_hospitalization_rate  avg_predicted_probability  avg_rx_claims_30d  \
0                     0.041363                   0.041438           1.506746   
1                     0.373433                   0.363718           2.203855   
2                     0.497642                   0.523542           2.468160   
3                     0.000000                   0.763206           2.000000   

   avg_rx_claims_90d  avg_unique_drugs_90d  avg_rx_days_supply_90d  \
0           4.479365              4.479270              153.242407   
1           5.970484              5.970425              188.242232   
2      

In [ ]:
# ============================================================
# Step H15 - Hospitalization Pathway Mining
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H15 - Hospitalization pathway mining")

pathway_df_h = df_test_timeline.copy()
pathway_df_h["actual"] = y_test_h.values

if "calibrated_test_proba_h" in globals():
    pathway_df_h["predicted_probability"] = calibrated_test_proba_h
else:
    pathway_df_h["predicted_probability"] = test_proba_h

pathway_df_h["rx_escalation_flag"] = (
    pathway_df_h.get("rx_claim_growth_30_vs_90", 0) >= 0.75
).astype(int)

pathway_df_h["polypharmacy_flag"] = (
    pathway_df_h.get("high_polypharmacy_flag_90d", 0) == 1
).astype(int)

pathway_df_h["recent_ip_history_flag"] = (
    pathway_df_h.get("prior_ip_claims_365d", 0) >= 1
).astype(int)

pathway_df_h["chronic_burden_flag"] = (
    pathway_df_h.get("chronic_count", 0) >= 3
).astype(int)

pathway_df_h["pathway_pattern"] = (
    "RX_ESC_" + pathway_df_h["rx_escalation_flag"].astype(str)
    + "_POLY_" + pathway_df_h["polypharmacy_flag"].astype(str)
    + "_PRIORIP_" + pathway_df_h["recent_ip_history_flag"].astype(str)
    + "_CHRONIC_" + pathway_df_h["chronic_burden_flag"].astype(str)
)

pathway_summary_h = (
    pathway_df_h
    .groupby("pathway_pattern")
    .agg(
        patient_month_count=("member_id", "count"),
        unique_members=("member_id", "nunique"),
        actual_hospitalization_rate=("actual", "mean"),
        avg_predicted_probability=("predicted_probability", "mean"),
        avg_rx_claims_90d=("rx_claims_90d", "mean") if "rx_claims_90d" in pathway_df_h.columns else ("actual", "mean"),
        avg_unique_drugs_90d=("unique_drugs_90d", "mean") if "unique_drugs_90d" in pathway_df_h.columns else ("actual", "mean"),
        avg_prior_ip_claims_365d=("prior_ip_claims_365d", "mean") if "prior_ip_claims_365d" in pathway_df_h.columns else ("actual", "mean"),
        avg_chronic_count=("chronic_count", "mean") if "chronic_count" in pathway_df_h.columns else ("actual", "mean")
    )
    .reset_index()
    .sort_values(
        by=["actual_hospitalization_rate", "patient_month_count"],
        ascending=[False, False]
    )
)

print("\nTop hospitalization pathway patterns:")
print(pathway_summary_h.head(30))

pathway_summary_h.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "stepH15_hospitalization_pathway_summary_train_sets1_2_test_set3.csv"
    ),
    index=False
)

print("\nStep H15 completed successfully.")


STEP H15 - Hospitalization pathway mining

Top hospitalization pathway patterns:
                        pathway_pattern  patient_month_count  unique_members  \
15  RX_ESC_1_POLY_1_PRIORIP_1_CHRONIC_1                 2156            2047   
7   RX_ESC_0_POLY_1_PRIORIP_1_CHRONIC_1               188357           18003   
11  RX_ESC_1_POLY_0_PRIORIP_1_CHRONIC_1                17576           11517   
3   RX_ESC_0_POLY_0_PRIORIP_1_CHRONIC_1               189186           23755   
13  RX_ESC_1_POLY_1_PRIORIP_0_CHRONIC_1                11919           11094   
9   RX_ESC_1_POLY_0_PRIORIP_0_CHRONIC_1                59899           40020   
5   RX_ESC_0_POLY_1_PRIORIP_0_CHRONIC_1               476751           34901   
1   RX_ESC_0_POLY_0_PRIORIP_0_CHRONIC_1               615560           59497   
6   RX_ESC_0_POLY_1_PRIORIP_1_CHRONIC_0                38273            6769   
14  RX_ESC_1_POLY_1_PRIORIP_1_CHRONIC_0                  482             466   
10  RX_ESC_1_POLY_0_PRIORIP_1_CHRONIC_

In [ ]:
# ============================================================
# Step H16 - Explainable Patient Timelines
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H16 - Explainable patient timelines")

timeline_explain_df_h = df_test_timeline.copy()
timeline_explain_df_h["actual"] = y_test_h.values

if "calibrated_test_proba_h" in globals():
    timeline_explain_df_h["predicted_probability"] = calibrated_test_proba_h
else:
    timeline_explain_df_h["predicted_probability"] = test_proba_h

timeline_explain_df_h["hospitalization_risk_tier"] = pd.cut(
    timeline_explain_df_h["predicted_probability"],
    bins=[
        -0.001,
        WATCHLIST_THRESHOLD,
        CARE_MANAGEMENT_THRESHOLD,
        HIGH_RISK_THRESHOLD,
        0.90,
        1.001
    ],
    labels=[
        "LOW_RISK",
        "WATCHLIST",
        "CARE_MANAGEMENT",
        "HIGH_RISK",
        "CRITICAL_RISK"
    ]
)

top_members_h = (
    timeline_explain_df_h
    .sort_values("predicted_probability", ascending=False)
    .head(100)["member_id"]
    .unique()
)

timeline_cols_h = [
    "member_id",
    PREDICTION_DATE_COL,
    "predicted_probability",
    "hospitalization_risk_tier",
    "actual",
    "rx_claims_30d",
    "rx_claims_90d",
    "unique_drugs_90d",
    "rx_days_supply_90d",
    "rx_claim_growth_30_vs_90",
    "high_polypharmacy_flag_90d",
    "prior_ip_claims_90d",
    "prior_ip_claims_365d",
    "days_since_last_ip",
    "ever_hospitalized_before",
    "chronic_count",
    "high_risk_patient"
]

timeline_cols_h = [
    c for c in timeline_cols_h
    if c in timeline_explain_df_h.columns
]

patient_timeline_explanations_h = (
    timeline_explain_df_h[
        timeline_explain_df_h["member_id"].isin(top_members_h)
    ][timeline_cols_h]
    .sort_values(["member_id", PREDICTION_DATE_COL])
)

timeline_file_h = os.path.join(
    OUTPUT_FOLDER,
    "stepH16_explainable_patient_timelines_train_sets1_2_test_set3.xlsx"
)

patient_timeline_explanations_h.to_excel(
    timeline_file_h,
    index=False
)

print("\nStep H16 completed successfully.")
print("Saved:", timeline_file_h)


STEP H16 - Explainable patient timelines

Step H16 completed successfully.
Saved: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Output/stepH16_explainable_patient_timelines_train_sets1_2_test_set3.xlsx


In [ ]:
# ============================================================
# Step H17 - Feedback Learning System
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H17 - Feedback learning system")

feedback_candidates_h = timeline_explain_df_h.copy()

feedback_candidates_h = (
    feedback_candidates_h
    .sort_values("predicted_probability", ascending=False)
    .head(500)
)

feedback_cols_h = [
    "member_id",
    PREDICTION_DATE_COL,
    "predicted_probability",
    "hospitalization_risk_tier",
    "actual",
    "rx_claims_90d",
    "unique_drugs_90d",
    "prior_ip_claims_365d",
    "days_since_last_ip",
    "chronic_count",
    "high_risk_patient"
]

feedback_cols_h = [
    c for c in feedback_cols_h
    if c in feedback_candidates_h.columns
]

feedback_template_h = feedback_candidates_h[feedback_cols_h].copy()

feedback_template_h["reviewer_role"] = ""
feedback_template_h["reviewer_name"] = ""
feedback_template_h["review_date"] = ""
feedback_template_h["review_decision"] = ""
feedback_template_h["intervention_recommended"] = ""
feedback_template_h["intervention_type"] = ""
feedback_template_h["avoidable_hospitalization_opportunity"] = ""
feedback_template_h["clinical_reason_code"] = ""
feedback_template_h["operational_reason_code"] = ""
feedback_template_h["notes"] = ""

feedback_file_h = os.path.join(
    OUTPUT_FOLDER,
    "stepH17_hospitalization_feedback_template_train_sets1_2_test_set3.xlsx"
)

feedback_template_h.to_excel(
    feedback_file_h,
    index=False
)

print("\nStep H17 completed successfully.")
print("Saved:", feedback_file_h)


STEP H17 - Feedback learning system

Step H17 completed successfully.
Saved: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Output/stepH17_hospitalization_feedback_template_train_sets1_2_test_set3.xlsx


In [ ]:
# ============================================================
# Step H18 - Real-Time Surveillance Scoring Layer
# TRAIN Sets 1+2, TEST Set 3
# ============================================================

print("\nSTEP H18 - Real-time surveillance scoring design")

surveillance_scoring_schema_h = pd.DataFrame({
    "input_domain": [
        "New Rx/PDE fill",
        "New professional claim",
        "New outpatient event",
        "New inpatient event",
        "Beneficiary update",
        "Reviewer feedback",
        "Care management intervention",
        "Medical director override"
    ],
    "refresh_action": [
        "Recalculate Rx lookback windows, drug diversity, polypharmacy, and medication escalation signals.",
        "Update outpatient utilization, diagnosis progression, provider-fragmentation, and professional-risk features.",
        "Update outpatient utilization trajectory and ED-like escalation signals.",
        "Update prior inpatient history, readmission risk, and post-discharge monitoring windows.",
        "Update age, chronic burden, demographics, and beneficiary clinical profile.",
        "Update human-confirmed labels and retraining readiness queue.",
        "Track whether outreach, medication review, or care plan change occurred after model alert.",
        "Capture physician judgment when model risk tier should be overridden."
    ],
    "scoring_impact": [
        "May raise/lower hospitalization risk based on medication trajectory.",
        "May detect emerging deterioration before admission.",
        "May identify acute escalation before inpatient event.",
        "May increase readmission or recurrence risk.",
        "May adjust baseline clinical risk.",
        "Improves future supervised learning.",
        "Supports intervention ROI and preventability analysis.",
        "Improves governance and clinical trust."
    ],
    "recommended_refresh_frequency": [
        "Daily or weekly",
        "Daily or weekly",
        "Daily or weekly",
        "Daily",
        "Monthly or as updated",
        "After review completion",
        "After intervention documentation",
        "After medical director review"
    ]
})

surveillance_file_h = os.path.join(
    OUTPUT_FOLDER,
    "stepH18_real_time_surveillance_scoring_design_train_sets1_2_test_set3.xlsx"
)

surveillance_scoring_schema_h.to_excel(
    surveillance_file_h,
    index=False
)

print("\nStep H18 completed successfully.")
print("Saved:", surveillance_file_h)


STEP H18 - Real-time surveillance scoring design

Step H18 completed successfully.
Saved: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Output/stepH18_real_time_surveillance_scoring_design_train_sets1_2_test_set3.xlsx


In [ ]:
# ============================================================
# Phase 2 - Hospitalization Pathway / Progression Intelligence
# Step P0 - Configuration
# ============================================================

print("\nPHASE 2 - Initializing hospitalization pathway intelligence")

PHASE2_OUTPUT_FOLDER = os.path.join(
    BASE_FOLDER,
    "Phase2_Pathway_Intelligence"
)

os.makedirs(PHASE2_OUTPUT_FOLDER, exist_ok=True)

PHASE2_LOOKBACK_MONTHS = [1, 3, 6, 12]
PHASE2_TARGET = TARGET_LABEL_NAME

MIN_PATHWAY_ROWS = 500
HIGH_RISK_PATHWAY_RATE = 0.20
PATHWAY_LIFT_THRESHOLD = 2.0

print("PHASE2_OUTPUT_FOLDER:", PHASE2_OUTPUT_FOLDER)
print("Target:", PHASE2_TARGET)
print("Step P0 completed.")


PHASE 2 - Initializing hospitalization pathway intelligence
PHASE2_OUTPUT_FOLDER: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Phase2_Pathway_Intelligence
Target: hospitalized_next_90d
Step P0 completed.


In [ ]:
# ============================================================
# Step P1 - Chronic Condition Combination Engine
# ============================================================

print("\nSTEP P1 - Building chronic condition combination profiles")

def build_chronic_profile(df, dataset_label):

    df = df.copy()

    chronic_cols_available = [
        c for c in CHRONIC_CONDITION_COLUMNS
        if c in df.columns
    ]

    for col in chronic_cols_available:
        df[col] = safe_numeric(df[col], fill_value=0).astype(int)

    df["chronic_profile"] = ""

    for col in chronic_cols_available:
        short_name = col.replace("SP_", "")
        df["chronic_profile"] += np.where(
            df[col] == 1,
            short_name + "|",
            ""
        )

    df["chronic_profile"] = df["chronic_profile"].replace("", "NO_MAJOR_CHRONIC")
    df["chronic_profile"] = df["chronic_profile"].str.rstrip("|")

    df["chronic_profile_simple"] = np.where(
        df["chronic_count"] >= 5,
        "HIGH_MULTI_CHRONIC",
        np.where(
            df["chronic_count"] >= 3,
            "MODERATE_MULTI_CHRONIC",
            np.where(
                df["chronic_count"] >= 1,
                "LOW_CHRONIC",
                "NO_CHRONIC"
            )
        )
    )

    print(f"{dataset_label} chronic profile created.")
    print(df["chronic_profile_simple"].value_counts(dropna=False))

    return df


df_train_timeline = build_chronic_profile(df_train_timeline, "TRAIN Sets 1+2")
df_test_timeline = build_chronic_profile(df_test_timeline, "TEST Set 3")

print("Step P1 completed.")


STEP P1 - Building chronic condition combination profiles
TRAIN Sets 1+2 chronic profile created.
chronic_profile_simple
NO_CHRONIC                2184252
LOW_CHRONIC               1966464
MODERATE_MULTI_CHRONIC    1595136
HIGH_MULTI_CHRONIC        1519812
Name: count, dtype: int64
TEST Set 3 chronic profile created.
chronic_profile_simple
NO_CHRONIC                1094232
LOW_CHRONIC                979104
MODERATE_MULTI_CHRONIC     801972
HIGH_MULTI_CHRONIC         759432
Name: count, dtype: int64
Step P1 completed.


In [ ]:
# ============================================================
# Step P2 - Rx Progression / Escalation Engine
# ============================================================

print("\nSTEP P2 - Building Rx progression features")

def add_rx_progression_features(df, dataset_label):

    df = df.copy()

    required_cols = [
        "rx_claims_30d", "rx_claims_90d", "rx_claims_180d", "rx_claims_365d",
        "unique_drugs_30d", "unique_drugs_90d", "unique_drugs_180d", "unique_drugs_365d",
        "rx_days_supply_30d", "rx_days_supply_90d", "rx_days_supply_180d", "rx_days_supply_365d",
        "rx_total_cost_30d", "rx_total_cost_90d", "rx_total_cost_180d", "rx_total_cost_365d"
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = safe_numeric(df[col], fill_value=0)

    df["rx_claims_velocity_30_vs_90"] = (
        df["rx_claims_30d"] /
        (df["rx_claims_90d"] / 3).replace(0, np.nan)
    )

    df["drug_velocity_30_vs_90"] = (
        df["unique_drugs_30d"] /
        (df["unique_drugs_90d"] / 3).replace(0, np.nan)
    )

    df["days_supply_velocity_30_vs_90"] = (
        df["rx_days_supply_30d"] /
        (df["rx_days_supply_90d"] / 3).replace(0, np.nan)
    )

    df["rx_cost_velocity_30_vs_90"] = (
        df["rx_total_cost_30d"] /
        (df["rx_total_cost_90d"] / 3).replace(0, np.nan)
    )

    df["rx_escalation_score"] = (
        0.30 * df["rx_claims_velocity_30_vs_90"] +
        0.25 * df["drug_velocity_30_vs_90"] +
        0.25 * df["days_supply_velocity_30_vs_90"] +
        0.20 * df["rx_cost_velocity_30_vs_90"]
    )

    df["rx_escalation_score"] = (
        df["rx_escalation_score"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(lower=0, upper=10)
    )

    df["rx_escalation_flag"] = (
        df["rx_escalation_score"] >= 1.50
    ).astype(int)

    df["polypharmacy_progression_flag"] = (
        (df["unique_drugs_90d"] >= 5) &
        (df["drug_velocity_30_vs_90"] >= 1.25)
    ).astype(int)

    print(f"{dataset_label} Rx progression features created.")
    print(df[["rx_escalation_score", "rx_escalation_flag", "polypharmacy_progression_flag"]].describe())

    return df


df_train_timeline = add_rx_progression_features(df_train_timeline, "TRAIN Sets 1+2")
df_test_timeline = add_rx_progression_features(df_test_timeline, "TEST Set 3")

print("Step P2 completed.")


STEP P2 - Building Rx progression features
TRAIN Sets 1+2 Rx progression features created.
       rx_escalation_score  rx_escalation_flag  polypharmacy_progression_flag
count         7.265664e+06        7.265664e+06                   7.265664e+06
mean          6.375409e-01        1.429572e-01                   1.129437e-01
std           8.220053e-01        3.500292e-01                   3.165240e-01
min           0.000000e+00        0.000000e+00                   0.000000e+00
25%           0.000000e+00        0.000000e+00                   0.000000e+00
50%           0.000000e+00        0.000000e+00                   0.000000e+00
75%           1.120833e+00        0.000000e+00                   0.000000e+00
max           3.000000e+00        1.000000e+00                   1.000000e+00
TEST Set 3 Rx progression features created.
       rx_escalation_score  rx_escalation_flag  polypharmacy_progression_flag
count         3.634740e+06        3.634740e+06                   3.634740e+06
mean  

In [ ]:
# ============================================================
# Step P3 - Prior Inpatient Deterioration / Recurrence Engine
# ============================================================

print("\nSTEP P3 - Building inpatient recurrence and deterioration features")

def add_ip_progression_features(df, dataset_label):

    df = df.copy()

    ip_cols = [
        "prior_ip_claims_30d", "prior_ip_claims_90d", "prior_ip_claims_180d", "prior_ip_claims_365d",
        "prior_ip_los_30d", "prior_ip_los_90d", "prior_ip_los_180d", "prior_ip_los_365d",
        "days_since_last_ip", "ever_hospitalized_before"
    ]

    for col in ip_cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = safe_numeric(df[col], fill_value=0)

    df["ip_recurrence_velocity_90_vs_365"] = (
        df["prior_ip_claims_90d"] /
        (df["prior_ip_claims_365d"] / 4).replace(0, np.nan)
    )

    df["ip_los_velocity_90_vs_365"] = (
        df["prior_ip_los_90d"] /
        (df["prior_ip_los_365d"] / 4).replace(0, np.nan)
    )

    df["recent_post_discharge_window_flag"] = (
        df["days_since_last_ip"] <= 90
    ).astype(int)

    df["very_recent_discharge_flag"] = (
        df["days_since_last_ip"] <= 30
    ).astype(int)

    df["ip_recurrence_risk_score"] = (
        0.40 * df["ever_hospitalized_before"] +
        0.25 * df["recent_post_discharge_window_flag"] +
        0.20 * df["ip_recurrence_velocity_90_vs_365"].replace([np.inf, -np.inf], 0).fillna(0) +
        0.15 * df["ip_los_velocity_90_vs_365"].replace([np.inf, -np.inf], 0).fillna(0)
    )

    df["ip_recurrence_risk_score"] = (
        df["ip_recurrence_risk_score"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(lower=0, upper=10)
    )

    print(f"{dataset_label} IP progression features created.")
    print(df[["ip_recurrence_risk_score", "recent_post_discharge_window_flag", "very_recent_discharge_flag"]].describe())

    return df


df_train_timeline = add_ip_progression_features(df_train_timeline, "TRAIN Sets 1+2")
df_test_timeline = add_ip_progression_features(df_test_timeline, "TEST Set 3")

print("Step P3 completed.")


STEP P3 - Building inpatient recurrence and deterioration features
TRAIN Sets 1+2 IP progression features created.
       ip_recurrence_risk_score  recent_post_discharge_window_flag  \
count              7.265664e+06                       7.265664e+06   
mean               1.484350e-01                       3.555972e-02   
std                3.957984e-01                       1.851897e-01   
min                0.000000e+00                       0.000000e+00   
25%                0.000000e+00                       0.000000e+00   
50%                0.000000e+00                       0.000000e+00   
75%                0.000000e+00                       0.000000e+00   
max                2.050000e+00                       1.000000e+00   

       very_recent_discharge_flag  
count                7.265664e+06  
mean                 7.037898e-03  
std                  8.359645e-02  
min                  0.000000e+00  
25%                  0.000000e+00  
50%                  0.000000e+00  
7

In [ ]:
# ============================================================
# Step P4 - Hospitalization Pathway Signature Builder
# ============================================================

print("\nSTEP P4 - Building pathway signatures")

def build_pathway_signatures(df, dataset_label):

    df = df.copy()

    required_cols = [
        "chronic_profile_simple",
        "rx_escalation_flag",
        "polypharmacy_progression_flag",
        "recent_post_discharge_window_flag",
        "very_recent_discharge_flag",
        "high_risk_patient"
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = 0

    df["pathway_signature"] = (
        "CHRONIC_" + df["chronic_profile_simple"].astype(str)
        + "_RXESC_" + df["rx_escalation_flag"].astype(str)
        + "_POLYPROG_" + df["polypharmacy_progression_flag"].astype(str)
        + "_POSTDC90_" + df["recent_post_discharge_window_flag"].astype(str)
        + "_POSTDC30_" + df["very_recent_discharge_flag"].astype(str)
        + "_HRP_" + df["high_risk_patient"].astype(str)
    )

    print(f"{dataset_label} pathway signatures created.")
    print("Unique pathway signatures:", df["pathway_signature"].nunique())

    return df


df_train_timeline = build_pathway_signatures(df_train_timeline, "TRAIN Sets 1+2")
df_test_timeline = build_pathway_signatures(df_test_timeline, "TEST Set 3")

print("Step P4 completed.")


STEP P4 - Building pathway signatures
TRAIN Sets 1+2 pathway signatures created.
Unique pathway signatures: 48
TEST Set 3 pathway signatures created.
Unique pathway signatures: 48
Step P4 completed.


In [ ]:
# ============================================================
# Step P5 - Train-Only Pathway Probability Table
# ============================================================

print("\nSTEP P5 - Building train-only pathway probability table")

if PHASE2_TARGET not in df_train_timeline.columns:
    raise ValueError(f"{PHASE2_TARGET} not found in df_train_timeline.")

global_hosp_rate_p2 = df_train_timeline[PHASE2_TARGET].mean()

pathway_probability_table = (
    df_train_timeline
    .groupby("pathway_signature")
    .agg(
        pathway_patient_months=("member_id", "count"),
        pathway_unique_members=("member_id", "nunique"),
        pathway_hospitalization_rate=(PHASE2_TARGET, "mean"),
        avg_chronic_count=("chronic_count", "mean"),
        avg_rx_escalation_score=("rx_escalation_score", "mean"),
        avg_ip_recurrence_risk_score=("ip_recurrence_risk_score", "mean")
    )
    .reset_index()
)

pathway_probability_table["pathway_lift_vs_global"] = (
    pathway_probability_table["pathway_hospitalization_rate"] /
    max(global_hosp_rate_p2, 0.000001)
)

pathway_probability_table["high_risk_pathway_flag"] = (
    (pathway_probability_table["pathway_patient_months"] >= MIN_PATHWAY_ROWS) &
    (pathway_probability_table["pathway_hospitalization_rate"] >= HIGH_RISK_PATHWAY_RATE) &
    (pathway_probability_table["pathway_lift_vs_global"] >= PATHWAY_LIFT_THRESHOLD)
).astype(int)

pathway_probability_table = pathway_probability_table.sort_values(
    by=["pathway_hospitalization_rate", "pathway_patient_months"],
    ascending=[False, False]
)

print("\nTop hospitalization pathways:")
print(pathway_probability_table.head(25))

pathway_probability_table.to_csv(
    os.path.join(
        PHASE2_OUTPUT_FOLDER,
        "stepP5_train_only_pathway_probability_table.csv"
    ),
    index=False
)

print("\nGlobal hospitalization rate:", global_hosp_rate_p2)
print("Step P5 completed.")


STEP P5 - Building train-only pathway probability table

Top hospitalization pathways:
                                    pathway_signature  pathway_patient_months  \
11  CHRONIC_HIGH_MULTI_CHRONIC_RXESC_1_POLYPROG_1_...                    2764   
8   CHRONIC_HIGH_MULTI_CHRONIC_RXESC_1_POLYPROG_0_...                    2396   
5   CHRONIC_HIGH_MULTI_CHRONIC_RXESC_0_POLYPROG_1_...                    2312   
2   CHRONIC_HIGH_MULTI_CHRONIC_RXESC_0_POLYPROG_0_...                   26190   
10  CHRONIC_HIGH_MULTI_CHRONIC_RXESC_1_POLYPROG_1_...                   10094   
7   CHRONIC_HIGH_MULTI_CHRONIC_RXESC_1_POLYPROG_0_...                    8639   
4   CHRONIC_HIGH_MULTI_CHRONIC_RXESC_0_POLYPROG_1_...                    9280   
1   CHRONIC_HIGH_MULTI_CHRONIC_RXESC_0_POLYPROG_0_...                   98673   
6   CHRONIC_HIGH_MULTI_CHRONIC_RXESC_1_POLYPROG_0_...                   88311   
9   CHRONIC_HIGH_MULTI_CHRONIC_RXESC_1_POLYPROG_1_...                  100211   
3   CHRONIC_HIGH_MULT

In [ ]:
# ============================================================
# Step P6 - Apply Train-Only Pathway Probability Features
# ============================================================

print("\nSTEP P6 - Applying pathway probability features")

pathway_lookup = pathway_probability_table[
    [
        "pathway_signature",
        "pathway_patient_months",
        "pathway_hospitalization_rate",
        "pathway_lift_vs_global",
        "high_risk_pathway_flag"
    ]
].copy()

def apply_pathway_features(df, dataset_label):

    df = df.copy()

    df = df.merge(
        pathway_lookup,
        on="pathway_signature",
        how="left"
    )

    df["pathway_patient_months"] = safe_numeric(
        df["pathway_patient_months"],
        fill_value=0
    )

    df["pathway_hospitalization_rate"] = safe_numeric(
        df["pathway_hospitalization_rate"],
        fill_value=global_hosp_rate_p2
    )

    df["pathway_lift_vs_global"] = safe_numeric(
        df["pathway_lift_vs_global"],
        fill_value=1
    )

    df["high_risk_pathway_flag"] = safe_numeric(
        df["high_risk_pathway_flag"],
        fill_value=0
    ).astype(int)

    print(f"{dataset_label} pathway features applied.")
    print(
        df[
            [
                "pathway_hospitalization_rate",
                "pathway_lift_vs_global",
                "high_risk_pathway_flag"
            ]
        ].describe()
    )

    return df


df_train_timeline = apply_pathway_features(
    df_train_timeline,
    "TRAIN Sets 1+2"
)

df_test_timeline = apply_pathway_features(
    df_test_timeline,
    "TEST Set 3"
)

print("Step P6 completed.")


STEP P6 - Applying pathway probability features
TRAIN Sets 1+2 pathway features applied.
       pathway_hospitalization_rate  pathway_lift_vs_global  \
count                  7.265664e+06            7.265664e+06   
mean                   4.612545e-02            1.000000e+00   
std                    5.322060e-02            1.153823e+00   
min                    0.000000e+00            0.000000e+00   
25%                    2.349468e-03            5.093649e-02   
50%                    1.944967e-02            4.216689e-01   
75%                    4.942095e-02            1.071447e+00   
max                    2.995658e-01            6.494590e+00   

       high_risk_pathway_flag  
count            7.265664e+06  
mean             2.206928e-02  
std              1.469089e-01  
min              0.000000e+00  
25%              0.000000e+00  
50%              0.000000e+00  
75%              0.000000e+00  
max              1.000000e+00  
TEST Set 3 pathway features applied.
       pathway_ho

In [ ]:
# ============================================================
# Step P7 - Temporal Deterioration Score
# ============================================================

print("\nSTEP P7 - Building temporal deterioration score")

def add_deterioration_score(df, dataset_label):

    df = df.copy()

    required_cols = [
        "chronic_count",
        "rx_escalation_score",
        "ip_recurrence_risk_score",
        "pathway_lift_vs_global",
        "high_risk_pathway_flag",
        "days_since_last_ip"
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = safe_numeric(df[col], fill_value=0)

    df["chronic_burden_norm"] = (
        df["chronic_count"] / max(df["chronic_count"].max(), 1)
    )

    df["rx_escalation_norm"] = (
        df["rx_escalation_score"] / max(df["rx_escalation_score"].max(), 1)
    )

    df["ip_recurrence_norm"] = (
        df["ip_recurrence_risk_score"] / max(df["ip_recurrence_risk_score"].max(), 1)
    )

    df["pathway_lift_norm"] = (
        df["pathway_lift_vs_global"] / max(df["pathway_lift_vs_global"].quantile(0.99), 1)
    ).clip(0, 1)

    df["recent_ip_pressure"] = np.where(
        df["days_since_last_ip"] <= 90,
        1,
        0
    )

    df["temporal_deterioration_score"] = (
        0.30 * df["chronic_burden_norm"] +
        0.20 * df["rx_escalation_norm"] +
        0.20 * df["ip_recurrence_norm"] +
        0.20 * df["pathway_lift_norm"] +
        0.10 * df["recent_ip_pressure"]
    )

    df["temporal_deterioration_score"] = (
        df["temporal_deterioration_score"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(0, 1)
    )

    df["deterioration_tier"] = pd.cut(
        df["temporal_deterioration_score"],
        bins=[-0.001, 0.20, 0.40, 0.60, 0.80, 1.001],
        labels=[
            "STABLE",
            "WATCH",
            "RISING_RISK",
            "HIGH_RISK",
            "CRITICAL"
        ]
    )

    print(f"{dataset_label} deterioration score created.")
    print(df["deterioration_tier"].value_counts(dropna=False))
    print(df["temporal_deterioration_score"].describe())

    return df


df_train_timeline = add_deterioration_score(
    df_train_timeline,
    "TRAIN Sets 1+2"
)

df_test_timeline = add_deterioration_score(
    df_test_timeline,
    "TEST Set 3"
)

print("Step P7 completed.")


STEP P7 - Building temporal deterioration score
TRAIN Sets 1+2 deterioration score created.
deterioration_tier
STABLE         4858226
WATCH          1912951
RISING_RISK     335908
HIGH_RISK       141878
CRITICAL         16701
Name: count, dtype: int64
count    7.265664e+06
mean     1.668627e-01
std      1.498389e-01
min      1.299243e-03
25%      5.220311e-02
50%      1.242485e-01
75%      2.429914e-01
max      1.000000e+00
Name: temporal_deterioration_score, dtype: float64
TEST Set 3 deterioration score created.
deterioration_tier
STABLE         2429033
WATCH           958872
RISING_RISK     167501
HIGH_RISK        71078
CRITICAL          8256
Name: count, dtype: int64
count    3.634740e+06
mean     1.668646e-01
std      1.497928e-01
min      1.299243e-03
25%      5.201424e-02
50%      1.242485e-01
75%      2.429914e-01
max      1.000000e+00
Name: temporal_deterioration_score, dtype: float64
Step P7 completed.


In [ ]:
# ============================================================
# Step P8 - Phase 2 Sequence-Aware Model
# ============================================================

print("\nSTEP P8 - Training Phase 2 pathway-enhanced model")

phase2_features = safe_features_h.copy()

additional_phase2_features = [
    "rx_escalation_score",
    "rx_escalation_flag",
    "polypharmacy_progression_flag",
    "ip_recurrence_velocity_90_vs_365",
    "ip_los_velocity_90_vs_365",
    "recent_post_discharge_window_flag",
    "very_recent_discharge_flag",
    "ip_recurrence_risk_score",
    "pathway_patient_months",
    "pathway_hospitalization_rate",
    "pathway_lift_vs_global",
    "high_risk_pathway_flag",
    "temporal_deterioration_score",
    "recent_ip_pressure"
]

phase2_features = list(dict.fromkeys(
    phase2_features + [
        c for c in additional_phase2_features
        if c in df_train_timeline.columns
        and c in df_test_timeline.columns
    ]
))

phase2_leakage_terms = [
    "future_",
    "hospitalized_next_",
    "admission_date",
    "discharge_date",
    "post_prediction"
]

phase2_features = [
    c for c in phase2_features
    if not any(term in c.lower() for term in phase2_leakage_terms)
]

X_train_p2 = df_train_timeline[phase2_features].copy()
X_test_p2 = df_test_timeline[phase2_features].copy()

y_train_p2 = df_train_timeline[PHASE2_TARGET].astype(int)
y_test_p2 = df_test_timeline[PHASE2_TARGET].astype(int)

for col in phase2_features:
    X_train_p2[col] = safe_numeric(X_train_p2[col], fill_value=0)
    X_test_p2[col] = safe_numeric(X_test_p2[col], fill_value=0)

X_train_p2 = X_train_p2.replace([np.inf, -np.inf], 0).fillna(0)
X_test_p2 = X_test_p2.replace([np.inf, -np.inf], 0).fillna(0)

phase2_model = HistGradientBoostingClassifier(
    **HOSPITALIZATION_MODEL_PARAMS
)

phase2_model.fit(X_train_p2, y_train_p2)

train_proba_p2 = phase2_model.predict_proba(X_train_p2)[:, 1]
test_proba_p2 = phase2_model.predict_proba(X_test_p2)[:, 1]

train_auc_p2 = roc_auc_score(y_train_p2, train_proba_p2)
test_auc_p2 = roc_auc_score(y_test_p2, test_proba_p2)

print("\nPHASE 2 MODEL RESULTS")
print("TRAIN AUC:", round(train_auc_p2, 4))
print("TEST AUC:", round(test_auc_p2, 4))
print("AUC GAP:", round(train_auc_p2 - test_auc_p2, 4))

phase2_results = pd.DataFrame({
    "metric": [
        "train_auc",
        "test_auc",
        "auc_gap",
        "feature_count",
        "train_rows",
        "test_rows"
    ],
    "value": [
        train_auc_p2,
        test_auc_p2,
        train_auc_p2 - test_auc_p2,
        len(phase2_features),
        X_train_p2.shape[0],
        X_test_p2.shape[0]
    ]
})

phase2_results.to_csv(
    os.path.join(
        PHASE2_OUTPUT_FOLDER,
        "stepP8_phase2_model_results.csv"
    ),
    index=False
)

pd.DataFrame({"phase2_feature": phase2_features}).to_csv(
    os.path.join(
        PHASE2_OUTPUT_FOLDER,
        "stepP8_phase2_feature_list.csv"
    ),
    index=False
)

print("Step P8 completed.")


STEP P8 - Training Phase 2 pathway-enhanced model

PHASE 2 MODEL RESULTS
TRAIN AUC: 0.8437
TEST AUC: 0.839
AUC GAP: 0.0047
Step P8 completed.


In [ ]:
# ============================================================
# Step P9 - Intervention Opportunity Score
# ============================================================

print("\nSTEP P9 - Building intervention opportunity score")

intervention_df_p2 = df_test_timeline.copy()
intervention_df_p2["actual"] = y_test_p2.values
intervention_df_p2["predicted_probability_p2"] = test_proba_p2

intervention_df_p2["intervention_opportunity_score"] = (
    0.40 * intervention_df_p2["predicted_probability_p2"] +
    0.25 * intervention_df_p2["temporal_deterioration_score"] +
    0.20 * intervention_df_p2["pathway_lift_norm"] +
    0.15 * intervention_df_p2["rx_escalation_norm"]
)

intervention_df_p2["intervention_opportunity_score"] = (
    intervention_df_p2["intervention_opportunity_score"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .clip(0, 1)
)

intervention_df_p2["intervention_priority_tier"] = pd.cut(
    intervention_df_p2["intervention_opportunity_score"],
    bins=[-0.001, 0.20, 0.40, 0.60, 0.80, 1.001],
    labels=[
        "ROUTINE",
        "WATCHLIST",
        "CARE_MANAGEMENT",
        "HIGH_PRIORITY",
        "URGENT"
    ]
)

intervention_summary_p2 = (
    intervention_df_p2
    .groupby("intervention_priority_tier", observed=True)
    .agg(
        patient_month_count=("member_id", "count"),
        unique_members=("member_id", "nunique"),
        actual_hospitalization_rate=("actual", "mean"),
        avg_predicted_probability=("predicted_probability_p2", "mean"),
        avg_deterioration_score=("temporal_deterioration_score", "mean"),
        avg_intervention_opportunity_score=("intervention_opportunity_score", "mean")
    )
    .reset_index()
)

print("\nIntervention opportunity summary:")
print(intervention_summary_p2)

intervention_summary_p2.to_csv(
    os.path.join(
        PHASE2_OUTPUT_FOLDER,
        "stepP9_intervention_opportunity_summary.csv"
    ),
    index=False
)

intervention_df_p2[
    [
        "member_id",
        PREDICTION_DATE_COL,
        "actual",
        "predicted_probability_p2",
        "temporal_deterioration_score",
        "intervention_opportunity_score",
        "intervention_priority_tier",
        "pathway_signature"
    ]
].to_csv(
    os.path.join(
        PHASE2_OUTPUT_FOLDER,
        "stepP9_test_intervention_scored.csv"
    ),
    index=False
)

print("Step P9 completed.")


STEP P9 - Building intervention opportunity score

Intervention opportunity summary:
  intervention_priority_tier  patient_month_count  unique_members  \
0                    ROUTINE              2701172           98341   
1                  WATCHLIST               810812           99036   
2            CARE_MANAGEMENT               112658           31274   
3              HIGH_PRIORITY                10064            6045   
4                     URGENT                   34              32   

   actual_hospitalization_rate  avg_predicted_probability  \
0                     0.020619                   0.020655   
1                     0.103288                   0.103249   
2                     0.218813                   0.214961   
3                     0.368641                   0.366431   
4                     0.470588                   0.568259   

   avg_deterioration_score  avg_intervention_opportunity_score  
0                 0.098470                            0.075253  
1 

In [ ]:
# ============================================================
# Step P10 - Phase 2 Validation / Governance Summary
# ============================================================

print("\nSTEP P10 - Phase 2 validation summary")

baseline_auc = test_auc_h if "test_auc_h" in globals() else np.nan

auc_lift_vs_phase1 = (
    test_auc_p2 - baseline_auc
    if not pd.isna(baseline_auc)
    else np.nan
)

phase2_validation_summary = pd.DataFrame({
    "metric": [
        "phase1_test_auc",
        "phase2_test_auc",
        "auc_lift_vs_phase1",
        "phase2_train_auc",
        "phase2_auc_gap",
        "phase2_feature_count",
        "phase2_train_rows",
        "phase2_test_rows",
        "global_train_hospitalization_rate",
        "global_test_hospitalization_rate"
    ],
    "value": [
        baseline_auc,
        test_auc_p2,
        auc_lift_vs_phase1,
        train_auc_p2,
        train_auc_p2 - test_auc_p2,
        len(phase2_features),
        X_train_p2.shape[0],
        X_test_p2.shape[0],
        y_train_p2.mean(),
        y_test_p2.mean()
    ]
})

phase2_governance_checklist = pd.DataFrame({
    "control": [
        "Phase 1 baseline preserved",
        "Train-only pathway table used",
        "Future labels excluded from features",
        "Pathway lift created",
        "Temporal deterioration score created",
        "Intervention opportunity score created",
        "External test set used"
    ],
    "status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS"
    ],
    "notes": [
        "H1-H18 retained as baseline framework.",
        "Pathway probabilities learned from TRAIN Sets 1+2 only.",
        "No hospitalized_next or future_ip fields used as predictors.",
        "Pathway lift compares pathway rate against global train rate.",
        "Score combines chronic burden, Rx escalation, prior IP, and pathway lift.",
        "Used for care management prioritization, not automatic clinical decisions.",
        f"TEST Sample {TEST_SAMPLE}"
    ]
})

print("\nPhase 2 validation summary:")
print(phase2_validation_summary)

print("\nPhase 2 governance checklist:")
print(phase2_governance_checklist)

phase2_validation_summary.to_csv(
    os.path.join(
        PHASE2_OUTPUT_FOLDER,
        "stepP10_phase2_validation_summary.csv"
    ),
    index=False
)

phase2_governance_checklist.to_csv(
    os.path.join(
        PHASE2_OUTPUT_FOLDER,
        "stepP10_phase2_governance_checklist.csv"
    ),
    index=False
)

print("\nStep P10 completed.")


STEP P10 - Phase 2 validation summary

Phase 2 validation summary:
                              metric         value
0                    phase1_test_auc  8.389718e-01
1                    phase2_test_auc  8.389837e-01
2                 auc_lift_vs_phase1  1.189554e-05
3                   phase2_train_auc  8.437232e-01
4                     phase2_auc_gap  4.739528e-03
5               phase2_feature_count  9.200000e+01
6                  phase2_train_rows  7.265664e+06
7                   phase2_test_rows  3.634740e+06
8  global_train_hospitalization_rate  4.612545e-02
9   global_test_hospitalization_rate  4.617084e-02

Phase 2 governance checklist:
                                  control status  \
0              Phase 1 baseline preserved   PASS   
1           Train-only pathway table used   PASS   
2    Future labels excluded from features   PASS   
3                    Pathway lift created   PASS   
4    Temporal deterioration score created   PASS   
5  Intervention opportunity 

In [ ]:
# Steps to Determine if hospitalization was voluntary or involuntary as we care more about the involuntary hospitalization right now

In [ ]:
# ============================================================
# Option 1 - Scheduled vs Unscheduled Hospitalization Model
# Step U0 - Configuration
# ============================================================

print("\nOPTION 1 - Initializing scheduled vs unscheduled hospitalization segmentation")

UNSCHEDULED_OUTPUT_FOLDER = os.path.join(
    BASE_FOLDER,
    "Unscheduled_Hospitalization_Model"
)

os.makedirs(UNSCHEDULED_OUTPUT_FOLDER, exist_ok=True)

# Primary future target
UNSCHEDULED_TARGET_LABEL = "unscheduled_hospitalized_next_90d"
SCHEDULED_TARGET_LABEL = "scheduled_hospitalized_next_90d"

# Useful for later model comparison
ANY_HOSPITALIZATION_TARGET = TARGET_LABEL_NAME

# Planned / expected procedure indicators
ELECTIVE_DRG_KEYWORDS = [
    "JOINT",
    "HIP",
    "KNEE",
    "REPLACEMENT",
    "REHAB",
    "CARDIAC PROCEDURE",
    "PACEMAKER",
    "DEFIBRILLATOR",
    "CABG",
    "VALVE",
    "CHEMOTHERAPY",
    "RADIATION"
]

# Diagnosis/procedure categories often associated with unexpected admission
UNSCHEDULED_DIAGNOSIS_KEYWORDS = [
    "SEPSIS",
    "PNEUMONIA",
    "RESPIRATORY",
    "COPD",
    "HEART FAILURE",
    "CHF",
    "RENAL FAILURE",
    "DEHYDRATION",
    "SYNCOPE",
    "FALL",
    "FRACTURE",
    "STROKE",
    "INFECTION"
]

# Since SynPUF has codes, not descriptions, we will create rule placeholders.
# Later, we can enrich with code maps.
PLANNED_DRG_CODES = set([
    # Placeholder examples; update after reviewing DRG frequency tables.
])

UNSCHEDULED_DRG_CODES = set([
    # Placeholder examples; update after reviewing DRG frequency tables.
])

MIN_UNSCHEDULED_REVIEW_ROWS = 100

print("UNSCHEDULED_OUTPUT_FOLDER:", UNSCHEDULED_OUTPUT_FOLDER)
print("Unscheduled target:", UNSCHEDULED_TARGET_LABEL)
print("Scheduled target:", SCHEDULED_TARGET_LABEL)
print("Step U0 completed.")


OPTION 1 - Initializing scheduled vs unscheduled hospitalization segmentation
UNSCHEDULED_OUTPUT_FOLDER: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Unscheduled_Hospitalization_Model
Unscheduled target: unscheduled_hospitalized_next_90d
Scheduled target: scheduled_hospitalized_next_90d
Step U0 completed.


In [ ]:
# ============================================================
# Step U1 - Classify Inpatient Admissions as Scheduled vs Unscheduled
# ============================================================

print("\nSTEP U1 - Classifying inpatient admissions")

def classify_admission_type(ip_events, dataset_label):

    ip = ip_events.copy()

    required_cols = [
        "member_id",
        "claim_id",
        "admission_date",
        "drg_code",
        "admitting_diagnosis",
        "length_of_stay"
    ]

    for col in required_cols:
        if col not in ip.columns:
            ip[col] = ""

    ip["drg_code"] = (
        ip["drg_code"]
        .astype(str)
        .str.strip()
        .replace(["", "nan", "None", "NaN"], "UNKNOWN")
    )

    ip["admitting_diagnosis"] = (
        ip["admitting_diagnosis"]
        .astype(str)
        .str.strip()
        .replace(["", "nan", "None", "NaN"], "UNKNOWN")
    )

    ip["length_of_stay"] = safe_numeric(ip["length_of_stay"], fill_value=0)

    # --------------------------------------------------------
    # Rule-based segmentation
    # --------------------------------------------------------

    ip["scheduled_rule_drg_flag"] = (
        ip["drg_code"].isin(PLANNED_DRG_CODES)
    ).astype(int)

    ip["unscheduled_rule_drg_flag"] = (
        ip["drg_code"].isin(UNSCHEDULED_DRG_CODES)
    ).astype(int)

    # Proxy: very short admissions with surgical/planned DRG code can be scheduled.
    # Since code maps are not yet loaded, this is conservative.
    ip["possible_scheduled_proxy_flag"] = (
        (ip["scheduled_rule_drg_flag"] == 1)
    ).astype(int)

    # Proxy: emergency-like diagnoses will be refined later with ICD9 group mapping.
    ip["possible_unscheduled_proxy_flag"] = (
        (ip["unscheduled_rule_drg_flag"] == 1)
    ).astype(int)

    # Conservative default:
    # If not clearly scheduled, treat as unscheduled/unknown for prevention modeling.
    ip["admission_type_segment"] = np.where(
        ip["possible_scheduled_proxy_flag"] == 1,
        "SCHEDULED_EXPECTED",
        "UNSCHEDULED_OR_UNKNOWN"
    )

    ip["scheduled_admission_flag"] = (
        ip["admission_type_segment"] == "SCHEDULED_EXPECTED"
    ).astype(int)

    ip["unscheduled_admission_flag"] = (
        ip["admission_type_segment"] == "UNSCHEDULED_OR_UNKNOWN"
    ).astype(int)

    print(f"\n{dataset_label} admission type distribution:")
    print(ip["admission_type_segment"].value_counts(dropna=False))

    summary = (
        ip.groupby("admission_type_segment")
        .agg(
            admissions=("claim_id", "count"),
            unique_members=("member_id", "nunique"),
            avg_los=("length_of_stay", "mean")
        )
        .reset_index()
    )

    print(summary)

    return ip, summary


ip_train_classified, ip_train_admission_type_summary = classify_admission_type(
    ip_train_events,
    "TRAIN Sets 1+2"
)

ip_test_classified, ip_test_admission_type_summary = classify_admission_type(
    ip_test_events,
    "TEST Set 3"
)

ip_train_admission_type_summary.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU1_train_admission_type_summary.csv"
    ),
    index=False
)

ip_test_admission_type_summary.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU1_test_admission_type_summary.csv"
    ),
    index=False
)

print("\nStep U1 completed.")


STEP U1 - Classifying inpatient admissions

TRAIN Sets 1+2 admission type distribution:
admission_type_segment
UNSCHEDULED_OR_UNKNOWN    133267
Name: count, dtype: int64
   admission_type_segment  admissions  unique_members   avg_los
0  UNSCHEDULED_OR_UNKNOWN      133267           75441  5.555284

TEST Set 3 admission type distribution:
admission_type_segment
UNSCHEDULED_OR_UNKNOWN    66672
Name: count, dtype: int64
   admission_type_segment  admissions  unique_members   avg_los
0  UNSCHEDULED_OR_UNKNOWN       66672           37826  5.580454

Step U1 completed.


In [ ]:
# ============================================================
# Step U2 - DRG Frequency and Hospitalization Type Review Table
# ============================================================

print("\nSTEP U2 - Building DRG frequency review table")

def build_drg_review_table(ip_classified, dataset_label):

    ip = ip_classified.copy()

    drg_review = (
        ip.groupby("drg_code")
        .agg(
            admission_count=("claim_id", "count"),
            unique_members=("member_id", "nunique"),
            avg_los=("length_of_stay", "mean"),
            scheduled_rate=("scheduled_admission_flag", "mean"),
            unscheduled_rate=("unscheduled_admission_flag", "mean")
        )
        .reset_index()
        .sort_values("admission_count", ascending=False)
    )

    drg_review["dataset_label"] = dataset_label

    return drg_review


drg_train_review = build_drg_review_table(
    ip_train_classified,
    "TRAIN Sets 1+2"
)

drg_test_review = build_drg_review_table(
    ip_test_classified,
    "TEST Set 3"
)

drg_review_all = pd.concat(
    [drg_train_review, drg_test_review],
    ignore_index=True
)

print("\nTop TRAIN DRGs:")
print(drg_train_review.head(25))

print("\nTop TEST DRGs:")
print(drg_test_review.head(25))

drg_review_all.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU2_drg_review_table.csv"
    ),
    index=False
)

print("\nStep U2 completed.")
print("Review this table to refine PLANNED_DRG_CODES and UNSCHEDULED_DRG_CODES in U0.")


STEP U2 - Building DRG frequency review table

Top TRAIN DRGs:
    drg_code  admission_count  unique_members    avg_los  scheduled_rate  \
673      887              540             530   9.198148             0.0   
672      886              538             535  10.464684             0.0   
132      177              533             532   5.724203             0.0   
668      882              519             515  10.186898             0.0   
136      181              519             517   5.755299             0.0   
130      175              517             515   5.464217             0.0   
131      176              511             510   6.121331             0.0   
138      183              508             506   6.055118             0.0   
665      876              505             499  10.164356             0.0   
666      880              494             491  10.354251             0.0   
671      885              494             491   9.708502             0.0   
669      883            

In [ ]:
# ============================================================
# Step U2.5 - MS-DRG Enrichment - FULL FIXED VERSION
# Adds DRG title, type, weight, GMLOS, and initial classification
# ============================================================

print("\nSTEP U2.5 - Robust MS-DRG enrichment")

import os
import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# DRG reference file
# ------------------------------------------------------------

DRG_REFERENCE_FILE = os.path.join(
    BASE_FOLDER,
    "MS-DRG-Tables-2008-2022.xlsx"
)

if not os.path.exists(DRG_REFERENCE_FILE):
    raise FileNotFoundError(DRG_REFERENCE_FILE)

print("DRG reference file found:", DRG_REFERENCE_FILE)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def standardize_drg_code(value):

    if pd.isna(value):
        return "UNKNOWN"

    value = str(value).strip().upper()

    if value in ["", "NAN", "NONE"]:
        return "UNKNOWN"

    if value in ["OTH", "OTHER"]:
        return "OTH"

    value = value.replace(".0", "")

    try:
        value = str(int(float(value)))
    except Exception:
        pass

    if value.isdigit():
        return value.zfill(3)

    return value


def detect_header_row(raw_df):

    for i in range(min(40, len(raw_df))):

        row_values = (
            raw_df.iloc[i]
            .astype(str)
            .str.upper()
            .str.strip()
            .tolist()
        )

        row_text = " | ".join(row_values)

        if (
            ("MS-DRG" in row_text or "MS DRG" in row_text or "DRG" in row_text)
            and
            ("TITLE" in row_text or "DESCRIPTION" in row_text)
        ):
            return i

    return None


# ------------------------------------------------------------
# Load DRG reference table
# ------------------------------------------------------------

def load_drg_reference_table_robust(drg_file):

    excel = pd.ExcelFile(drg_file)
    drg_parts = []

    for sheet in excel.sheet_names:

        print("\nInspecting sheet:", sheet)

        year_match = re.search(r"(2008|2009|2010|20\d{2})", str(sheet))

        if year_match is None:
            print("Skipping sheet; no year found:", sheet)
            continue

        sheet_year = int(year_match.group(1))

        raw = pd.read_excel(
            drg_file,
            sheet_name=sheet,
            header=None,
            dtype=str
        )

        header_row = detect_header_row(raw)

        if header_row is None:
            print("Could not detect header row. Skipping:", sheet)
            continue

        temp = pd.read_excel(
            drg_file,
            sheet_name=sheet,
            header=header_row,
            dtype=str
        )

        temp.columns = (
            temp.columns.astype(str)
            .str.strip()
            .str.upper()
            .str.replace("\n", " ", regex=False)
            .str.replace("  ", " ", regex=False)
        )

        print("Detected columns:", list(temp.columns))

        drg_col = None
        title_col = None
        type_col = None
        weight_col = None
        gmlos_col = None
        arith_los_col = None
        mdc_col = None

        for c in temp.columns:

            c_clean = c.upper().strip()

            # IMPORTANT: exact match only, so MS-DRG TITLE is not used as code
            if c_clean in ["MS-DRG", "MS DRG", "DRG"]:
                drg_col = c

            elif "TITLE" in c_clean or "DESCRIPTION" in c_clean:
                title_col = c

            elif c_clean == "TYPE":
                type_col = c

            elif "WEIGHT" in c_clean:
                weight_col = c

            elif "GMLOS" in c_clean or "GEOMETRIC" in c_clean:
                gmlos_col = c

            elif "ARITH" in c_clean and "LOS" in c_clean:
                arith_los_col = c

            elif c_clean == "MDC":
                mdc_col = c

        if drg_col is None or title_col is None:
            print("Required DRG/title columns not found. Skipping:", sheet)
            continue

        out = pd.DataFrame()

        out["drg_year"] = sheet_year
        out["drg_code"] = temp[drg_col].apply(standardize_drg_code)
        out["drg_title"] = temp[title_col].astype(str).str.strip().str.upper()

        if type_col is not None:
            out["drg_type"] = temp[type_col].astype(str).str.strip().str.upper()
        else:
            out["drg_type"] = "UNKNOWN"

        if weight_col is not None:
            out["drg_weight"] = safe_numeric(temp[weight_col], fill_value=0)
        else:
            out["drg_weight"] = 0

        if gmlos_col is not None:
            out["drg_gmlos"] = safe_numeric(temp[gmlos_col], fill_value=0)
        else:
            out["drg_gmlos"] = 0

        if arith_los_col is not None:
            out["drg_arithmetic_los"] = safe_numeric(temp[arith_los_col], fill_value=0)
        else:
            out["drg_arithmetic_los"] = 0

        if mdc_col is not None:
            out["drg_mdc"] = temp[mdc_col].astype(str).str.strip().str.upper()
        else:
            out["drg_mdc"] = "UNKNOWN"

        out = out[
            (out["drg_code"] != "UNKNOWN")
            &
            (out["drg_title"].notna())
            &
            (out["drg_title"] != "")
            &
            (out["drg_title"] != "NAN")
        ]

        drg_parts.append(out)

    if len(drg_parts) == 0:
        raise ValueError("No usable DRG sheets were parsed.")

    drg_ref = pd.concat(drg_parts, ignore_index=True)

    drg_ref = (
        drg_ref
        .drop_duplicates(["drg_year", "drg_code"])
        .reset_index(drop=True)
    )

    print("\nDRG reference loaded.")
    print("Shape:", drg_ref.shape)
    print("Years:", sorted(drg_ref["drg_year"].dropna().unique()))
    print("\nReference sample:")
    print(drg_ref.head(10))

    return drg_ref


# ------------------------------------------------------------
# Initial clinical classification
# ------------------------------------------------------------

def classify_drg_title(title):

    title = str(title).upper()

    potentially_preventable_terms = [
        "HEART FAILURE",
        "COPD",
        "PNEUMONIA",
        "SEPTICEMIA",
        "SEPSIS",
        "RESPIRATORY",
        "RENAL FAILURE",
        "KIDNEY",
        "DEHYDRATION",
        "DIABETES",
        "CELLULITIS",
        "URINARY",
        "HYPERTENSION",
        "ASTHMA"
    ]

    acute_unscheduled_terms = [
        "SHOCK",
        "STROKE",
        "INTRACRANIAL",
        "SYNCOPE",
        "FRACTURE",
        "TRAUMA",
        "BLEED",
        "HEMORRHAGE",
        "INFECTION",
        "COMPLICATION",
        "ACUTE"
    ]

    scheduled_terms = [
        "REPLACEMENT",
        "REVISION",
        "REHABILITATION",
        "AFTERCARE",
        "TRANSPLANT",
        "PACEMAKER",
        "DEFIBRILLATOR",
        "VALVE",
        "CABG",
        "SPINAL FUSION",
        "MAJOR JOINT",
        "HIP",
        "KNEE"
    ]

    if any(term in title for term in potentially_preventable_terms):
        return "POTENTIALLY_PREVENTABLE"

    if any(term in title for term in acute_unscheduled_terms):
        return "UNSCHEDULED_ACUTE"

    if any(term in title for term in scheduled_terms):
        return "LIKELY_SCHEDULED"

    return "AMBIGUOUS"


# ------------------------------------------------------------
# Load reference
# ------------------------------------------------------------

drg_reference = load_drg_reference_table_robust(DRG_REFERENCE_FILE)

drg_reference["drg_initial_classification"] = (
    drg_reference["drg_title"].apply(classify_drg_title)
)

print("\nDRG reference classification distribution:")
print(drg_reference["drg_initial_classification"].value_counts(dropna=False))

# ------------------------------------------------------------
# Enrich inpatient events
# ------------------------------------------------------------

def enrich_ip_with_drg(ip_events, drg_ref, dataset_label):

    ip = ip_events.copy()

    ip["drg_code"] = ip["drg_code"].apply(standardize_drg_code)

    # Use one row per DRG code because drg_year is not parsing correctly
    ref = (
        drg_ref
        .sort_values(["drg_code"])
        .drop_duplicates("drg_code")
        .copy()
    )

    ip = ip.merge(
        ref,
        on="drg_code",
        how="left"
    )

    text_cols = [
        "drg_title",
        "drg_type",
        "drg_mdc",
        "drg_initial_classification"
    ]

    for col in text_cols:
        ip[col] = ip[col].fillna("UNKNOWN")

    numeric_cols = [
        "drg_weight",
        "drg_gmlos",
        "drg_arithmetic_los"
    ]

    for col in numeric_cols:
        ip[col] = safe_numeric(ip[col], fill_value=0)

    match_rate = (ip["drg_title"] != "UNKNOWN").mean()

    print(f"\n{dataset_label} DRG enrichment complete.")
    print("Rows:", len(ip))
    print("DRG match rate:", round(match_rate, 4))

    print("\nInitial classification distribution:")
    print(ip["drg_initial_classification"].value_counts(dropna=False))

    return ip


ip_train_drg_enriched = enrich_ip_with_drg(
    ip_train_events,
    drg_reference,
    "TRAIN Sets 1+2"
)

ip_test_drg_enriched = enrich_ip_with_drg(
    ip_test_events,
    drg_reference,
    "TEST Set 3"
)

# ------------------------------------------------------------
# Review tables
# ------------------------------------------------------------

def build_enriched_drg_review(ip_enriched, dataset_label):

    review = (
        ip_enriched
        .groupby(
            [
                "drg_code",
                "drg_title",
                "drg_type",
                "drg_initial_classification"
            ],
            dropna=False
        )
        .agg(
            admission_count=("claim_id", "count"),
            unique_members=("member_id", "nunique"),
            avg_los=("length_of_stay", "mean"),
            avg_drg_weight=("drg_weight", "mean"),
            avg_gmlos=("drg_gmlos", "mean")
        )
        .reset_index()
        .sort_values("admission_count", ascending=False)
    )

    review["dataset_label"] = dataset_label

    return review


drg_train_enriched_review = build_enriched_drg_review(
    ip_train_drg_enriched,
    "TRAIN Sets 1+2"
)

drg_test_enriched_review = build_enriched_drg_review(
    ip_test_drg_enriched,
    "TEST Set 3"
)

drg_enriched_review_all = pd.concat(
    [drg_train_enriched_review, drg_test_enriched_review],
    ignore_index=True
)

print("\nTop TRAIN enriched DRGs:")
print(drg_train_enriched_review.head(30))

print("\nTop TEST enriched DRGs:")
print(drg_test_enriched_review.head(30))

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

drg_reference.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU25_drg_reference_standardized.csv"
    ),
    index=False
)

drg_enriched_review_all.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU25_enriched_drg_review_table.csv"
    ),
    index=False
)

ip_train_drg_enriched.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU25_train_ip_drg_enriched.csv"
    ),
    index=False
)

ip_test_drg_enriched.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU25_test_ip_drg_enriched.csv"
    ),
    index=False
)

print("\nStep U2.5 completed.")
print("Saved:")
print(os.path.join(UNSCHEDULED_OUTPUT_FOLDER, "stepU25_drg_reference_standardized.csv"))
print(os.path.join(UNSCHEDULED_OUTPUT_FOLDER, "stepU25_enriched_drg_review_table.csv"))
print(os.path.join(UNSCHEDULED_OUTPUT_FOLDER, "stepU25_train_ip_drg_enriched.csv"))
print(os.path.join(UNSCHEDULED_OUTPUT_FOLDER, "stepU25_test_ip_drg_enriched.csv"))


STEP U2.5 - Robust MS-DRG enrichment
DRG reference file found: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/MS-DRG-Tables-2008-2022.xlsx

Inspecting sheet: 2008
Detected columns: ['MS-DRG', 'FY08 FINAL RULE POST-ACUTE DRG', 'FY08 FINAL RULE SPECIAL PAY DRG', 'MDC', 'TYPE', 'MS-DRG TITLE', 'WEIGHTS', 'GEOMETRIC MEAN LOS', 'ARITHMETIC MEAN LOS']

Inspecting sheet: 2009
Detected columns: ['MS-DRG', 'FY09 FINAL RULE POST-ACUTE DRG', 'FY09 FINAL RULE SPECIAL PAY DRG', 'MDC', 'TYPE', 'MS-DRG TITLE', 'WEIGHTS', 'GEOMETRIC MEAN LOS', 'ARITHMETIC MEAN LOS']

Inspecting sheet: 2010
Detected columns: ['MS-DRG', 'TYPE', 'MS-DRG TITLE', 'WEIGHTS', 'GMLOS']

DRG reference loaded.
Shape: (746, 8)
Years: []

Reference sample:
   drg_year drg_code                                          drg_title  \
0       NaN      001  HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SY...   
1       NaN      002  HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SY...   
2       NaN      003  ECMO

In [ ]:
# ============================================================
# Step U2.6 - Hospitalization Classification Library
# ============================================================

print("\nSTEP U2.6 - Building hospitalization classification library")

# ------------------------------------------------------------
# Build enhanced classification logic
# ------------------------------------------------------------

def build_hospitalization_classification(title):

    title = str(title).upper()

    # ----------------------------------------
    # Mental health / behavioral health
    # ----------------------------------------

    mental_terms = [
        "PSYCHOSIS",
        "DEPRESSIVE",
        "MENTAL",
        "NEUROSES",
        "BEHAVIORAL",
        "PERSONALITY",
        "ADJUSTMENT REACTION",
        "MENTAL RETARDATION",
        "PSYCHOSOCIAL"
    ]

    # ----------------------------------------
    # Scheduled / expected
    # ----------------------------------------

    scheduled_terms = [
        "REHABILITATION",
        "AFTERCARE",
        "TRANSPLANT",
        "PACEMAKER",
        "DEFIBRILLATOR",
        "VALVE",
        "CABG",
        "SPINAL FUSION",
        "REPLACEMENT",
        "HIP",
        "KNEE",
        "JOINT"
    ]

    # ----------------------------------------
    # Potentially preventable
    # ----------------------------------------

    preventable_terms = [
        "HEART FAILURE",
        "RESPIRATORY FAILURE",
        "PNEUMONIA",
        "PLEURISY",
        "COPD",
        "BRONCHITIS",
        "ASTHMA",
        "RESPIRATORY INFECTIONS",
        "PULMONARY EDEMA",
        "SEPSIS",
        "SEPTICEMIA",
        "DEHYDRATION",
        "RENAL FAILURE",
        "URINARY",
        "CELLULITIS",
        "DIABETES"
    ]

    # ----------------------------------------
    # Acute unexpected
    # ----------------------------------------

    acute_terms = [
        "TRAUMA",
        "FRACTURE",
        "STROKE",
        "INTRACRANIAL",
        "HEMORRHAGE",
        "BLEED",
        "PULMONARY EMBOLISM",
        "SHOCK",
        "SYNCOPE",
        "ACUTE"
    ]

    # ----------------------------------------
    # Classification priority
    # ----------------------------------------

    if any(term in title for term in mental_terms):
        return "MENTAL_HEALTH"

    if any(term in title for term in scheduled_terms):
        return "LIKELY_SCHEDULED"

    if any(term in title for term in preventable_terms):
        return "POTENTIALLY_PREVENTABLE"

    if any(term in title for term in acute_terms):
        return "UNSCHEDULED_ACUTE"

    return "AMBIGUOUS"


# ------------------------------------------------------------
# Apply classification
# ------------------------------------------------------------

drg_reference["hospitalization_classification"] = (
    drg_reference["drg_title"]
    .apply(build_hospitalization_classification)
)

print("\nClassification distribution:")
print(
    drg_reference["hospitalization_classification"]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# Build library table
# ------------------------------------------------------------

hospitalization_library = (
    drg_reference
    [
        [
            "drg_code",
            "drg_title",
            "drg_type",
            "drg_weight",
            "drg_gmlos",
            "hospitalization_classification"
        ]
    ]
    .copy()
)

hospitalization_library = (
    hospitalization_library
    .sort_values(
        [
            "hospitalization_classification",
            "drg_code"
        ]
    )
    .reset_index(drop=True)
)

print("\nHospitalization classification library sample:")
print(hospitalization_library.head(50))

# ------------------------------------------------------------
# Attach classifications to admissions
# ------------------------------------------------------------

ip_train_drg_enriched = (
    ip_train_drg_enriched
    .drop(
        columns=["hospitalization_classification"],
        errors="ignore"
    )
)

ip_test_drg_enriched = (
    ip_test_drg_enriched
    .drop(
        columns=["hospitalization_classification"],
        errors="ignore"
    )
)

ip_train_drg_enriched = ip_train_drg_enriched.merge(
    hospitalization_library[
        [
            "drg_code",
            "hospitalization_classification"
        ]
    ],
    on="drg_code",
    how="left"
)

ip_test_drg_enriched = ip_test_drg_enriched.merge(
    hospitalization_library[
        [
            "drg_code",
            "hospitalization_classification"
        ]
    ],
    on="drg_code",
    how="left"
)

ip_train_drg_enriched[
    "hospitalization_classification"
] = (
    ip_train_drg_enriched[
        "hospitalization_classification"
    ].fillna("UNKNOWN")
)

ip_test_drg_enriched[
    "hospitalization_classification"
] = (
    ip_test_drg_enriched[
        "hospitalization_classification"
    ].fillna("UNKNOWN")
)

# ------------------------------------------------------------
# Admission volume summary
# ------------------------------------------------------------

train_summary = (
    ip_train_drg_enriched
    .groupby("hospitalization_classification")
    .agg(
        admissions=("claim_id", "count"),
        unique_members=("member_id", "nunique"),
        avg_los=("length_of_stay", "mean")
    )
    .reset_index()
    .sort_values("admissions", ascending=False)
)

test_summary = (
    ip_test_drg_enriched
    .groupby("hospitalization_classification")
    .agg(
        admissions=("claim_id", "count"),
        unique_members=("member_id", "nunique"),
        avg_los=("length_of_stay", "mean")
    )
    .reset_index()
    .sort_values("admissions", ascending=False)
)

print("\nTRAIN classification summary:")
print(train_summary)

print("\nTEST classification summary:")
print(test_summary)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

hospitalization_library.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU26_hospitalization_classification_library.csv"
    ),
    index=False
)

train_summary.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU26_train_classification_summary.csv"
    ),
    index=False
)

test_summary.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU26_test_classification_summary.csv"
    ),
    index=False
)

print("\nStep U2.6 completed.")

print("\nSaved:")
print(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU26_hospitalization_classification_library.csv"
    )
)
print(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU26_train_classification_summary.csv"
    )
)
print(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU26_test_classification_summary.csv"
    )
)


STEP U2.6 - Building hospitalization classification library

Classification distribution:
hospitalization_classification
AMBIGUOUS                  521
UNSCHEDULED_ACUTE           86
LIKELY_SCHEDULED            85
POTENTIALLY_PREVENTABLE     46
MENTAL_HEALTH                8
Name: count, dtype: int64

Hospitalization classification library sample:
   drg_code                                          drg_title drg_type  \
0       003  ECMO OR TRACH W MV 96+ HRS OR PDX EXC FACE, MO...     SURG   
1       004  TRACH W MV 96+ HRS OR PDX EXC FACE, MOUTH & NE...     SURG   
2       011  TRACHEOSTOMY FOR FACE,MOUTH & NECK DIAGNOSES W...     SURG   
3       012  TRACHEOSTOMY FOR FACE,MOUTH & NECK DIAGNOSES W CC     SURG   
4       013  TRACHEOSTOMY FOR FACE,MOUTH & NECK DIAGNOSES W...     SURG   
5       028                            SPINAL PROCEDURES W MCC     SURG   
6       029  SPINAL PROCEDURES W CC OR SPINAL NEUROSTIMULATORS     SURG   
7       030                       SPINAL PROCEDUR

In [ ]:
print("\nTRAIN DRG SAMPLE")
print(
    ip_train_events["drg_code"]
    .astype(str)
    .value_counts()
    .head(20)
)

print("\nTEST DRG SAMPLE")
print(
    ip_test_events["drg_code"]
    .astype(str)
    .value_counts()
    .head(20)
)

print("\nTRAIN DRG examples")
print(
    sorted(
        ip_train_events["drg_code"]
        .astype(str)
        .dropna()
        .unique()
    )[:50]
)

print("\nReference DRG examples")
print(
    sorted(
        drg_reference["drg_code"]
        .astype(str)
        .dropna()
        .unique()
    )[:50]
)

print("\nReference years")
print(
    drg_reference["drg_year"]
    .value_counts(dropna=False)
)


TRAIN DRG SAMPLE
drg_code
887    540
886    538
177    533
181    519
882    519
175    517
176    511
183    508
876    505
880    494
885    494
883    494
184    493
OTH    493
949    492
202    491
166    485
182    483
180    482
189    481
Name: count, dtype: int64

TEST DRG SAMPLE
drg_code
200    269
194    268
199    267
886    264
167    254
208    254
880    254
885    253
882    253
189    250
947    248
946    247
884    247
881    244
164    243
196    243
203    243
207    242
182    241
195    241
Name: count, dtype: int64

TRAIN DRG examples
['000', '001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '020', '021', '022', '023', '024', '025', '026', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061', '062', '063', '064']

Reference DRG examples
['001', '002', '003', '004', '005', '006', '007', '008', '00

In [ ]:
# ============================================================
# Step U3 - Monthly Scheduled vs Unscheduled Inpatient Tables
# ============================================================

print("\nSTEP U3 - Building monthly scheduled/unscheduled inpatient tables")

def build_admission_type_monthly(ip_classified, dataset_label):

    ip = ip_classified.copy()

    ip["admission_month_start"] = pd.to_datetime(
        ip["admission_month_start"],
        errors="coerce"
    )

    monthly = (
        ip.groupby(["member_id", "admission_month_start"])
        .agg(
            ip_claims_month=("ip_event_key", "nunique"),
            scheduled_ip_claims_month=("scheduled_admission_flag", "sum"),
            unscheduled_ip_claims_month=("unscheduled_admission_flag", "sum"),
            scheduled_los_month=(
                "length_of_stay",
                lambda x: x[ip.loc[x.index, "scheduled_admission_flag"] == 1].sum()
            ),
            unscheduled_los_month=(
                "length_of_stay",
                lambda x: x[ip.loc[x.index, "unscheduled_admission_flag"] == 1].sum()
            ),
            unique_drg_month=("drg_code", "nunique")
        )
        .reset_index()
    )

    print(f"{dataset_label} monthly admission type table shape:", monthly.shape)

    return monthly


ip_train_type_monthly = build_admission_type_monthly(
    ip_train_classified,
    "TRAIN Sets 1+2"
)

ip_test_type_monthly = build_admission_type_monthly(
    ip_test_classified,
    "TEST Set 3"
)

ip_train_type_monthly.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU3_train_monthly_admission_type_table.csv"
    ),
    index=False
)

ip_test_type_monthly.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU3_test_monthly_admission_type_table.csv"
    ),
    index=False
)

print("\nStep U3 completed.")


STEP U3 - Building monthly scheduled/unscheduled inpatient tables
TRAIN Sets 1+2 monthly admission type table shape: (124937, 8)
TEST Set 3 monthly admission type table shape: (62548, 8)

Step U3 completed.


In [ ]:
# ============================================================
# Step U4 - Create Future Scheduled / Unscheduled Hospitalization Labels
# ============================================================

print("\nSTEP U4 - Creating scheduled and unscheduled future hospitalization labels")

def add_scheduled_unscheduled_labels(timeline, monthly_type, dataset_label):

    df = timeline.copy()

    df[PREDICTION_DATE_COL] = pd.to_datetime(
        df[PREDICTION_DATE_COL],
        errors="coerce"
    )

    monthly_type = monthly_type.copy()
    monthly_type["admission_month_start"] = pd.to_datetime(
        monthly_type["admission_month_start"],
        errors="coerce"
    )

    df = df.merge(
        monthly_type,
        left_on=["member_id", PREDICTION_DATE_COL],
        right_on=["member_id", "admission_month_start"],
        how="left"
    )

    df = df.drop(columns=["admission_month_start"], errors="ignore")

    monthly_cols = [
        "ip_claims_month",
        "scheduled_ip_claims_month",
        "unscheduled_ip_claims_month",
        "scheduled_los_month",
        "unscheduled_los_month",
        "unique_drg_month"
    ]

    for col in monthly_cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = safe_numeric(df[col], fill_value=0)

    df = df.sort_values(["member_id", PREDICTION_DATE_COL]).reset_index(drop=True)

    future_window_map = {
        30: 1,
        90: 3,
        180: 6
    }

    for days, months_forward in future_window_map.items():

        future_unscheduled = (
            df.groupby("member_id")["unscheduled_ip_claims_month"]
            .transform(
                lambda x: x.iloc[::-1]
                .rolling(window=months_forward, min_periods=1)
                .sum()
                .iloc[::-1]
            )
        )

        future_scheduled = (
            df.groupby("member_id")["scheduled_ip_claims_month"]
            .transform(
                lambda x: x.iloc[::-1]
                .rolling(window=months_forward, min_periods=1)
                .sum()
                .iloc[::-1]
            )
        )

        df[f"future_unscheduled_ip_claims_{days}d"] = future_unscheduled
        df[f"future_scheduled_ip_claims_{days}d"] = future_scheduled

        df[f"unscheduled_hospitalized_next_{days}d"] = (
            future_unscheduled > 0
        ).astype(int)

        df[f"scheduled_hospitalized_next_{days}d"] = (
            future_scheduled > 0
        ).astype(int)

    label_cols = [
        "hospitalized_next_90d",
        "unscheduled_hospitalized_next_90d",
        "scheduled_hospitalized_next_90d"
    ]

    label_cols = [c for c in label_cols if c in df.columns]

    print(f"\n{dataset_label} hospitalization label rates:")
    print(df[label_cols].mean())

    return df


df_train_unscheduled = add_scheduled_unscheduled_labels(
    df_train_timeline,
    ip_train_type_monthly,
    "TRAIN Sets 1+2"
)

df_test_unscheduled = add_scheduled_unscheduled_labels(
    df_test_timeline,
    ip_test_type_monthly,
    "TEST Set 3"
)

df_train_unscheduled.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU4_train_timeline_scheduled_unscheduled_labels.csv"
    ),
    index=False
)

df_test_unscheduled.to_csv(
    os.path.join(
        UNSCHEDULED_OUTPUT_FOLDER,
        "stepU4_test_timeline_scheduled_unscheduled_labels.csv"
    ),
    index=False
)

print("\nStep U4 completed.")
print("df_train_unscheduled:", df_train_unscheduled.shape)
print("df_test_unscheduled:", df_test_unscheduled.shape)


STEP U4 - Creating scheduled and unscheduled future hospitalization labels

TRAIN Sets 1+2 hospitalization label rates:
hospitalized_next_90d                0.046125
unscheduled_hospitalized_next_90d    0.046125
scheduled_hospitalized_next_90d      0.000000
dtype: float64

TEST Set 3 hospitalization label rates:
hospitalized_next_90d                0.046171
unscheduled_hospitalized_next_90d    0.046171
scheduled_hospitalized_next_90d      0.000000
dtype: float64

Step U4 completed.
df_train_unscheduled: (7265664, 147)
df_test_unscheduled: (3634740, 147)


In [ ]:
# ============================================================
# Diagnosis Progression Mining
# Step D0 - Configuration
# ============================================================

print("\nSTEP D0 - Initializing diagnosis progression mining")

DIAGNOSIS_OUTPUT_FOLDER = os.path.join(
    BASE_FOLDER,
    "Diagnosis_Progression_Mining"
)

os.makedirs(DIAGNOSIS_OUTPUT_FOLDER, exist_ok=True)

DIAGNOSIS_LOOKBACK_DAYS = 365
MIN_PATHWAY_ROWS = 250
HIGH_RISK_DIAG_PATHWAY_RATE = 0.15

DIAGNOSIS_GROUP_MAP = {
    "DIABETES": ["250"],
    "CKD": ["585", "586"],
    "AKI": ["584"],
    "CHF": ["428"],
    "COPD": ["490", "491", "492", "496"],
    "PNEUMONIA": ["480", "481", "482", "483", "484", "485", "486"],
    "RESPIRATORY_FAILURE": ["518"],
    "ASTHMA": ["493"],
    "SEPSIS": ["038", "99591", "99592", "78552"],
    "HYPERTENSION": ["401", "402", "403", "404", "405"],
    "CAD": ["410", "411", "412", "413", "414"],
    "STROKE_TIA": ["430", "431", "432", "433", "434", "435", "436"],
    "FALL_FRACTURE": ["E880", "E881", "E882", "E883", "E884", "E885", "E886", "E887", "E888", "800", "801", "802", "803", "804", "805", "806", "807", "808", "809", "810", "811", "812", "813", "814", "815", "816", "817", "818", "819", "820", "821", "822", "823", "824", "825", "826", "827", "828", "829"],
    "DEHYDRATION": ["2765"],
    "UTI": ["5990"],
    "CELLULITIS": ["681", "682"],
    "DEMENTIA": ["290", "294", "331"],
    "DEPRESSION_BEHAVIORAL": ["296", "300", "311"],
    "CANCER": ["140", "141", "142", "143", "144", "145", "146", "147", "148", "149", "150", "151", "152", "153", "154", "155", "156", "157", "158", "159", "160", "161", "162", "163", "164", "165", "170", "171", "172", "174", "175", "176", "179", "180", "181", "182", "183", "184", "185", "186", "187", "188", "189"]
}

print("Diagnosis output folder:", DIAGNOSIS_OUTPUT_FOLDER)
print("Diagnosis groups:", len(DIAGNOSIS_GROUP_MAP))
print("Step D0 completed.")


STEP D0 - Initializing diagnosis progression mining
Diagnosis output folder: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Diagnosis_Progression_Mining
Diagnosis groups: 19
Step D0 completed.


In [ ]:
# ============================================================
# Step D0.5 - Reload Inpatient Events for Diagnosis Mining
# Needed before D1 if session was restarted
# ============================================================

print("\nSTEP D0.5 - Reloading inpatient events for diagnosis mining")

import os
import gc
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Validate required H0 objects/functions
# ------------------------------------------------------------

required_items = [
    "TRAIN_SAMPLES",
    "TEST_SAMPLE",
    "INPUT_FOLDER",
    "INPATIENT_FIELD_MAP",
    "inpatient_file",
    "clean_id",
    "safe_date",
    "safe_numeric"
]

missing_items = [
    x for x in required_items
    if x not in globals()
]

if missing_items:
    raise ValueError(
        f"Missing required H0 objects/functions: {missing_items}. Run H0 first."
    )

# ------------------------------------------------------------
# Reload inpatient events using same H2 logic
# ------------------------------------------------------------

def reload_inpatient_events_for_diagnosis(samples, dataset_label):

    all_parts = []

    for sample in samples:

        path = inpatient_file(sample)

        print(f"\nLoading inpatient {dataset_label} Sample {sample}: {path}")

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        ip = pd.read_csv(path, dtype=str, low_memory=False)
        ip["source_sample"] = sample

        rename_map = {v: k for k, v in INPATIENT_FIELD_MAP.items()}
        ip = ip.rename(columns=rename_map)

        # Standard IDs
        ip["member_id"] = ip["member_id"].apply(clean_id)
        ip["claim_id"] = ip["claim_id"].apply(clean_id)

        # Dates
        for col in [
            "admission_date",
            "discharge_date",
            "claim_from_date",
            "claim_thru_date"
        ]:
            if col in ip.columns:
                ip[col] = safe_date(ip[col])

        # Numeric LOS
        if "utilization_days" in ip.columns:
            ip["length_of_stay"] = safe_numeric(
                ip["utilization_days"],
                fill_value=0
            )
        elif "length_of_stay" not in ip.columns:
            ip["length_of_stay"] = 0

        # DRG
        if "drg_code" in ip.columns:
            ip["drg_code"] = (
                ip["drg_code"]
                .astype(str)
                .str.strip()
                .replace(["", "nan", "None", "NaN"], "UNKNOWN")
            )
        else:
            ip["drg_code"] = "UNKNOWN"

        # Admitting diagnosis
        if "admitting_diagnosis" in ip.columns:
            ip["admitting_diagnosis"] = (
                ip["admitting_diagnosis"]
                .astype(str)
                .str.strip()
                .replace(["", "nan", "None", "NaN"], "UNKNOWN")
            )
        else:
            ip["admitting_diagnosis"] = "UNKNOWN"

        ip = ip.dropna(subset=["admission_date"])
        ip = ip[ip["member_id"] != ""]

        ip["admission_month_start"] = (
            ip["admission_date"]
            .values
            .astype("datetime64[M]")
        )

        ip["ip_event_key"] = (
            ip["source_sample"].astype(str)
            + "_"
            + ip["member_id"].astype(str)
            + "_"
            + ip["claim_id"].astype(str)
        )

        all_parts.append(ip)

    combined = pd.concat(all_parts, ignore_index=True)

    print(f"\n{dataset_label} inpatient events restored:", combined.shape)
    print("Admission range:", combined["admission_date"].min(), "to", combined["admission_date"].max())
    print("source_sample distribution:")
    print(combined["source_sample"].value_counts(dropna=False).sort_index())

    return combined


ip_train_events = reload_inpatient_events_for_diagnosis(
    TRAIN_SAMPLES,
    "TRAIN Sets 1+2"
)

ip_test_events = reload_inpatient_events_for_diagnosis(
    [TEST_SAMPLE],
    "TEST Set 3"
)

gc.collect()

print("\nStep D0.5 completed successfully.")
print("ip_train_events:", ip_train_events.shape)
print("ip_test_events:", ip_test_events.shape)


STEP D0.5 - Reloading inpatient events for diagnosis mining

Loading inpatient TRAIN Sets 1+2 Sample 1: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv

Loading inpatient TRAIN Sets 1+2 Sample 2: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Inpatient_Claims_Sample_2.csv

TRAIN Sets 1+2 inpatient events restored: (133267, 85)
Admission range: 2007-11-27 00:00:00 to 2010-12-30 00:00:00
source_sample distribution:
source_sample
1    66773
2    66494
Name: count, dtype: int64

Loading inpatient TEST Set 3 Sample 3: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Inpatient_Claims_Sample_3.csv

TEST Set 3 inpatient events restored: (66672, 85)
Admission range: 2007-11-29 00:00:00 to 2010-12-30 00:00:00
source_sample distribution:
source_sample
3    66672
Name: count, dtype: int64

Step D0.5 completed successfully.
ip_train_events: (133267, 85)
ip_te

In [ ]:
# ============================================================
# Step D1 - Build Diagnosis Timeline
# Carrier + Outpatient + Inpatient
# ============================================================

print("\nSTEP D1 - Building diagnosis timeline")


def clean_icd9(code):
    if pd.isna(code):
        return ""
    code = str(code).strip().upper().replace(".", "")
    if code in ["", "NAN", "NONE"]:
        return ""
    return code


def map_icd9_to_group(code):
    code = clean_icd9(code)

    if code == "":
        return "UNKNOWN"

    for group, prefixes in DIAGNOSIS_GROUP_MAP.items():
        for prefix in prefixes:
            prefix = prefix.replace(".", "").upper()
            if code.startswith(prefix):
                return group

    return "OTHER"


# ------------------------------------------------------------
# Carrier files - Medicare SynPUF carrier claims are split A/B
# ------------------------------------------------------------

def carrier_files(sample):

    possible_files = [
        os.path.join(
            INPUT_FOLDER,
            f"DE1_0_2008_to_2010_Carrier_Claims_Sample_{sample}A.csv"
        ),
        os.path.join(
            INPUT_FOLDER,
            f"DE1_0_2008_to_2010_Carrier_Claims_Sample_{sample}B.csv"
        ),
        os.path.join(
            INPUT_FOLDER,
            f"DEI_0_2008_to_2010_Carrier_Claims_Sample_{sample}A.csv"
        ),
        os.path.join(
            INPUT_FOLDER,
            f"DEI_0_2008_to_2010_Carrier_Claims_Sample_{sample}B.csv"
        )
    ]

    existing_files = [
        path for path in possible_files
        if os.path.exists(path)
    ]

    if len(existing_files) == 0:
        print(f"No Carrier A/B files found for Sample {sample}. Checked:")
        for path in possible_files:
            print(path)

    return existing_files


def outpatient_file(sample):
    return os.path.join(
        INPUT_FOLDER,
        f"DE1_0_2008_to_2010_Outpatient_Claims_Sample_{sample}.csv"
    )


def load_carrier_diagnoses(samples, dataset_label):

    parts = []

    for sample in samples:

        sample_files = carrier_files(sample)

        for path in sample_files:

            carrier_part = os.path.basename(path).replace(".csv", "")

            print(
                f"\nLoading Carrier diagnoses {dataset_label} "
                f"Sample {sample}: {path}"
            )

            df = pd.read_csv(
                path,
                dtype=str,
                low_memory=False
            )

            df["source_sample"] = sample
            df["carrier_part"] = carrier_part
            df["claim_source"] = "CARRIER"

            member_col = "DESYNPUF_ID"
            claim_col = "CLM_ID"

            date_col = None

            for possible in [
                "CLM_FROM_DT",
                "LINE_SRVC_DT_1",
                "LINE_SRVC_DT"
            ]:
                if possible in df.columns:
                    date_col = possible
                    break

            if date_col is None:
                print("No usable date column found in carrier file. Skipping:", path)
                continue

            diagnosis_cols = [
                c for c in df.columns
                if c.startswith("ICD9_DGNS_CD_")
                or c.startswith("LINE_ICD9_DGNS_CD_")
            ]

            if len(diagnosis_cols) == 0:
                print("No diagnosis columns found in carrier file. Skipping:", path)
                continue

            keep_cols = [
                member_col,
                claim_col,
                date_col
            ] + diagnosis_cols

            temp = df[keep_cols].copy()

            temp = temp.rename(
                columns={
                    member_col: "member_id",
                    claim_col: "claim_id",
                    date_col: "diagnosis_date"
                }
            )

            temp["member_id"] = temp["member_id"].apply(clean_id)
            temp["claim_id"] = temp["claim_id"].apply(clean_id)
            temp["diagnosis_date"] = safe_date(temp["diagnosis_date"])
            temp["source_sample"] = sample
            temp["carrier_part"] = carrier_part
            temp["claim_source"] = "CARRIER"

            long_df = temp.melt(
                id_vars=[
                    "member_id",
                    "claim_id",
                    "diagnosis_date",
                    "source_sample",
                    "carrier_part",
                    "claim_source"
                ],
                value_vars=diagnosis_cols,
                var_name="diagnosis_position",
                value_name="diagnosis_code"
            )

            long_df["diagnosis_code"] = long_df["diagnosis_code"].apply(clean_icd9)

            long_df = long_df[
                (long_df["member_id"] != "")
                &
                (long_df["diagnosis_code"] != "")
                &
                (long_df["diagnosis_date"].notna())
            ]

            print(
                "Carrier diagnosis rows extracted:",
                long_df.shape[0]
            )

            parts.append(long_df)

    if len(parts) == 0:
        return pd.DataFrame()

    out = pd.concat(parts, ignore_index=True)

    print(f"\n{dataset_label} combined Carrier diagnosis rows:", out.shape)
    print(out["source_sample"].value_counts(dropna=False))

    return out


def load_outpatient_diagnoses(samples, dataset_label):

    parts = []

    for sample in samples:
        path = outpatient_file(sample)

        print(f"\nLoading Outpatient diagnoses {dataset_label} Sample {sample}: {path}")

        if not os.path.exists(path):
            print("Outpatient file not found, skipping:", path)
            continue

        df = pd.read_csv(path, dtype=str, low_memory=False)
        df["source_sample"] = sample
        df["claim_source"] = "OUTPATIENT"

        member_col = "DESYNPUF_ID"
        claim_col = "CLM_ID"
        date_col = "CLM_FROM_DT"

        diagnosis_cols = [
            c for c in df.columns
            if c.startswith("ICD9_DGNS_CD_")
            or c == "ADMTNG_ICD9_DGNS_CD"
        ]

        keep_cols = [member_col, claim_col, date_col] + diagnosis_cols

        temp = df[keep_cols].copy()
        temp = temp.rename(
            columns={
                member_col: "member_id",
                claim_col: "claim_id",
                date_col: "diagnosis_date"
            }
        )

        temp["member_id"] = temp["member_id"].apply(clean_id)
        temp["claim_id"] = temp["claim_id"].apply(clean_id)
        temp["diagnosis_date"] = safe_date(temp["diagnosis_date"])
        temp["source_sample"] = sample
        temp["claim_source"] = "OUTPATIENT"

        long_df = temp.melt(
            id_vars=["member_id", "claim_id", "diagnosis_date", "source_sample", "claim_source"],
            value_vars=diagnosis_cols,
            var_name="diagnosis_position",
            value_name="diagnosis_code"
        )

        long_df["diagnosis_code"] = long_df["diagnosis_code"].apply(clean_icd9)
        long_df = long_df[
            (long_df["member_id"] != "")
            &
            (long_df["diagnosis_code"] != "")
            &
            (long_df["diagnosis_date"].notna())
        ]

        parts.append(long_df)

    if len(parts) == 0:
        return pd.DataFrame()

    return pd.concat(parts, ignore_index=True)


def load_inpatient_diagnoses_from_events(ip_events, dataset_label):

    ip = ip_events.copy()
    ip["claim_source"] = "INPATIENT"

    diagnosis_cols = [
        c for c in ip.columns
        if c.startswith("ICD9_DGNS_CD_")
        or c in ["admitting_diagnosis"]
    ]

    keep_cols = ["member_id", "claim_id", "admission_date"] + diagnosis_cols
    keep_cols = [c for c in keep_cols if c in ip.columns]

    temp = ip[keep_cols].copy()
    temp = temp.rename(columns={"admission_date": "diagnosis_date"})
    temp["diagnosis_date"] = safe_date(temp["diagnosis_date"])
    temp["source_sample"] = dataset_label
    temp["claim_source"] = "INPATIENT"

    long_df = temp.melt(
        id_vars=["member_id", "claim_id", "diagnosis_date", "source_sample", "claim_source"],
        value_vars=diagnosis_cols,
        var_name="diagnosis_position",
        value_name="diagnosis_code"
    )

    long_df["diagnosis_code"] = long_df["diagnosis_code"].apply(clean_icd9)
    long_df = long_df[
        (long_df["member_id"] != "")
        &
        (long_df["diagnosis_code"] != "")
        &
        (long_df["diagnosis_date"].notna())
    ]

    return long_df


carrier_train_dx = load_carrier_diagnoses(TRAIN_SAMPLES, "TRAIN Sets 1+2")
carrier_test_dx = load_carrier_diagnoses([TEST_SAMPLE], "TEST Set 3")

op_train_dx = load_outpatient_diagnoses(TRAIN_SAMPLES, "TRAIN Sets 1+2")
op_test_dx = load_outpatient_diagnoses([TEST_SAMPLE], "TEST Set 3")

ip_train_dx = load_inpatient_diagnoses_from_events(ip_train_events, "TRAIN Sets 1+2")
ip_test_dx = load_inpatient_diagnoses_from_events(ip_test_events, "TEST Set 3")

dx_train_timeline = pd.concat(
    [carrier_train_dx, op_train_dx, ip_train_dx],
    ignore_index=True
)

dx_test_timeline = pd.concat(
    [carrier_test_dx, op_test_dx, ip_test_dx],
    ignore_index=True
)

for df in [dx_train_timeline, dx_test_timeline]:
    df["diagnosis_group"] = df["diagnosis_code"].apply(map_icd9_to_group)

dx_train_timeline = dx_train_timeline.drop_duplicates(
    ["member_id", "claim_id", "diagnosis_date", "diagnosis_code", "claim_source"]
)

dx_test_timeline = dx_test_timeline.drop_duplicates(
    ["member_id", "claim_id", "diagnosis_date", "diagnosis_code", "claim_source"]
)

print("\nTRAIN diagnosis timeline shape:", dx_train_timeline.shape)
print(dx_train_timeline["diagnosis_group"].value_counts().head(25))

print("\nTEST diagnosis timeline shape:", dx_test_timeline.shape)
print(dx_test_timeline["diagnosis_group"].value_counts().head(25))

dx_train_timeline.to_csv(
    os.path.join(DIAGNOSIS_OUTPUT_FOLDER, "stepD1_train_diagnosis_timeline.csv"),
    index=False
)

dx_test_timeline.to_csv(
    os.path.join(DIAGNOSIS_OUTPUT_FOLDER, "stepD1_test_diagnosis_timeline.csv"),
    index=False
)

print("\nStep D1 completed.")


STEP D1 - Building diagnosis timeline

Loading Carrier diagnoses TRAIN Sets 1+2 Sample 1: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Carrier_Claims_Sample_1A.csv
Carrier diagnosis rows extracted: 9618645

Loading Carrier diagnoses TRAIN Sets 1+2 Sample 1: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Carrier_Claims_Sample_1B.csv
Carrier diagnosis rows extracted: 9620662

Loading Carrier diagnoses TRAIN Sets 1+2 Sample 2: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Carrier_Claims_Sample_2A.csv
Carrier diagnosis rows extracted: 9628549

Loading Carrier diagnoses TRAIN Sets 1+2 Sample 2: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Carrier_Claims_Sample_2B.csv
Carrier diagnosis rows extracted: 9630864

TRAIN Sets 1+2 combined Carrier diagnosis rows: (38498720, 8)
source_sample
2    19259413
1    19239307
Name: count, dtype: int6

In [ ]:
# ============================================================
# Step D2 - Diagnosis Sequence Mining
# ============================================================

print("\nSTEP D2 - Mining diagnosis sequences")


def build_member_diagnosis_sequences(dx_timeline, dataset_label):

    dx = dx_timeline.copy()

    dx = dx[dx["diagnosis_group"].notna()]
    dx = dx[~dx["diagnosis_group"].isin(["UNKNOWN", "OTHER"])]

    first_seen = (
        dx.groupby(["member_id", "diagnosis_group"])
        .agg(first_seen_date=("diagnosis_date", "min"))
        .reset_index()
    )

    first_seen = first_seen.sort_values(["member_id", "first_seen_date", "diagnosis_group"])

    seq = (
        first_seen
        .groupby("member_id")
        .agg(
            diagnosis_sequence=("diagnosis_group", lambda x: " → ".join(list(x))),
            diagnosis_sequence_count=("diagnosis_group", "count"),
            first_dx_date=("first_seen_date", "min"),
            last_dx_date=("first_seen_date", "max")
        )
        .reset_index()
    )

    seq["diagnosis_progression_days"] = (
        seq["last_dx_date"] - seq["first_dx_date"]
    ).dt.days.fillna(0)

    print(f"\n{dataset_label} member diagnosis sequences:", seq.shape)
    print(seq.head(10))

    return first_seen, seq


dx_train_first_seen, dx_train_member_sequences = build_member_diagnosis_sequences(
    dx_train_timeline,
    "TRAIN Sets 1+2"
)

dx_test_first_seen, dx_test_member_sequences = build_member_diagnosis_sequences(
    dx_test_timeline,
    "TEST Set 3"
)


def build_transition_table(first_seen, dataset_label):

    fs = first_seen.copy()
    fs = fs.sort_values(["member_id", "first_seen_date", "diagnosis_group"])

    fs["next_diagnosis_group"] = fs.groupby("member_id")["diagnosis_group"].shift(-1)
    fs["next_diagnosis_date"] = fs.groupby("member_id")["first_seen_date"].shift(-1)

    transitions = fs.dropna(subset=["next_diagnosis_group"]).copy()

    transitions["diagnosis_transition"] = (
        transitions["diagnosis_group"].astype(str)
        + " → "
        + transitions["next_diagnosis_group"].astype(str)
    )

    transitions["days_to_next_diagnosis"] = (
        transitions["next_diagnosis_date"] - transitions["first_seen_date"]
    ).dt.days

    transition_summary = (
        transitions
        .groupby("diagnosis_transition")
        .agg(
            transition_count=("member_id", "count"),
            unique_members=("member_id", "nunique"),
            avg_days_to_next=("days_to_next_diagnosis", "mean"),
            median_days_to_next=("days_to_next_diagnosis", "median")
        )
        .reset_index()
        .sort_values("transition_count", ascending=False)
    )

    print(f"\nTop {dataset_label} diagnosis transitions:")
    print(transition_summary.head(30))

    return transitions, transition_summary


dx_train_transitions, dx_train_transition_summary = build_transition_table(
    dx_train_first_seen,
    "TRAIN Sets 1+2"
)

dx_test_transitions, dx_test_transition_summary = build_transition_table(
    dx_test_first_seen,
    "TEST Set 3"
)

dx_train_transition_summary.to_csv(
    os.path.join(DIAGNOSIS_OUTPUT_FOLDER, "stepD2_train_diagnosis_transition_summary.csv"),
    index=False
)

dx_test_transition_summary.to_csv(
    os.path.join(DIAGNOSIS_OUTPUT_FOLDER, "stepD2_test_diagnosis_transition_summary.csv"),
    index=False
)

dx_train_member_sequences.to_csv(
    os.path.join(DIAGNOSIS_OUTPUT_FOLDER, "stepD2_train_member_diagnosis_sequences.csv"),
    index=False
)

dx_test_member_sequences.to_csv(
    os.path.join(DIAGNOSIS_OUTPUT_FOLDER, "stepD2_test_member_diagnosis_sequences.csv"),
    index=False
)

print("\nStep D2 completed.")


STEP D2 - Mining diagnosis sequences

TRAIN Sets 1+2 member diagnosis sequences: (189145, 6)
          member_id                                 diagnosis_sequence  \
0  00000B48BCF4AD29  DEHYDRATION → FALL_FRACTURE → CAD → HYPERTENSI...   
1  0000525AB30E4DEF  FALL_FRACTURE → CANCER → HYPERTENSION → UTI → ...   
2  00009C897C3D8372  HYPERTENSION → CKD → COPD → PNEUMONIA → DEPRES...   
3  00013D2EFD8E45D1         CHF → HYPERTENSION → DEPRESSION_BEHAVIORAL   
4  00016F745862898F  FALL_FRACTURE → HYPERTENSION → DEPRESSION_BEHA...   
5  0001FDD721E223DC                                           DIABETES   
6  00021CA6FF03E670                      HYPERTENSION → CAD → DIABETES   
7  00024B3D2352D2D0  HYPERTENSION → DEMENTIA → DIABETES → UTI → CAD...   
8  0002DAE1C81CC70D         DIABETES → CKD → COPD → HYPERTENSION → CAD   
9  0002EC0BCA99CACF  RESPIRATORY_FAILURE → ASTHMA → CELLULITIS → FA...   

   diagnosis_sequence_count first_dx_date last_dx_date  \
0                        16    20

In [ ]:
# ============================================================
# Step D3 - Link Diagnosis Pathways to Hospitalization Risk
# ============================================================

print("\nSTEP D3 - Linking diagnosis pathways to hospitalization")


def add_sequence_to_timeline(timeline, member_sequences, dataset_label):

    df = timeline.copy()

    df = df.drop(
        columns=[
            "diagnosis_sequence",
            "diagnosis_sequence_count",
            "diagnosis_progression_days"
        ],
        errors="ignore"
    )

    seq = member_sequences[
        [
            "member_id",
            "diagnosis_sequence",
            "diagnosis_sequence_count",
            "diagnosis_progression_days"
        ]
    ].copy()

    df = df.merge(
        seq,
        on="member_id",
        how="left"
    )

    df["diagnosis_sequence"] = df["diagnosis_sequence"].fillna("NO_GROUPED_DIAGNOSIS")
    df["diagnosis_sequence_count"] = safe_numeric(df["diagnosis_sequence_count"], fill_value=0)
    df["diagnosis_progression_days"] = safe_numeric(df["diagnosis_progression_days"], fill_value=0)

    print(f"{dataset_label} timeline with diagnosis sequences:", df.shape)

    return df


df_train_diag = add_sequence_to_timeline(
    df_train_timeline,
    dx_train_member_sequences,
    "TRAIN Sets 1+2"
)

df_test_diag = add_sequence_to_timeline(
    df_test_timeline,
    dx_test_member_sequences,
    "TEST Set 3"
)


diagnosis_pathway_risk_table = (
    df_train_diag
    .groupby("diagnosis_sequence")
    .agg(
        patient_month_count=("member_id", "count"),
        unique_members=("member_id", "nunique"),
        hospitalization_rate=(TARGET_LABEL_NAME, "mean"),
        avg_diagnosis_sequence_count=("diagnosis_sequence_count", "mean"),
        avg_diagnosis_progression_days=("diagnosis_progression_days", "mean")
    )
    .reset_index()
)

global_hosp_rate_d = df_train_diag[TARGET_LABEL_NAME].mean()

diagnosis_pathway_risk_table["lift_vs_global"] = (
    diagnosis_pathway_risk_table["hospitalization_rate"] /
    max(global_hosp_rate_d, 0.000001)
)

diagnosis_pathway_risk_table["high_risk_diagnosis_pathway_flag"] = (
    (diagnosis_pathway_risk_table["patient_month_count"] >= MIN_PATHWAY_ROWS)
    &
    (diagnosis_pathway_risk_table["hospitalization_rate"] >= HIGH_RISK_DIAG_PATHWAY_RATE)
).astype(int)

diagnosis_pathway_risk_table = diagnosis_pathway_risk_table.sort_values(
    ["hospitalization_rate", "patient_month_count"],
    ascending=[False, False]
)

print("\nTop diagnosis pathways by hospitalization rate:")
print(diagnosis_pathway_risk_table.head(30))

diagnosis_pathway_risk_table.to_csv(
    os.path.join(DIAGNOSIS_OUTPUT_FOLDER, "stepD3_train_diagnosis_pathway_risk_table.csv"),
    index=False
)

print("\nStep D3 completed.")


STEP D3 - Linking diagnosis pathways to hospitalization
TRAIN Sets 1+2 timeline with diagnosis sequences: (7265664, 131)
TEST Set 3 timeline with diagnosis sequences: (3634740, 131)

Top diagnosis pathways by hospitalization rate:
                                       diagnosis_sequence  \
62709   DEHYDRATION → CANCER → CELLULITIS → DIABETES →...   
90      AKI → CAD → CHF → CKD → UTI → CELLULITIS → DIA...   
42979   CHF → DEMENTIA → RESPIRATORY_FAILURE → CELLULI...   
53412   CKD → STROKE_TIA → SEPSIS → AKI → CAD → HYPERT...   
81631   DIABETES → CAD → RESPIRATORY_FAILURE → HYPERTE...   
44812   CHF → FALL_FRACTURE → PNEUMONIA → UTI → DEMENT...   
49709   CKD → DIABETES → CHF → CELLULITIS → COPD → DEP...   
54752   COPD → CANCER → HYPERTENSION → CKD → SEPSIS → ...   
108690  FALL_FRACTURE → HYPERTENSION → CHF → RESPIRATO...   
27598   CANCER → CHF → COPD → HYPERTENSION → CKD → RES...   
29526   CANCER → DEHYDRATION → PNEUMONIA → SEPSIS → DI...   
47611   CKD → CAD → CELLULITIS → DIA

In [ ]:
print([x for x in globals().keys() if "diagnosis" in x.lower()])

['DIAGNOSIS_OUTPUT_FOLDER', 'DIAGNOSIS_LOOKBACK_DAYS', 'DIAGNOSIS_GROUP_MAP', 'reload_inpatient_events_for_diagnosis', 'build_member_diagnosis_sequences', 'DIAGNOSIS_CHECKPOINT_FOLDER']


In [ ]:
# ============================================================
# Step D2.5 - Reload Diagnosis Timeline Objects from Saved Files
# ============================================================

print("\nSTEP D2.5 - Reloading diagnosis timeline objects")

import os
import pandas as pd
import gc

DIAGNOSIS_OUTPUT_FOLDER = os.path.join(
    BASE_FOLDER,
    "Diagnosis_Progression_Mining"
)

diagnosis_train_timeline_file = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD1_train_diagnosis_timeline.csv"
)

diagnosis_test_timeline_file = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD1_test_diagnosis_timeline.csv"
)

if not os.path.exists(diagnosis_train_timeline_file):
    raise FileNotFoundError(diagnosis_train_timeline_file)

if not os.path.exists(diagnosis_test_timeline_file):
    raise FileNotFoundError(diagnosis_test_timeline_file)

diagnosis_train_timeline = pd.read_csv(
    diagnosis_train_timeline_file,
    low_memory=False
)

diagnosis_test_timeline = pd.read_csv(
    diagnosis_test_timeline_file,
    low_memory=False
)

for df in [diagnosis_train_timeline, diagnosis_test_timeline]:
    if "prediction_month" in df.columns:
        df["prediction_month"] = pd.to_datetime(
            df["prediction_month"],
            errors="coerce"
        )

print("diagnosis_train_timeline:", diagnosis_train_timeline.shape)
print("diagnosis_test_timeline:", diagnosis_test_timeline.shape)

print("\nTrain columns:")
print(diagnosis_train_timeline.columns.tolist())

print("\nTest columns:")
print(diagnosis_test_timeline.columns.tolist())

gc.collect()

print("\nStep D2.5 completed successfully.")


STEP D2.5 - Reloading diagnosis timeline objects
diagnosis_train_timeline: (35667966, 9)
diagnosis_test_timeline: (17846788, 9)

Train columns:
['member_id', 'claim_id', 'diagnosis_date', 'source_sample', 'carrier_part', 'claim_source', 'diagnosis_position', 'diagnosis_code', 'diagnosis_group']

Test columns:
['member_id', 'claim_id', 'diagnosis_date', 'source_sample', 'carrier_part', 'claim_source', 'diagnosis_position', 'diagnosis_code', 'diagnosis_group']

Step D2.5 completed successfully.


In [ ]:
# ============================================================
# Step D2.6 - Reload Hospitalization Timelines for D3B
# ============================================================

print("\nSTEP D2.6 - Reloading hospitalization timelines for D3B")

import os
import pandas as pd
import gc

train_h2_file = os.path.join(
    OUTPUT_FOLDER,
    "stepH2_train_sets1_2_timeline_with_labels.csv"
)

test_h2_file = os.path.join(
    OUTPUT_FOLDER,
    "stepH2_test_set3_timeline_with_labels.csv"
)

print("Train H2 file exists:", os.path.exists(train_h2_file))
print("Test H2 file exists:", os.path.exists(test_h2_file))

if not os.path.exists(train_h2_file):
    raise FileNotFoundError(train_h2_file)

if not os.path.exists(test_h2_file):
    raise FileNotFoundError(test_h2_file)

df_train_timeline = pd.read_csv(
    train_h2_file,
    low_memory=False
)

df_test_timeline = pd.read_csv(
    test_h2_file,
    low_memory=False
)

for df in [df_train_timeline, df_test_timeline]:
    if "prediction_month" in df.columns:
        df["prediction_month"] = pd.to_datetime(
            df["prediction_month"],
            errors="coerce"
        )

print("df_train_timeline:", df_train_timeline.shape)
print("df_test_timeline:", df_test_timeline.shape)

gc.collect()

print("\nStep D2.6 completed successfully.")


STEP D2.6 - Reloading hospitalization timelines for D3B
Train H2 file exists: True
Test H2 file exists: True
df_train_timeline: (7265664, 62)
df_test_timeline: (3634740, 62)

Step D2.6 completed successfully.


In [ ]:
# ============================================================
# Step D3A-CP - Diagnosis Mining Checkpoint
# REVISED: Does NOT require df_train_timeline / df_test_timeline
# ============================================================

print("\nSTEP D3A-CP - Diagnosis mining checkpoint save/recovery")

import os
import gc
import json
import pandas as pd
import numpy as np

DIAGNOSIS_OUTPUT_FOLDER = os.path.join(
    BASE_FOLDER,
    "Diagnosis_Progression_Mining"
)

DIAGNOSIS_CHECKPOINT_FOLDER = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "checkpoint_before_D3B"
)

os.makedirs(DIAGNOSIS_CHECKPOINT_FOLDER, exist_ok=True)

# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

SAVE_CHECKPOINT = True
LOAD_CHECKPOINT = False

# After a crash:
# SAVE_CHECKPOINT = False
# LOAD_CHECKPOINT = True

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def save_df_checkpoint(df, file_name):
    path = os.path.join(DIAGNOSIS_CHECKPOINT_FOLDER, file_name)
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df.to_parquet(path, index=False)
    print("Saved:", path, df.shape)
    return path


def load_df_checkpoint(file_name):
    path = os.path.join(DIAGNOSIS_CHECKPOINT_FOLDER, file_name)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    df = df.loc[:, ~df.columns.duplicated()].copy()
    print("Loaded:", path, df.shape)
    return df


# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------

if SAVE_CHECKPOINT:

    required_objects = [
        "diagnosis_train_timeline",
        "diagnosis_test_timeline"
    ]

    missing_required = [
        obj for obj in required_objects
        if obj not in globals()
    ]

    if missing_required:
        raise ValueError(
            f"Missing required diagnosis objects: {missing_required}. "
            "Run D1/D2 first or alias the diagnosis timeline variables."
        )

    optional_objects = [
        "diagnosis_group_summary",
        "diagnosis_groups",
        "diagnosis_train_features",
        "diagnosis_test_features",
        "diagnosis_transition_train",
        "diagnosis_transition_test"
    ]

    manifest = []

    for obj in required_objects + optional_objects:

        if obj not in globals():
            continue

        value = globals()[obj]

        if isinstance(value, pd.DataFrame):
            file_name = f"{obj}_before_D3B.parquet"
            path = save_df_checkpoint(value, file_name)

            manifest.append({
                "object_name": obj,
                "file_name": file_name,
                "path": path,
                "type": "dataframe"
            })

        elif isinstance(value, (list, dict, tuple, set)):
            file_name = f"{obj}_before_D3B.json"
            path = os.path.join(DIAGNOSIS_CHECKPOINT_FOLDER, file_name)

            serializable_value = list(value) if isinstance(value, set) else value

            with open(path, "w") as f:
                json.dump(serializable_value, f, indent=2)

            print("Saved:", path)

            manifest.append({
                "object_name": obj,
                "file_name": file_name,
                "path": path,
                "type": "json"
            })

    manifest_df = pd.DataFrame(manifest)

    manifest_file = os.path.join(
        DIAGNOSIS_CHECKPOINT_FOLDER,
        "checkpoint_manifest_before_D3B.csv"
    )

    manifest_df.to_csv(manifest_file, index=False)

    print("\nCheckpoint saved successfully.")
    print("Manifest:", manifest_file)
    print("Objects saved:", len(manifest_df))

    gc.collect()


# ------------------------------------------------------------
# Load checkpoint after crash
# ------------------------------------------------------------

if LOAD_CHECKPOINT:

    manifest_file = os.path.join(
        DIAGNOSIS_CHECKPOINT_FOLDER,
        "checkpoint_manifest_before_D3B.csv"
    )

    if not os.path.exists(manifest_file):
        raise FileNotFoundError(manifest_file)

    manifest_df = pd.read_csv(manifest_file)

    for _, row in manifest_df.iterrows():

        obj = row["object_name"]
        file_name = row["file_name"]
        obj_type = row.get("type", "")

        if file_name.endswith(".parquet"):
            globals()[obj] = load_df_checkpoint(file_name)

        elif file_name.endswith(".json"):
            path = os.path.join(DIAGNOSIS_CHECKPOINT_FOLDER, file_name)

            with open(path, "r") as f:
                globals()[obj] = json.load(f)

            print("Loaded:", obj, path)

    # Date cleanup
    for obj in [
        "diagnosis_train_timeline",
        "diagnosis_test_timeline"
    ]:
        if obj in globals():
            for date_col in [
                "prediction_month",
                PREDICTION_DATE_COL if "PREDICTION_DATE_COL" in globals() else None
            ]:
                if date_col and date_col in globals()[obj].columns:
                    globals()[obj][date_col] = pd.to_datetime(
                        globals()[obj][date_col],
                        errors="coerce"
                    )

    print("\nCheckpoint restored successfully.")

    for obj in manifest_df["object_name"].tolist():
        if obj in globals():
            if isinstance(globals()[obj], pd.DataFrame):
                print(obj, globals()[obj].shape)
            else:
                print(obj, type(globals()[obj]))

    gc.collect()


print("\nStep D3A-CP completed.")


STEP D3A-CP - Diagnosis mining checkpoint save/recovery
Saved: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Diagnosis_Progression_Mining/checkpoint_before_D3B/diagnosis_train_timeline_before_D3B.parquet (35667966, 9)
Saved: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Diagnosis_Progression_Mining/checkpoint_before_D3B/diagnosis_test_timeline_before_D3B.parquet (17846788, 9)

Checkpoint saved successfully.
Manifest: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Diagnosis_Progression_Mining/checkpoint_before_D3B/checkpoint_manifest_before_D3B.csv
Objects saved: 2

Step D3A-CP completed.


In [ ]:
import os
import glob

diag_files = glob.glob(
    os.path.join(
        BASE_FOLDER,
        "Diagnosis_Progression_Mining",
        "*"
    )
)

for f in diag_files:
    print(os.path.basename(f))

stepD1_train_diagnosis_timeline.csv
stepD1_test_diagnosis_timeline.csv
stepD2_train_diagnosis_transition_summary.csv
stepD2_test_diagnosis_transition_summary.csv
stepD2_train_member_diagnosis_sequences.csv
stepD2_test_member_diagnosis_sequences.csv
stepD3_train_diagnosis_pathway_risk_table.csv
checkpoint_before_D3B
stepD3B_population_diagnosis_pathway_risk_table.csv
stepD3B_population_diagnosis_pathway_test_validation.csv
stepD3B_train_population_transitions.csv
stepD3B_test_population_transitions.csv
stepD3B_train_pathway_member_months.csv
stepD3B_test_pathway_member_months.csv
stepD3C_ranked_high_risk_population_pathways.csv
stepD3C_train_pathway_exposure_features.csv
stepD3C_test_pathway_exposure_features.csv
stepD3C_train_exposure_tier_validation.csv
stepD3C_test_exposure_tier_validation.csv


In [ ]:
# ============================================================
# Reload H2 timelines required by D3B
# ============================================================

print("\nReloading H2 hospitalization timelines for D3B")

import os
import pandas as pd
import gc

train_h2_file = os.path.join(
    OUTPUT_FOLDER,
    "stepH2_train_sets1_2_timeline_with_labels.csv"
)

test_h2_file = os.path.join(
    OUTPUT_FOLDER,
    "stepH2_test_set3_timeline_with_labels.csv"
)

print("Train H2 exists:", os.path.exists(train_h2_file))
print("Test H2 exists:", os.path.exists(test_h2_file))

df_train_timeline = pd.read_csv(train_h2_file, low_memory=False)
df_test_timeline = pd.read_csv(test_h2_file, low_memory=False)

df_train_timeline[PREDICTION_DATE_COL] = pd.to_datetime(
    df_train_timeline[PREDICTION_DATE_COL],
    errors="coerce"
)

df_test_timeline[PREDICTION_DATE_COL] = pd.to_datetime(
    df_test_timeline[PREDICTION_DATE_COL],
    errors="coerce"
)

print("df_train_timeline:", df_train_timeline.shape)
print("df_test_timeline:", df_test_timeline.shape)

gc.collect()

print("H2 timelines restored.")


Reloading H2 hospitalization timelines for D3B
Train H2 exists: True
Test H2 exists: True
df_train_timeline: (7265664, 62)
df_test_timeline: (3634740, 62)
H2 timelines restored.


In [ ]:
# ============================================================
# Step D3B - Population-Level Diagnosis Pathway Mining
# REVISED FULL VERSION
# Creates standardized pathway table for D4/D5
# ============================================================

print("\nSTEP D3B - Population-level diagnosis pathway mining")


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_objects = [
    "dx_train_timeline", # Corrected variable name
    "dx_test_timeline", # Corrected variable name
    "df_train_timeline",
    "df_test_timeline"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"{obj} not found. Run D1/H2 steps first.")

if "safe_numeric" not in globals():
    def safe_numeric(series, fill_value=0):
        return pd.to_numeric(series, errors="coerce").fillna(fill_value)


# ------------------------------------------------------------
# Config defaults
# ------------------------------------------------------------

if "MIN_POPULATION_PATHWAY_MONTHS" not in globals():
    MIN_POPULATION_PATHWAY_MONTHS = 500

if "MIN_POPULATION_PATHWAY_MEMBERS" not in globals():
    MIN_POPULATION_PATHWAY_MEMBERS = 20

if "HIGH_RISK_POPULATION_PATHWAY_RATE" not in globals():
    HIGH_RISK_POPULATION_PATHWAY_RATE = 0.10

if "HIGH_RISK_POPULATION_PATHWAY_LIFT" not in globals():
    HIGH_RISK_POPULATION_PATHWAY_LIFT = 2.0


# ------------------------------------------------------------
# Prepare diagnosis timeline
# ------------------------------------------------------------

def prep_dx_timeline_for_pathways(dx, dataset_label):

    dx = dx.copy()

    required_cols = ["member_id", "diagnosis_date", "diagnosis_group"]

    missing = [c for c in required_cols if c not in dx.columns]
    if len(missing) > 0:
        raise ValueError(f"{dataset_label} diagnosis timeline missing: {missing}")

    dx["diagnosis_date"] = pd.to_datetime(
        dx["diagnosis_date"],
        errors="coerce"
    )

    dx["diagnosis_group"] = (
        dx["diagnosis_group"]
        .fillna("OTHER")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    dx = dx.dropna(subset=["member_id", "diagnosis_date", "diagnosis_group"])

    # Remove broad OTHER from pathway mining to avoid noisy sequences
    dx = dx[dx["diagnosis_group"] != "OTHER"].copy()

    # Keep one diagnosis group per member/date/group
    dx = (
        dx[["member_id", "diagnosis_date", "diagnosis_group"]]
        .drop_duplicates()
        .sort_values(["member_id", "diagnosis_date", "diagnosis_group"])
        .reset_index(drop=True)
    )

    print(f"{dataset_label} prepared diagnosis timeline:", dx.shape)

    return dx


dx_train_for_pathways = prep_dx_timeline_for_pathways(
    dx_train_timeline, # Corrected variable name
    "TRAIN Sets 1+2"
)

dx_test_for_pathways = prep_dx_timeline_for_pathways(
    dx_test_timeline, # Corrected variable name
    "TEST Set 3"
)


# ------------------------------------------------------------
# Build member-level ordered diagnosis event table
# ------------------------------------------------------------

def build_member_ordered_dx(dx, dataset_label):

    dx = dx.copy()

    dx = dx.sort_values(["member_id", "diagnosis_date", "diagnosis_group"])

    dx["dx_order"] = (
        dx.groupby("member_id")
        .cumcount()
    )

    print(f"{dataset_label} ordered diagnosis rows:", dx.shape)

    return dx


dx_train_ordered = build_member_ordered_dx(
    dx_train_for_pathways,
    "TRAIN Sets 1+2"
)

dx_test_ordered = build_member_ordered_dx(
    dx_test_for_pathways,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Build 2-step and 3-step member transitions
# ------------------------------------------------------------

def build_population_transitions(dx_ordered, dataset_label):

    dx = dx_ordered.copy()

    # Current diagnosis
    base = dx[
        ["member_id", "dx_order", "diagnosis_date", "diagnosis_group"]
    ].copy()

    base = base.rename(
        columns={
            "diagnosis_date": "dx_date_1",
            "diagnosis_group": "dx_1"
        }
    )

    # Next diagnosis
    next1 = dx[
        ["member_id", "dx_order", "diagnosis_date", "diagnosis_group"]
    ].copy()

    next1["dx_order"] = next1["dx_order"] - 1

    next1 = next1.rename(
        columns={
            "diagnosis_date": "dx_date_2",
            "diagnosis_group": "dx_2"
        }
    )

    two_step = base.merge(
        next1,
        on=["member_id", "dx_order"],
        how="inner"
    )

    two_step = two_step[
        two_step["dx_1"] != two_step["dx_2"]
    ].copy()

    two_step["diagnosis_transition"] = (
        two_step["dx_1"] + " \u2192 " + two_step["dx_2"]
    )

    two_step["pathway_type"] = "2_STEP"

    two_step["days_to_next_diagnosis"] = (
        two_step["dx_date_2"] - two_step["dx_date_1"]
    ).dt.days.clip(lower=0)

    two_step = two_step[
        [
            "member_id",
            "pathway_type",
            "diagnosis_transition",
            "dx_date_1",
            "dx_date_2",
            "days_to_next_diagnosis"
        ]
    ].drop_duplicates()

    # Third diagnosis
    next2 = dx[
        ["member_id", "dx_order", "diagnosis_date", "diagnosis_group"]
    ].copy()

    next2["dx_order"] = next2["dx_order"] - 2

    next2 = next2.rename(
        columns={
            "diagnosis_date": "dx_date_3",
            "diagnosis_group": "dx_3"
        }
    )

    three_step = base.merge(
        next1,
        on=["member_id", "dx_order"],
        how="inner"
    ).merge(
        next2,
        on=["member_id", "dx_order"],
        how="inner"
    )

    three_step = three_step[
        (three_step["dx_1"] != three_step["dx_2"]) &
        (three_step["dx_2"] != three_step["dx_3"])
    ].copy()

    three_step["diagnosis_transition"] = (
        three_step["dx_1"] + " \u2192 " +
        three_step["dx_2"] + " \u2192 " +
        three_step["dx_3"]
    )

    three_step["pathway_type"] = "3_STEP"

    three_step["days_to_next_diagnosis"] = (
        three_step["dx_date_3"] - three_step["dx_date_1"]
    ).dt.days.clip(lower=0)

    three_step = three_step[
        [
            "member_id",
            "pathway_type",
            "diagnosis_transition",
            "dx_date_1",
            "dx_date_3",
            "days_to_next_diagnosis"
        ]
    ].drop_duplicates()

    three_step = three_step.rename(columns={"dx_date_3": "dx_date_2"})

    transitions = pd.concat(
        [two_step, three_step],
        ignore_index=True
    )

    transitions = transitions.drop_duplicates(
        [
            "member_id",
            "pathway_type",
            "diagnosis_transition",
            "dx_date_1",
            "dx_date_2"
        ]
    )

    print(
        f"{dataset_label} population transitions:",
        transitions.shape
    )

    print(
        transitions["pathway_type"]
        .value_counts(dropna=False)
    )

    return transitions


dx_train_population_transitions = build_population_transitions(
    dx_train_ordered,
    "TRAIN Sets 1+2"
)

dx_test_population_transitions = build_population_transitions(
    dx_test_ordered,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Attach transitions to member-month timeline
# ------------------------------------------------------------

def attach_pathways_to_member_months(timeline, transitions, dataset_label):

    df = timeline.copy()
    trans = transitions.copy()

    df[PREDICTION_DATE_COL] = pd.to_datetime(
        df[PREDICTION_DATE_COL],
        errors="coerce"
    )

    trans["dx_date_2"] = pd.to_datetime(
        trans["dx_date_2"],
        errors="coerce"
    )

    # Merge transitions to all member-months, then retain pathways completed
    # on or before prediction date.
    merged = df[
        ["member_id", PREDICTION_DATE_COL, TARGET_LABEL_NAME]
    ].merge(
        trans[
            [
                "member_id",
                "pathway_type",
                "diagnosis_transition",
                "dx_date_1",
                "dx_date_2",
                "days_to_next_diagnosis"
            ]
        ],
        on="member_id",
        how="inner"
    )

    merged = merged[
        merged["dx_date_2"] <= merged[PREDICTION_DATE_COL]
    ].copy()

    merged["days_since_pathway_completed"] = (
        merged[PREDICTION_DATE_COL] - merged["dx_date_2"]
    ).dt.days.clip(lower=0)

    # Optional freshness control: keep pathways completed in prior 365 days
    merged = merged[
        merged["days_since_pathway_completed"] <= 365
    ].copy()

    print(
        f"{dataset_label} pathway-member-month rows:",
        merged.shape
    )

    return merged


train_pathway_member_months = attach_pathways_to_member_months(
    df_train_timeline,
    dx_train_population_transitions,
    "TRAIN Sets 1+2"
)

test_pathway_member_months = attach_pathways_to_member_months(
    df_test_timeline,
    dx_test_population_transitions,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Build standardized TRAIN pathway risk table
# ------------------------------------------------------------

def build_population_pathway_risk_table(pathway_member_months, dataset_label):

    global_hosp_rate = pathway_member_months[TARGET_LABEL_NAME].mean()

    pathway_risk = (
        pathway_member_months
        .groupby(["pathway_type", "diagnosis_transition"])
        .agg(
            patient_month_count=("member_id", "count"),
            unique_members=("member_id", "nunique"),
            hospitalization_rate=(TARGET_LABEL_NAME, "mean"),
            avg_days_to_next_diagnosis=("days_to_next_diagnosis", "mean"),
            median_min_days_to_next=("days_to_next_diagnosis", "median"),
            avg_max_days_to_next=("days_to_next_diagnosis", "max")
        )
        .reset_index()
    )

    pathway_risk["lift_vs_global"] = (
        pathway_risk["hospitalization_rate"] /
        max(global_hosp_rate, 0.000001)
    )

    pathway_risk["high_risk_population_pathway_flag"] = (
        (
            pathway_risk["patient_month_count"] >= MIN_POPULATION_PATHWAY_MONTHS
        )
        &
        (
            pathway_risk["unique_members"] >= MIN_POPULATION_PATHWAY_MEMBERS
        )
        &
        (
            pathway_risk["hospitalization_rate"] >= HIGH_RISK_POPULATION_PATHWAY_RATE
        )
        &
        (
            pathway_risk["lift_vs_global"] >= HIGH_RISK_POPULATION_PATHWAY_LIFT
        )
    ).astype(int)

    pathway_risk = pathway_risk.sort_values(
        [
            "hospitalization_rate",
            "patient_month_count"
        ],
        ascending=[False, False]
    ).reset_index(drop=True)

    print(f"\nTop {dataset_label} population-level diagnosis pathways:")
    print(pathway_risk.head(50))

    print("\nHigh-risk population pathways:")
    print(
        pathway_risk[
            pathway_risk["high_risk_population_pathway_flag"] == 1
        ].head(50)
    )

    print("\nPathway risk table summary:")
    print(
        pathway_risk[
            [
                "patient_month_count",
                "unique_members",
                "hospitalization_rate",
                "lift_vs_global",
                "high_risk_population_pathway_flag"
            ]
        ].describe()
    )

    return pathway_risk, global_hosp_rate


population_diagnosis_pathway_risk_table, global_pathway_hosp_rate = (
    build_population_pathway_risk_table(
        train_pathway_member_months,
        "TRAIN Sets 1+2"
    )
)


# ------------------------------------------------------------
# Validate TRAIN-derived pathways on TEST
# ------------------------------------------------------------

def validate_pathways_on_test(
    test_pathway_member_months,
    train_pathway_risk_table
):

    test_eval = (
        test_pathway_member_months
        .groupby(["pathway_type", "diagnosis_transition"])
        .agg(
            test_patient_month_count=("member_id", "count"),
            test_unique_members=("member_id", "nunique"),
            test_hospitalization_rate=(TARGET_LABEL_NAME, "mean")
        )
        .reset_index()
    )

    compare = test_eval.merge(
        train_pathway_risk_table[
            [
                "pathway_type",
                "diagnosis_transition",
                "hospitalization_rate",
                "lift_vs_global",
                "high_risk_population_pathway_flag"
            ]
        ],
        on=["pathway_type", "diagnosis_transition"],
        how="inner"
    )

    compare = compare.rename(
        columns={
            "hospitalization_rate": "train_pathway_rate",
            "lift_vs_global": "train_pathway_lift"
        }
    )

    compare["test_vs_train_rate_gap"] = (
        compare["test_hospitalization_rate"] -
        compare["train_pathway_rate"]
    )

    compare = compare.sort_values(
        "test_hospitalization_rate",
        ascending=False
    ).reset_index(drop=True)

    print("\nTEST validation of TRAIN-derived diagnosis pathways:")
    print(compare.head(50))

    return compare


population_diagnosis_pathway_test_validation = validate_pathways_on_test(
    test_pathway_member_months,
    population_diagnosis_pathway_risk_table
)


# ------------------------------------------------------------
# Save standardized outputs
# ------------------------------------------------------------

population_diagnosis_pathway_risk_table.to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3B_population_diagnosis_pathway_risk_table.csv"
    ),
    index=False
)

population_diagnosis_pathway_test_validation.to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3B_population_diagnosis_pathway_test_validation.csv"
    ),
    index=False
)

dx_train_population_transitions.to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3B_train_population_transitions.csv"
    ),
    index=False
)

dx_test_population_transitions.to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3B_test_population_transitions.csv"
    ),
    index=False
)

train_pathway_member_months.to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3B_train_pathway_member_months.csv"
    ),
    index=False
)

test_pathway_member_months.to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3B_test_pathway_member_months.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

required_d3b_cols = [
    "pathway_type",
    "diagnosis_transition",
    "patient_month_count",
    "unique_members",
    "hospitalization_rate",
    "avg_days_to_next_diagnosis",
    "median_min_days_to_next",
    "avg_max_days_to_next",
    "lift_vs_global",
    "high_risk_population_pathway_flag"
]

missing_d3b_cols = [
    c for c in required_d3b_cols
    if c not in population_diagnosis_pathway_risk_table.columns
]

if len(missing_d3b_cols) > 0:
    raise ValueError(f"D3B missing required standardized columns: {missing_d3b_cols}")

print("\nStep D3B completed successfully.")
print("Standardized population_diagnosis_pathway_risk_table is ready for D4/D5.")


STEP D3B - Population-level diagnosis pathway mining
TRAIN Sets 1+2 prepared diagnosis timeline: (6634515, 3)
TEST Set 3 prepared diagnosis timeline: (3323730, 3)
TRAIN Sets 1+2 ordered diagnosis rows: (6634515, 4)
TEST Set 3 ordered diagnosis rows: (3323730, 4)
TRAIN Sets 1+2 population transitions: (10953526, 6)
pathway_type
2_STEP    5803489
3_STEP    5150037
Name: count, dtype: int64
TEST Set 3 population transitions: (5485810, 6)
pathway_type
2_STEP    2906798
3_STEP    2579012
Name: count, dtype: int64
TRAIN Sets 1+2 pathway-member-month rows: (115519692, 9)
TEST Set 3 pathway-member-month rows: (57836043, 9)

Top TRAIN Sets 1+2 population-level diagnosis pathways:
   pathway_type                          diagnosis_transition  \
0        3_STEP                      SEPSIS → ASTHMA → SEPSIS   
1        3_STEP              CELLULITIS → STROKE_TIA → SEPSIS   
2        3_STEP                     SEPSIS → CELLULITIS → AKI   
3        3_STEP                         CKD → ASTHMA → SEPS

In [ ]:
# ============================================================
# Step D3C - High-Risk Diagnosis Pathway Exposure Flags
# Builds Top-N pathway exposure features from D3B outputs
# ============================================================

print("\nSTEP D3C - Building high-risk diagnosis pathway exposure flags")


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_objects = [
    "df_train_timeline",
    "df_test_timeline",
    "population_diagnosis_pathway_risk_table",
    "train_pathway_member_months",
    "test_pathway_member_months"
]

for obj in required_objects:
    if obj not in globals():
        raise ValueError(f"{obj} not found. Run revised D3B first.")

if "safe_numeric" not in globals():
    def safe_numeric(series, fill_value=0):
        return pd.to_numeric(series, errors="coerce").fillna(fill_value)


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

TOP_PATHWAY_CUTOFFS = [10, 25, 50, 100]

MIN_HIGH_RISK_PATHWAY_LIFT = 2.0
MIN_HIGH_RISK_PATHWAY_RATE = 0.10
MIN_HIGH_RISK_PATHWAY_MEMBER_SUPPORT = 20
MIN_HIGH_RISK_PATHWAY_MONTH_SUPPORT = 500


# ------------------------------------------------------------
# Prepare ranked D3B pathway table
# ------------------------------------------------------------

pathway_rank_table = population_diagnosis_pathway_risk_table.copy()

required_cols = [
    "pathway_type",
    "diagnosis_transition",
    "patient_month_count",
    "unique_members",
    "hospitalization_rate",
    "lift_vs_global",
    "high_risk_population_pathway_flag"
]

missing = [c for c in required_cols if c not in pathway_rank_table.columns]
if len(missing) > 0:
    raise ValueError(f"D3B pathway table missing columns: {missing}")

for col in [
    "patient_month_count",
    "unique_members",
    "hospitalization_rate",
    "lift_vs_global",
    "high_risk_population_pathway_flag"
]:
    pathway_rank_table[col] = safe_numeric(pathway_rank_table[col], fill_value=0)

pathway_rank_table["diagnosis_transition"] = (
    pathway_rank_table["diagnosis_transition"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

# Keep only reliable high-risk pathways
pathway_rank_table["d3c_eligible_high_risk_pathway"] = (
    (pathway_rank_table["patient_month_count"] >= MIN_HIGH_RISK_PATHWAY_MONTH_SUPPORT) &
    (pathway_rank_table["unique_members"] >= MIN_HIGH_RISK_PATHWAY_MEMBER_SUPPORT) &
    (pathway_rank_table["hospitalization_rate"] >= MIN_HIGH_RISK_PATHWAY_RATE) &
    (pathway_rank_table["lift_vs_global"] >= MIN_HIGH_RISK_PATHWAY_LIFT)
).astype(int)

eligible_pathways = pathway_rank_table[
    pathway_rank_table["d3c_eligible_high_risk_pathway"] == 1
].copy()

# Severity score balances rate, lift, and support
eligible_pathways["pathway_severity_score"] = (
    0.45 * eligible_pathways["hospitalization_rate"].clip(0, 1) +
    0.35 * (
        eligible_pathways["lift_vs_global"] /
        max(eligible_pathways["lift_vs_global"].quantile(0.99), 1)
    ).clip(0, 1) +
    0.20 * (
        np.log1p(eligible_pathways["patient_month_count"]) /
        max(np.log1p(eligible_pathways["patient_month_count"]).quantile(0.99), 1)
    ).clip(0, 1)
)

eligible_pathways = eligible_pathways.sort_values(
    [
        "pathway_severity_score",
        "hospitalization_rate",
        "lift_vs_global",
        "patient_month_count"
    ],
    ascending=[False, False, False, False]
).reset_index(drop=True)

eligible_pathways["d3c_pathway_rank"] = eligible_pathways.index + 1

print("\nEligible high-risk population pathways:")
print(eligible_pathways.head(100))

print("\nEligible pathway count:", eligible_pathways.shape[0])


# ------------------------------------------------------------
# Build lookup tables
# ------------------------------------------------------------

rank_lookup = eligible_pathways.set_index("diagnosis_transition")[
    "d3c_pathway_rank"
].to_dict()

severity_lookup = eligible_pathways.set_index("diagnosis_transition")[
    "pathway_severity_score"
].to_dict()

rate_lookup = eligible_pathways.set_index("diagnosis_transition")[
    "hospitalization_rate"
].to_dict()

lift_lookup = eligible_pathways.set_index("diagnosis_transition")[
    "lift_vs_global"
].to_dict()


# ------------------------------------------------------------
# Helper: add D3C features to member-months
# ------------------------------------------------------------

def add_d3c_pathway_exposure_features(timeline, pathway_member_months, dataset_label):

    df = timeline.copy()
    pm = pathway_member_months.copy()

    df[PREDICTION_DATE_COL] = pd.to_datetime(
        df[PREDICTION_DATE_COL],
        errors="coerce"
    )

    pm[PREDICTION_DATE_COL] = pd.to_datetime(
        pm[PREDICTION_DATE_COL],
        errors="coerce"
    )

    pm["diagnosis_transition"] = (
        pm["diagnosis_transition"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Keep only eligible pathways
    pm = pm[
        pm["diagnosis_transition"].isin(set(eligible_pathways["diagnosis_transition"]))
    ].copy()

    if pm.empty:
        print(f"{dataset_label}: No eligible pathway exposures found.")
        for n in TOP_PATHWAY_CUTOFFS:
            df[f"d3c_top_{n}_pathway_flag"] = 0
        df["d3c_any_high_risk_pathway_flag"] = 0
        df["d3c_high_risk_pathway_count"] = 0
        df["d3c_max_pathway_severity_score"] = 0
        df["d3c_max_pathway_hospitalization_rate"] = 0
        df["d3c_max_pathway_lift"] = 1
        df["d3c_recent_high_risk_pathway_90d_flag"] = 0
        df["d3c_recent_high_risk_pathway_180d_flag"] = 0
        return df

    pm["d3c_pathway_rank"] = pm["diagnosis_transition"].map(rank_lookup)
    pm["d3c_pathway_severity_score"] = pm["diagnosis_transition"].map(severity_lookup)
    pm["d3c_pathway_hospitalization_rate"] = pm["diagnosis_transition"].map(rate_lookup)
    pm["d3c_pathway_lift"] = pm["diagnosis_transition"].map(lift_lookup)

    for col in [
        "d3c_pathway_rank",
        "d3c_pathway_severity_score",
        "d3c_pathway_hospitalization_rate",
        "d3c_pathway_lift"
    ]:
        pm[col] = safe_numeric(pm[col], fill_value=0)

    if "days_since_pathway_completed" not in pm.columns:
        if "dx_date_2" in pm.columns:
            pm["dx_date_2"] = pd.to_datetime(pm["dx_date_2"], errors="coerce")
            pm["days_since_pathway_completed"] = (
                pm[PREDICTION_DATE_COL] - pm["dx_date_2"]
            ).dt.days.clip(lower=0)
        else:
            pm["days_since_pathway_completed"] = 9999

    pm["days_since_pathway_completed"] = safe_numeric(
        pm["days_since_pathway_completed"],
        fill_value=9999
    )

    # Row-level flags
    pm["d3c_any_high_risk_pathway_flag"] = 1

    for n in TOP_PATHWAY_CUTOFFS:
        pm[f"d3c_top_{n}_pathway_flag"] = (
            pm["d3c_pathway_rank"] <= n
        ).astype(int)

    pm["d3c_recent_high_risk_pathway_90d_flag"] = (
        pm["days_since_pathway_completed"] <= 90
    ).astype(int)

    pm["d3c_recent_high_risk_pathway_180d_flag"] = (
        pm["days_since_pathway_completed"] <= 180
    ).astype(int)

    # Aggregate to member-month
    agg_dict = {
        "d3c_any_high_risk_pathway_flag": ("d3c_any_high_risk_pathway_flag", "max"),
        "d3c_high_risk_pathway_count": ("diagnosis_transition", "nunique"),
        "d3c_max_pathway_severity_score": ("d3c_pathway_severity_score", "max"),
        "d3c_max_pathway_hospitalization_rate": ("d3c_pathway_hospitalization_rate", "max"),
        "d3c_max_pathway_lift": ("d3c_pathway_lift", "max"),
        "d3c_recent_high_risk_pathway_90d_flag": ("d3c_recent_high_risk_pathway_90d_flag", "max"),
        "d3c_recent_high_risk_pathway_180d_flag": ("d3c_recent_high_risk_pathway_180d_flag", "max")
    }

    for n in TOP_PATHWAY_CUTOFFS:
        agg_dict[f"d3c_top_{n}_pathway_flag"] = (
            f"d3c_top_{n}_pathway_flag",
            "max"
        )

    pm_agg = (
        pm.groupby(["member_id", PREDICTION_DATE_COL])
        .agg(**agg_dict)
        .reset_index()
    )

    df = df.merge(
        pm_agg,
        on=["member_id", PREDICTION_DATE_COL],
        how="left"
    )

    # Fill defaults
    fill_defaults = {
        "d3c_any_high_risk_pathway_flag": 0,
        "d3c_high_risk_pathway_count": 0,
        "d3c_max_pathway_severity_score": 0,
        "d3c_max_pathway_hospitalization_rate": 0,
        "d3c_max_pathway_lift": 1,
        "d3c_recent_high_risk_pathway_90d_flag": 0,
        "d3c_recent_high_risk_pathway_180d_flag": 0
    }

    for n in TOP_PATHWAY_CUTOFFS:
        fill_defaults[f"d3c_top_{n}_pathway_flag"] = 0

    for col, default in fill_defaults.items():
        if col not in df.columns:
            df[col] = default
        df[col] = safe_numeric(df[col], fill_value=default)

    # Composite exposure score
    df["d3c_pathway_exposure_score"] = (
        0.35 * df["d3c_any_high_risk_pathway_flag"] +
        0.20 * df["d3c_recent_high_risk_pathway_90d_flag"] +
        0.15 * (
            df["d3c_high_risk_pathway_count"] /
            max(df["d3c_high_risk_pathway_count"].quantile(0.99), 1)
        ).clip(0, 1) +
        0.15 * (
            df["d3c_max_pathway_severity_score"] /
            max(df["d3c_max_pathway_severity_score"].quantile(0.99), 1)
        ).clip(0, 1) +
        0.15 * df["d3c_top_25_pathway_flag"]
    )

    df["d3c_pathway_exposure_score"] = (
        df["d3c_pathway_exposure_score"]
        .replace([np.inf, -np.inf], 0)
        .fillna(0)
        .clip(0, 1)
    )

    d3c_cols = [
        "d3c_any_high_risk_pathway_flag",
        "d3c_high_risk_pathway_count",
        "d3c_max_pathway_severity_score",
        "d3c_max_pathway_hospitalization_rate",
        "d3c_max_pathway_lift",
        "d3c_recent_high_risk_pathway_90d_flag",
        "d3c_recent_high_risk_pathway_180d_flag",
        "d3c_pathway_exposure_score"
    ] + [
        f"d3c_top_{n}_pathway_flag" for n in TOP_PATHWAY_CUTOFFS
    ]

    print(f"\n{dataset_label} D3C exposure feature summary:")
    print(df[d3c_cols].describe())

    print(f"\n{dataset_label} D3C hospitalization rate by exposure flag:")
    print(
        df.groupby("d3c_any_high_risk_pathway_flag")
        .agg(
            patient_month_count=("member_id", "count"),
            unique_members=("member_id", "nunique"),
            hospitalization_rate=(TARGET_LABEL_NAME, "mean"),
            avg_exposure_score=("d3c_pathway_exposure_score", "mean")
        )
        .reset_index()
    )

    return df


# ------------------------------------------------------------
# Apply to TRAIN and TEST timelines
# ------------------------------------------------------------

df_train_diag = add_d3c_pathway_exposure_features(
    df_train_timeline,
    train_pathway_member_months,
    "TRAIN Sets 1+2"
)

df_test_diag = add_d3c_pathway_exposure_features(
    df_test_timeline,
    test_pathway_member_months,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Validation by exposure score tier
# ------------------------------------------------------------

def build_d3c_validation_summary(df, dataset_label):

    temp = df.copy()

    temp["d3c_exposure_tier"] = pd.cut(
        temp["d3c_pathway_exposure_score"],
        bins=[-0.001, 0.001, 0.25, 0.50, 0.75, 1.001],
        labels=[
            "NO_EXPOSURE",
            "LOW_EXPOSURE",
            "MODERATE_EXPOSURE",
            "HIGH_EXPOSURE",
            "VERY_HIGH_EXPOSURE"
        ]
    )

    summary = (
        temp.groupby("d3c_exposure_tier", observed=True)
        .agg(
            patient_month_count=("member_id", "count"),
            unique_members=("member_id", "nunique"),
            hospitalization_rate=(TARGET_LABEL_NAME, "mean"),
            avg_exposure_score=("d3c_pathway_exposure_score", "mean"),
            avg_high_risk_pathway_count=("d3c_high_risk_pathway_count", "mean"),
            avg_max_pathway_lift=("d3c_max_pathway_lift", "mean"),
            recent_90d_rate=("d3c_recent_high_risk_pathway_90d_flag", "mean")
        )
        .reset_index()
    )

    print(f"\n{dataset_label} D3C exposure tier validation:")
    print(summary)

    return summary


train_d3c_validation_summary = build_d3c_validation_summary(
    df_train_diag,
    "TRAIN Sets 1+2"
)

test_d3c_validation_summary = build_d3c_validation_summary(
    df_test_diag,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

d3c_feature_cols = [
    "member_id",
    PREDICTION_DATE_COL,
    TARGET_LABEL_NAME,
    "d3c_any_high_risk_pathway_flag",
    "d3c_high_risk_pathway_count",
    "d3c_max_pathway_severity_score",
    "d3c_max_pathway_hospitalization_rate",
    "d3c_max_pathway_lift",
    "d3c_recent_high_risk_pathway_90d_flag",
    "d3c_recent_high_risk_pathway_180d_flag",
    "d3c_pathway_exposure_score"
] + [
    f"d3c_top_{n}_pathway_flag" for n in TOP_PATHWAY_CUTOFFS
]

eligible_pathways.to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3C_ranked_high_risk_population_pathways.csv"
    ),
    index=False
)

df_train_diag[d3c_feature_cols].to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3C_train_pathway_exposure_features.csv"
    ),
    index=False
)

df_test_diag[d3c_feature_cols].to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3C_test_pathway_exposure_features.csv"
    ),
    index=False
)

train_d3c_validation_summary.to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3C_train_exposure_tier_validation.csv"
    ),
    index=False
)

test_d3c_validation_summary.to_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3C_test_exposure_tier_validation.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

if df_train_diag["d3c_pathway_exposure_score"].max() == 0:
    raise ValueError("D3C failed: TRAIN d3c_pathway_exposure_score is all zero.")

if df_test_diag["d3c_pathway_exposure_score"].max() == 0:
    raise ValueError("D3C failed: TEST d3c_pathway_exposure_score is all zero.")

print("\nStep D3C completed successfully.")
print("High-risk diagnosis pathway exposure features are ready for D4/D5/model integration.")


STEP D3C - Building high-risk diagnosis pathway exposure flags

Eligible high-risk population pathways:
   pathway_type                               diagnosis_transition  \
0        3_STEP                             STROKE_TIA → CKD → AKI   
1        3_STEP                             CKD → AKI → CELLULITIS   
2        3_STEP                              PNEUMONIA → CKD → AKI   
3        3_STEP                                SEPSIS → CKD → COPD   
4        3_STEP                                 CKD → SEPSIS → CKD   
5        3_STEP                                 CKD → SEPSIS → AKI   
6        3_STEP                                 SEPSIS → CKD → CHF   
7        3_STEP                          SEPSIS → CKD → CELLULITIS   
8        3_STEP                              CKD → PNEUMONIA → CKD   
9        3_STEP                                   CKD → COPD → AKI   
10       3_STEP                             CKD → STROKE_TIA → AKI   
11       3_STEP                            DIABETES → S

In [ ]:
# ============================================================
# Step D4 - Memory-Safe Diagnosis Progression Velocity Features
# CHUNKED VERSION
# Uses saved D1/D3C files + H2 timelines
# ============================================================

print("\nSTEP D4 - Memory-safe diagnosis progression velocity features")

import os
import gc
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Folders / files
# ------------------------------------------------------------

DIAGNOSIS_OUTPUT_FOLDER = os.path.join(
    BASE_FOLDER,
    "Diagnosis_Progression_Mining"
)

TRAIN_H2_FILE = os.path.join(
    OUTPUT_FOLDER,
    "stepH2_train_sets1_2_timeline_with_labels.csv"
)

TEST_H2_FILE = os.path.join(
    OUTPUT_FOLDER,
    "stepH2_test_set3_timeline_with_labels.csv"
)

TRAIN_DX_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD1_train_diagnosis_timeline.csv"
)

TEST_DX_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD1_test_diagnosis_timeline.csv"
)

D3C_TRAIN_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD3C_train_high_risk_diagnosis_pathway_features.csv"
)

D3C_TEST_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD3C_test_high_risk_diagnosis_pathway_features.csv"
)

OUT_TRAIN_D4 = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD4_train_diagnosis_velocity_features.csv"
)

OUT_TEST_D4 = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD4_test_diagnosis_velocity_features.csv"
)

CHUNK_SIZE_D4 = 300000

# ------------------------------------------------------------
# Validate files
# ------------------------------------------------------------

required_files = [
    TRAIN_H2_FILE,
    TEST_H2_FILE,
    TRAIN_DX_FILE,
    TEST_DX_FILE
]

for f in required_files:
    if not os.path.exists(f):
        raise FileNotFoundError(f)

print("Required files found.")

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def safe_numeric_local(series, fill_value=0):
    return (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(fill_value)
    )


def load_dx_events(dx_file, label):
    print(f"\nLoading diagnosis events for {label}...")

    dx = pd.read_csv(
        dx_file,
        usecols=lambda c: c in [
            "member_id",
            "diagnosis_date",
            "diagnosis_group"
        ],
        low_memory=False
    )

    dx["member_id"] = dx["member_id"].astype(str).str.strip()

    dx["diagnosis_date"] = pd.to_datetime(
        dx["diagnosis_date"],
        errors="coerce"
    )

    dx["diagnosis_group"] = (
        dx["diagnosis_group"]
        .fillna("OTHER")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    dx = dx.dropna(
        subset=["member_id", "diagnosis_date", "diagnosis_group"]
    )

    dx = dx[dx["member_id"] != ""].copy()

    print(label, dx.shape)

    return dx


def build_member_month_velocity(dx, timeline_file, output_file, label):

    print(f"\nBuilding D4 velocity features for {label}")

    dx = dx.sort_values(["member_id", "diagnosis_date"])

    timeline_cols = [
        "member_id",
        PREDICTION_DATE_COL
    ]

    # Add target if present for validation only
    possible_extra_cols = [
        TARGET_LABEL_NAME,
        "hospitalized_next_90d"
    ]

    header_written = False
    total_rows = 0

    for chunk_num, tl in enumerate(
        pd.read_csv(
            timeline_file,
            usecols=lambda c: c in timeline_cols + possible_extra_cols,
            chunksize=CHUNK_SIZE_D4,
            low_memory=False
        ),
        start=1
    ):

        print(f"Processing {label} chunk {chunk_num:,}: {tl.shape}")

        tl["member_id"] = tl["member_id"].astype(str).str.strip()

        tl[PREDICTION_DATE_COL] = pd.to_datetime(
            tl[PREDICTION_DATE_COL],
            errors="coerce"
        )

        tl = tl.dropna(subset=["member_id", PREDICTION_DATE_COL])

        member_subset = set(tl["member_id"].unique())

        dx_sub = dx[dx["member_id"].isin(member_subset)].copy()

        out = tl[["member_id", PREDICTION_DATE_COL]].copy()

        for window in [30, 90, 180, 365]:

            temp = tl[["member_id", PREDICTION_DATE_COL]].copy()
            temp["window_start"] = (
                temp[PREDICTION_DATE_COL]
                - pd.to_timedelta(window, unit="D")
            )

            merged = temp.merge(
                dx_sub,
                on="member_id",
                how="left"
            )

            merged = merged[
                (merged["diagnosis_date"] >= merged["window_start"]) &
                (merged["diagnosis_date"] <= merged[PREDICTION_DATE_COL])
            ]

            agg = (
                merged
                .groupby(["member_id", PREDICTION_DATE_COL])
                .agg(
                    **{
                        f"new_diagnosis_groups_{window}d": (
                            "diagnosis_group",
                            "nunique"
                        ),
                        f"diagnosis_events_{window}d": (
                            "diagnosis_group",
                            "count"
                        )
                    }
                )
                .reset_index()
            )

            out = out.merge(
                agg,
                on=["member_id", PREDICTION_DATE_COL],
                how="left"
            )

            out[f"new_diagnosis_groups_{window}d"] = safe_numeric_local(
                out[f"new_diagnosis_groups_{window}d"],
                0
            )

            out[f"diagnosis_events_{window}d"] = safe_numeric_local(
                out[f"diagnosis_events_{window}d"],
                0
            )

            del temp, merged, agg
            gc.collect()

        out["diagnosis_burst_30d_flag"] = (
            out["new_diagnosis_groups_30d"] >= 2
        ).astype(int)

        out["diagnosis_burst_90d_flag"] = (
            out["new_diagnosis_groups_90d"] >= 3
        ).astype(int)

        out["diagnosis_complexity_365d_flag"] = (
            out["new_diagnosis_groups_365d"] >= 5
        ).astype(int)

        out["diagnosis_velocity_score"] = (
            0.35 * out["new_diagnosis_groups_30d"].clip(0, 5) +
            0.25 * out["new_diagnosis_groups_90d"].clip(0, 8) +
            0.15 * out["new_diagnosis_groups_180d"].clip(0, 10) +
            0.10 * out["new_diagnosis_groups_365d"].clip(0, 15) +
            0.10 * out["diagnosis_burst_30d_flag"] +
            0.05 * out["diagnosis_burst_90d_flag"]
        )

        out["diagnosis_velocity_score"] = (
            out["diagnosis_velocity_score"]
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
            .clip(0, 5)
        )

        out.to_csv(
            output_file,
            index=False,
            mode="w" if not header_written else "a",
            header=not header_written
        )

        header_written = True
        total_rows += len(out)

        del tl, dx_sub, out
        gc.collect()

    print(f"{label} D4 velocity file saved:", output_file)
    print(f"{label} total rows:", total_rows)


# ------------------------------------------------------------
# Load diagnosis event files once
# ------------------------------------------------------------

dx_train = load_dx_events(TRAIN_DX_FILE, "TRAIN Sets 1+2")
dx_test = load_dx_events(TEST_DX_FILE, "TEST Set 3")

# ------------------------------------------------------------
# Build chunked velocity files
# ------------------------------------------------------------

build_member_month_velocity(
    dx_train,
    TRAIN_H2_FILE,
    OUT_TRAIN_D4,
    "TRAIN Sets 1+2"
)

build_member_month_velocity(
    dx_test,
    TEST_H2_FILE,
    OUT_TEST_D4,
    "TEST Set 3"
)

del dx_train, dx_test
gc.collect()

# ------------------------------------------------------------
# Merge D3C high-risk pathway exposure features, if available
# ------------------------------------------------------------

def merge_d3c_features(d4_file, d3c_file, label):

    if not os.path.exists(d3c_file):
        print(f"\n{label}: D3C file not found. Skipping D3C merge.")
        return d4_file

    print(f"\n{label}: Merging D3C exposure features into D4 output")

    d3c_cols_keep = [
        "member_id",
        PREDICTION_DATE_COL,
        "d3c_any_high_risk_pathway_flag",
        "d3c_high_risk_pathway_count",
        "d3c_max_pathway_severity_score",
        "d3c_max_pathway_hospitalization_rate",
        "d3c_max_pathway_lift",
        "d3c_recent_high_risk_pathway_90d_flag",
        "d3c_recent_high_risk_pathway_180d_flag",
        "d3c_pathway_exposure_score",
        "d3c_top_10_pathway_flag",
        "d3c_top_25_pathway_flag",
        "d3c_top_50_pathway_flag",
        "d3c_top_100_pathway_flag"
    ]

    d3c = pd.read_csv(
        d3c_file,
        usecols=lambda c: c in d3c_cols_keep,
        low_memory=False
    )

    d3c["member_id"] = d3c["member_id"].astype(str).str.strip()

    d3c[PREDICTION_DATE_COL] = pd.to_datetime(
        d3c[PREDICTION_DATE_COL],
        errors="coerce"
    )

    merged_file = d4_file.replace(".csv", "_with_D3C.csv")

    header_written = False
    total_rows = 0

    for chunk_num, d4 in enumerate(
        pd.read_csv(d4_file, chunksize=CHUNK_SIZE_D4, low_memory=False),
        start=1
    ):

        print(f"{label}: D3C merge chunk {chunk_num:,}: {d4.shape}")

        d4["member_id"] = d4["member_id"].astype(str).str.strip()

        d4[PREDICTION_DATE_COL] = pd.to_datetime(
            d4[PREDICTION_DATE_COL],
            errors="coerce"
        )

        out = d4.merge(
            d3c,
            on=["member_id", PREDICTION_DATE_COL],
            how="left"
        )

        d3c_feature_cols = [
            c for c in d3c.columns
            if c not in ["member_id", PREDICTION_DATE_COL]
        ]

        for c in d3c_feature_cols:
            out[c] = safe_numeric_local(out[c], 0)

        out.to_csv(
            merged_file,
            index=False,
            mode="w" if not header_written else "a",
            header=not header_written
        )

        header_written = True
        total_rows += len(out)

        del d4, out
        gc.collect()

    print(f"{label}: merged D4+D3C saved:", merged_file)
    print(f"{label}: merged rows:", total_rows)

    return merged_file


final_train_d4_file = merge_d3c_features(
    OUT_TRAIN_D4,
    D3C_TRAIN_FILE,
    "TRAIN Sets 1+2"
)

final_test_d4_file = merge_d3c_features(
    OUT_TEST_D4,
    D3C_TEST_FILE,
    "TEST Set 3"
)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("\nFinal D4 files:")
print("Train:", final_train_d4_file)
print("Test:", final_test_d4_file)

print("\nPreview train D4:")
print(pd.read_csv(final_train_d4_file, nrows=5))

print("\nPreview test D4:")
print(pd.read_csv(final_test_d4_file, nrows=5))

print("\nStep D4 completed successfully.")


STEP D4 - Memory-safe diagnosis progression velocity features
Required files found.

Loading diagnosis events for TRAIN Sets 1+2...
TRAIN Sets 1+2 (35667966, 3)

Loading diagnosis events for TEST Set 3...
TEST Set 3 (17846788, 3)

Building D4 velocity features for TRAIN Sets 1+2
Processing TRAIN Sets 1+2 chunk 1: (300000, 3)
Processing TRAIN Sets 1+2 chunk 2: (300000, 3)
Processing TRAIN Sets 1+2 chunk 3: (300000, 3)
Processing TRAIN Sets 1+2 chunk 4: (300000, 3)
Processing TRAIN Sets 1+2 chunk 5: (300000, 3)
Processing TRAIN Sets 1+2 chunk 6: (300000, 3)
Processing TRAIN Sets 1+2 chunk 7: (300000, 3)
Processing TRAIN Sets 1+2 chunk 8: (300000, 3)
Processing TRAIN Sets 1+2 chunk 9: (300000, 3)
Processing TRAIN Sets 1+2 chunk 10: (300000, 3)
Processing TRAIN Sets 1+2 chunk 11: (300000, 3)
Processing TRAIN Sets 1+2 chunk 12: (300000, 3)
Processing TRAIN Sets 1+2 chunk 13: (300000, 3)
Processing TRAIN Sets 1+2 chunk 14: (300000, 3)
Processing TRAIN Sets 1+2 chunk 15: (300000, 3)
Processi

In [ ]:
# ============================================================
# Step D4.1 - Merge D4 Diagnosis Velocity + D3C Pathway Exposure
# ============================================================

print("\nSTEP D4.1 - Merging D4 velocity features with D3C pathway exposure")

import os
import gc
import numpy as np
import pandas as pd

D4_TRAIN_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD4_train_diagnosis_velocity_features.csv"
)

D4_TEST_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD4_test_diagnosis_velocity_features.csv"
)

D3C_TRAIN_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD3C_train_pathway_exposure_features.csv"
)

D3C_TEST_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD3C_test_pathway_exposure_features.csv"
)

D4_FINAL_TRAIN_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD4_train_final_diagnosis_features.csv"
)

D4_FINAL_TEST_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD4_test_final_diagnosis_features.csv"
)

CHUNK_SIZE_D41 = 300000

for f in [D4_TRAIN_FILE, D4_TEST_FILE, D3C_TRAIN_FILE, D3C_TEST_FILE]:
    if not os.path.exists(f):
        raise FileNotFoundError(f)

def safe_numeric_d41(series, fill_value=0):
    return (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(fill_value)
    )

def merge_d4_d3c_chunked(d4_file, d3c_file, output_file, label):

    print(f"\nMerging {label}")

    d3c_cols = [
        "member_id",
        "prediction_month",
        "d3c_any_high_risk_pathway_flag",
        "d3c_high_risk_pathway_count",
        "d3c_max_pathway_severity_score",
        "d3c_max_pathway_hospitalization_rate",
        "d3c_max_pathway_lift",
        "d3c_recent_high_risk_pathway_90d_flag",
        "d3c_recent_high_risk_pathway_180d_flag",
        "d3c_pathway_exposure_score",
        "d3c_top_10_pathway_flag",
        "d3c_top_25_pathway_flag",
        "d3c_top_50_pathway_flag",
        "d3c_top_100_pathway_flag"
    ]

    d3c = pd.read_csv(
        d3c_file,
        usecols=d3c_cols,
        low_memory=False
    )

    d3c["member_id"] = d3c["member_id"].astype(str).str.strip()
    d3c["prediction_month"] = pd.to_datetime(
        d3c["prediction_month"],
        errors="coerce"
    )

    d3c = d3c.drop_duplicates(
        subset=["member_id", "prediction_month"]
    )

    header_written = False
    total_rows = 0

    for i, d4 in enumerate(
        pd.read_csv(d4_file, chunksize=CHUNK_SIZE_D41, low_memory=False),
        start=1
    ):

        print(f"{label} chunk {i}: {d4.shape}")

        d4["member_id"] = d4["member_id"].astype(str).str.strip()
        d4["prediction_month"] = pd.to_datetime(
            d4["prediction_month"],
            errors="coerce"
        )

        before_rows = len(d4)

        out = d4.merge(
            d3c,
            on=["member_id", "prediction_month"],
            how="left"
        )

        if len(out) != before_rows:
            raise ValueError(
                f"{label}: row count changed during merge. "
                f"Before={before_rows}, After={len(out)}"
            )

        for col in d3c_cols:
            if col not in ["member_id", "prediction_month"]:
                out[col] = safe_numeric_d41(out[col], 0)

        out.to_csv(
            output_file,
            index=False,
            mode="w" if not header_written else "a",
            header=not header_written
        )

        header_written = True
        total_rows += len(out)

        del d4, out
        gc.collect()

    print(f"\n{label} final file saved:", output_file)
    print(f"{label} total rows:", total_rows)

    del d3c
    gc.collect()

merge_d4_d3c_chunked(
    D4_TRAIN_FILE,
    D3C_TRAIN_FILE,
    D4_FINAL_TRAIN_FILE,
    "TRAIN Sets 1+2"
)

merge_d4_d3c_chunked(
    D4_TEST_FILE,
    D3C_TEST_FILE,
    D4_FINAL_TEST_FILE,
    "TEST Set 3"
)

print("\nPreview final train file:")
print(pd.read_csv(D4_FINAL_TRAIN_FILE, nrows=5))

print("\nPreview final test file:")
print(pd.read_csv(D4_FINAL_TEST_FILE, nrows=5))

print("\nStep D4.1 completed successfully.")


STEP D4.1 - Merging D4 velocity features with D3C pathway exposure

Merging TRAIN Sets 1+2
TRAIN Sets 1+2 chunk 1: (300000, 14)
TRAIN Sets 1+2 chunk 2: (300000, 14)
TRAIN Sets 1+2 chunk 3: (300000, 14)
TRAIN Sets 1+2 chunk 4: (300000, 14)
TRAIN Sets 1+2 chunk 5: (300000, 14)
TRAIN Sets 1+2 chunk 6: (300000, 14)
TRAIN Sets 1+2 chunk 7: (300000, 14)
TRAIN Sets 1+2 chunk 8: (300000, 14)
TRAIN Sets 1+2 chunk 9: (300000, 14)
TRAIN Sets 1+2 chunk 10: (300000, 14)
TRAIN Sets 1+2 chunk 11: (300000, 14)
TRAIN Sets 1+2 chunk 12: (300000, 14)
TRAIN Sets 1+2 chunk 13: (300000, 14)
TRAIN Sets 1+2 chunk 14: (300000, 14)
TRAIN Sets 1+2 chunk 15: (300000, 14)
TRAIN Sets 1+2 chunk 16: (300000, 14)
TRAIN Sets 1+2 chunk 17: (300000, 14)
TRAIN Sets 1+2 chunk 18: (300000, 14)
TRAIN Sets 1+2 chunk 19: (300000, 14)
TRAIN Sets 1+2 chunk 20: (300000, 14)
TRAIN Sets 1+2 chunk 21: (300000, 14)
TRAIN Sets 1+2 chunk 22: (300000, 14)
TRAIN Sets 1+2 chunk 23: (300000, 14)
TRAIN Sets 1+2 chunk 24: (300000, 14)
TRAIN

In [ ]:
# ============================================================
# Step D5 - Diagnosis-Enhanced Pathway Score
# UPDATED: Uses D4.1 final diagnosis feature files
# ============================================================

print("\nSTEP D5 - Building diagnosis-enhanced pathway score")

import os
import gc
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Files
# ------------------------------------------------------------

D4_FINAL_TRAIN_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD4_train_final_diagnosis_features.csv"
)

D4_FINAL_TEST_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD4_test_final_diagnosis_features.csv"
)

D5_TRAIN_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD5_train_diagnosis_enhanced_score.csv"
)

D5_TEST_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD5_test_diagnosis_enhanced_score.csv"
)

D5_SUMMARY_FILE = os.path.join(
    DIAGNOSIS_OUTPUT_FOLDER,
    "stepD5_diagnosis_enhanced_score_summary.csv"
)

for f in [D4_FINAL_TRAIN_FILE, D4_FINAL_TEST_FILE]:
    if not os.path.exists(f):
        raise FileNotFoundError(f)

CHUNK_SIZE_D5 = 300000

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def safe_numeric_d5(series, fill_value=0):
    return (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(fill_value)
    )


def build_diagnosis_enhanced_score(input_file, output_file, label):

    print(f"\nProcessing {label}")

    header_written = False
    total_rows = 0
    summary_parts = []

    for chunk_num, df in enumerate(
        pd.read_csv(input_file, chunksize=CHUNK_SIZE_D5, low_memory=False),
        start=1
    ):

        print(f"{label} chunk {chunk_num}: {df.shape}")

        df["member_id"] = df["member_id"].astype(str).str.strip()
        df["prediction_month"] = pd.to_datetime(
            df["prediction_month"],
            errors="coerce"
        )

        # ----------------------------------------------------
        # Ensure required D4/D3C columns exist
        # ----------------------------------------------------

        required_numeric_defaults = {
            "diagnosis_velocity_score": 0,
            "new_diagnosis_groups_30d": 0,
            "new_diagnosis_groups_90d": 0,
            "new_diagnosis_groups_180d": 0,
            "new_diagnosis_groups_365d": 0,
            "diagnosis_events_30d": 0,
            "diagnosis_events_90d": 0,
            "diagnosis_events_180d": 0,
            "diagnosis_events_365d": 0,
            "diagnosis_burst_30d_flag": 0,
            "diagnosis_burst_90d_flag": 0,
            "diagnosis_complexity_365d_flag": 0,
            "d3c_any_high_risk_pathway_flag": 0,
            "d3c_high_risk_pathway_count": 0,
            "d3c_max_pathway_severity_score": 0,
            "d3c_max_pathway_hospitalization_rate": 0,
            "d3c_max_pathway_lift": 1,
            "d3c_recent_high_risk_pathway_90d_flag": 0,
            "d3c_recent_high_risk_pathway_180d_flag": 0,
            "d3c_pathway_exposure_score": 0,
            "d3c_top_10_pathway_flag": 0,
            "d3c_top_25_pathway_flag": 0,
            "d3c_top_50_pathway_flag": 0,
            "d3c_top_100_pathway_flag": 0
        }

        for col, default in required_numeric_defaults.items():
            if col not in df.columns:
                df[col] = default

            df[col] = safe_numeric_d5(df[col], default)

        # ----------------------------------------------------
        # Normalize component scores
        # ----------------------------------------------------

        df["d5_velocity_component"] = (
            df["diagnosis_velocity_score"] / 5
        ).clip(0, 1)

        df["d5_diagnosis_complexity_component"] = (
            df["new_diagnosis_groups_365d"] / 10
        ).clip(0, 1)

        df["d5_diagnosis_event_component"] = (
            df["diagnosis_events_90d"] / 25
        ).clip(0, 1)

        df["d5_burst_component"] = (
            0.60 * df["diagnosis_burst_30d_flag"] +
            0.40 * df["diagnosis_burst_90d_flag"]
        ).clip(0, 1)

        df["d5_pathway_exposure_component"] = (
            df["d3c_pathway_exposure_score"]
        ).clip(0, 1)

        df["d5_pathway_lift_component"] = (
            (df["d3c_max_pathway_lift"] - 1) / 1.6
        ).clip(0, 1)

        df["d5_pathway_rate_component"] = (
            df["d3c_max_pathway_hospitalization_rate"] / 0.25
        ).clip(0, 1)

        df["d5_recent_pathway_component"] = (
            0.65 * df["d3c_recent_high_risk_pathway_90d_flag"] +
            0.35 * df["d3c_recent_high_risk_pathway_180d_flag"]
        ).clip(0, 1)

        df["d5_top_pathway_component"] = (
            0.50 * df["d3c_top_10_pathway_flag"] +
            0.30 * df["d3c_top_25_pathway_flag"] +
            0.15 * df["d3c_top_50_pathway_flag"] +
            0.05 * df["d3c_top_100_pathway_flag"]
        ).clip(0, 1)

        # ----------------------------------------------------
        # Final diagnosis-enhanced score
        # ----------------------------------------------------

        df["diagnosis_enhanced_risk_score"] = (
            0.20 * df["d5_velocity_component"] +
            0.10 * df["d5_diagnosis_complexity_component"] +
            0.10 * df["d5_diagnosis_event_component"] +
            0.10 * df["d5_burst_component"] +
            0.20 * df["d5_pathway_exposure_component"] +
            0.10 * df["d5_pathway_lift_component"] +
            0.10 * df["d5_pathway_rate_component"] +
            0.05 * df["d5_recent_pathway_component"] +
            0.05 * df["d5_top_pathway_component"]
        )

        df["diagnosis_enhanced_risk_score"] = (
            df["diagnosis_enhanced_risk_score"]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
            .clip(0, 1)
        )

        # ----------------------------------------------------
        # Tiering
        # ----------------------------------------------------

        df["diagnosis_enhanced_tier"] = pd.cut(
            df["diagnosis_enhanced_risk_score"],
            bins=[-0.001, 0.10, 0.25, 0.45, 0.70, 1.001],
            labels=[
                "ROUTINE",
                "WATCHLIST",
                "CARE_MANAGEMENT",
                "HIGH_PRIORITY",
                "URGENT"
            ]
        )

        # ----------------------------------------------------
        # Output columns
        # ----------------------------------------------------

        output_cols = [
            "member_id",
            "prediction_month",

            # Final score
            "diagnosis_enhanced_risk_score",
            "diagnosis_enhanced_tier",

            # D4 velocity
            "new_diagnosis_groups_30d",
            "new_diagnosis_groups_90d",
            "new_diagnosis_groups_180d",
            "new_diagnosis_groups_365d",
            "diagnosis_events_30d",
            "diagnosis_events_90d",
            "diagnosis_events_180d",
            "diagnosis_events_365d",
            "diagnosis_burst_30d_flag",
            "diagnosis_burst_90d_flag",
            "diagnosis_complexity_365d_flag",
            "diagnosis_velocity_score",

            # D3C exposure
            "d3c_any_high_risk_pathway_flag",
            "d3c_high_risk_pathway_count",
            "d3c_max_pathway_severity_score",
            "d3c_max_pathway_hospitalization_rate",
            "d3c_max_pathway_lift",
            "d3c_recent_high_risk_pathway_90d_flag",
            "d3c_recent_high_risk_pathway_180d_flag",
            "d3c_pathway_exposure_score",
            "d3c_top_10_pathway_flag",
            "d3c_top_25_pathway_flag",
            "d3c_top_50_pathway_flag",
            "d3c_top_100_pathway_flag",

            # Components
            "d5_velocity_component",
            "d5_diagnosis_complexity_component",
            "d5_diagnosis_event_component",
            "d5_burst_component",
            "d5_pathway_exposure_component",
            "d5_pathway_lift_component",
            "d5_pathway_rate_component",
            "d5_recent_pathway_component",
            "d5_top_pathway_component"
        ]

        # Optional target if present
        if "hospitalized_next_90d" in df.columns:
            output_cols.insert(2, "hospitalized_next_90d")

        df[output_cols].to_csv(
            output_file,
            index=False,
            mode="w" if not header_written else "a",
            header=not header_written
        )

        # Chunk summary
        temp_summary = {
            "dataset": label,
            "chunk": chunk_num,
            "rows": len(df),
            "avg_diagnosis_enhanced_score": df["diagnosis_enhanced_risk_score"].mean(),
            "max_diagnosis_enhanced_score": df["diagnosis_enhanced_risk_score"].max(),
            "avg_diagnosis_velocity_score": df["diagnosis_velocity_score"].mean(),
            "avg_d3c_exposure_score": df["d3c_pathway_exposure_score"].mean(),
            "d3c_any_high_risk_pathway_rate": df["d3c_any_high_risk_pathway_flag"].mean(),
            "urgent_rate": (df["diagnosis_enhanced_tier"].astype(str) == "URGENT").mean(),
            "high_priority_rate": (df["diagnosis_enhanced_tier"].astype(str) == "HIGH_PRIORITY").mean(),
            "care_management_rate": (df["diagnosis_enhanced_tier"].astype(str) == "CARE_MANAGEMENT").mean(),
            "watchlist_rate": (df["diagnosis_enhanced_tier"].astype(str) == "WATCHLIST").mean(),
            "routine_rate": (df["diagnosis_enhanced_tier"].astype(str) == "ROUTINE").mean()
        }

        if "hospitalized_next_90d" in df.columns:
            temp_summary["hospitalization_rate"] = df["hospitalized_next_90d"].mean()

        summary_parts.append(temp_summary)

        header_written = True
        total_rows += len(df)

        del df
        gc.collect()

    summary_df = pd.DataFrame(summary_parts)

    print(f"\n{label} D5 file saved:", output_file)
    print(f"{label} total rows:", total_rows)

    print(f"\n{label} D5 chunk summary:")
    print(summary_df.describe(include="all"))

    return summary_df


train_d5_summary = build_diagnosis_enhanced_score(
    D4_FINAL_TRAIN_FILE,
    D5_TRAIN_FILE,
    "TRAIN Sets 1+2"
)

test_d5_summary = build_diagnosis_enhanced_score(
    D4_FINAL_TEST_FILE,
    D5_TEST_FILE,
    "TEST Set 3"
)

# ------------------------------------------------------------
# Combined summary
# ------------------------------------------------------------

d5_summary = pd.concat(
    [train_d5_summary, test_d5_summary],
    ignore_index=True
)

d5_summary.to_csv(
    D5_SUMMARY_FILE,
    index=False
)

print("\nD5 combined summary saved:", D5_SUMMARY_FILE)

# ------------------------------------------------------------
# Final validation previews
# ------------------------------------------------------------

print("\nPreview TRAIN D5:")
print(pd.read_csv(D5_TRAIN_FILE, nrows=5))

print("\nPreview TEST D5:")
print(pd.read_csv(D5_TEST_FILE, nrows=5))

print("\nStep D5 completed successfully.")


STEP D5 - Building diagnosis-enhanced pathway score

Processing TRAIN Sets 1+2
TRAIN Sets 1+2 chunk 1: (300000, 26)
TRAIN Sets 1+2 chunk 2: (300000, 26)
TRAIN Sets 1+2 chunk 3: (300000, 26)
TRAIN Sets 1+2 chunk 4: (300000, 26)
TRAIN Sets 1+2 chunk 5: (300000, 26)
TRAIN Sets 1+2 chunk 6: (300000, 26)
TRAIN Sets 1+2 chunk 7: (300000, 26)
TRAIN Sets 1+2 chunk 8: (300000, 26)
TRAIN Sets 1+2 chunk 9: (300000, 26)
TRAIN Sets 1+2 chunk 10: (300000, 26)
TRAIN Sets 1+2 chunk 11: (300000, 26)
TRAIN Sets 1+2 chunk 12: (300000, 26)
TRAIN Sets 1+2 chunk 13: (300000, 26)
TRAIN Sets 1+2 chunk 14: (300000, 26)
TRAIN Sets 1+2 chunk 15: (300000, 26)
TRAIN Sets 1+2 chunk 16: (300000, 26)
TRAIN Sets 1+2 chunk 17: (300000, 26)
TRAIN Sets 1+2 chunk 18: (300000, 26)
TRAIN Sets 1+2 chunk 19: (300000, 26)
TRAIN Sets 1+2 chunk 20: (300000, 26)
TRAIN Sets 1+2 chunk 21: (300000, 26)
TRAIN Sets 1+2 chunk 22: (300000, 26)
TRAIN Sets 1+2 chunk 23: (300000, 26)
TRAIN Sets 1+2 chunk 24: (300000, 26)
TRAIN Sets 1+2 ch

In [ ]:
# ============================================================
# Fix #1 Final Validation - D3B/D4/D5
# ============================================================

print("D3B table shape:", population_diagnosis_pathway_risk_table.shape)
print(population_diagnosis_pathway_risk_table[
    [
        "patient_month_count",
        "unique_members",
        "hospitalization_rate",
        "lift_vs_global",
        "high_risk_population_pathway_flag"
    ]
].describe())

print("\nD4 velocity checks:")
d4_cols = [
    "new_diagnosis_groups_30d",
    "new_diagnosis_groups_90d",
    "new_diagnosis_groups_180d",
    "new_diagnosis_groups_365d",
    "diagnosis_velocity_score",
    "population_diagnosis_pathway_lift",
    "population_pathway_count"
]

print(df_train_diag[d4_cols].describe())
print(df_test_diag[d4_cols].describe())

print("\nD5 score checks:")
d5_cols = [
    "diagnosis_enhanced_risk_score",
    "diagnosis_pathway_lift_for_score",
    "diagnosis_velocity_score",
    "population_pathway_count"
]

print(df_train_diag[d5_cols].describe())
print(df_test_diag[d5_cols].describe())

print("\nD5 tier counts:")
print(df_train_diag["diagnosis_enhanced_tier"].value_counts(dropna=False))
print(df_test_diag["diagnosis_enhanced_tier"].value_counts(dropna=False))

print("\nD5 hospitalization rate by tier - TEST:")
print(
    df_test_diag
    .groupby("diagnosis_enhanced_tier", observed=True)
    .agg(
        patient_month_count=("member_id", "count"),
        unique_members=("member_id", "nunique"),
        hospitalization_rate=(TARGET_LABEL_NAME, "mean"),
        avg_score=("diagnosis_enhanced_risk_score", "mean")
    )
    .reset_index()
)

D3B table shape: (6498, 10)
       patient_month_count  unique_members  hospitalization_rate  \
count         6.498000e+03     6498.000000           6498.000000   
mean          1.777773e+04     1396.856879              0.102226   
std           9.158409e+04     5041.481749              0.025674   
min           1.200000e+01        1.000000              0.000000   
25%           1.085000e+03      101.000000              0.085319   
50%           2.946500e+03      271.000000              0.098994   
75%           8.787750e+03      808.750000              0.114967   
max           4.378570e+06   134817.000000              0.416667   

       lift_vs_global  high_risk_population_pathway_flag  
count     6498.000000                        6498.000000  
mean         1.158942                           0.004771  
std          0.291061                           0.068911  
min          0.000000                           0.000000  
25%          0.967263                           0.000000  
50%  

In [ ]:
# ============================================================
# Lab Proxy / Lab Trajectory Branch
# Step L0 - Configuration
# ============================================================

print("\nSTEP L0 - Initializing lab proxy trajectory analysis")

LAB_OUTPUT_FOLDER = os.path.join(
    BASE_FOLDER,
    "Lab_Proxy_Trajectory"
)

os.makedirs(LAB_OUTPUT_FOLDER, exist_ok=True)

LAB_TARGET = TARGET_LABEL_NAME

# Broad HCPCS/CPT lab ranges
# 80000-89999 = pathology/laboratory CPT range
LAB_CPT_MIN = 80000
LAB_CPT_MAX = 89999

# Common lab panels / lab-like test categories
LAB_CATEGORY_PREFIX_MAP = {
    "CHEMISTRY_PANEL": ["800", "801", "802", "803", "804", "805", "806", "807", "808", "809"],
    "THERAPEUTIC_DRUG_MONITORING": ["801"],
    "URINALYSIS": ["810"],
    "HEMATOLOGY_CBC": ["850", "851", "852", "853", "854", "855", "856", "857", "858", "859"],
    "COAGULATION": ["856", "857"],
    "IMMUNOLOGY": ["860", "861", "862", "863", "864", "865", "866", "867", "868", "869"],
    "MICROBIOLOGY_CULTURE": ["870", "871", "872", "873", "874", "875", "876", "877", "878", "879"],
    "ANATOMIC_PATHOLOGY": ["880", "881", "882", "883"],
    "CYTOPATHOLOGY": ["881"],
    "GENETIC_MOLECULAR": ["812", "813", "814", "815", "816", "817", "818", "819"],
    "OTHER_LAB": ["82", "83", "84", "85", "86", "87", "88", "89"]
}

LAB_LOOKBACK_WINDOWS_DAYS = [30, 90, 180, 365]

LAB_ESCALATION_THRESHOLD = 1.50
LAB_BURST_THRESHOLD_30D = 5
LAB_DIVERSITY_THRESHOLD_90D = 5

print("LAB_OUTPUT_FOLDER:", LAB_OUTPUT_FOLDER)
print("LAB_TARGET:", LAB_TARGET)
print("Step L0 completed.")


STEP L0 - Initializing lab proxy trajectory analysis
LAB_OUTPUT_FOLDER: /content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Lab_Proxy_Trajectory
LAB_TARGET: hospitalized_next_90d
Step L0 completed.


In [ ]:
import pandas as pd

HCPCS_CROSSWALK_FILE = "/content/drive/MyDrive/Colab Notebooks/HCPCS_List.xlsx"

df_hcpcs = pd.read_excel(HCPCS_CROSSWALK_FILE)

df_hcpcs["HCPCS Code"] = (
    df_hcpcs["HCPCS Code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Crosswalk rows:", len(df_hcpcs))
print("\nColumns:")
print(df_hcpcs.columns.tolist())

print("\nCategory distribution:")
print(df_hcpcs["Category"].value_counts().head(20))

print("\nSample codes:")
print(
    df_hcpcs[
        df_hcpcs["HCPCS Code"].isin(
            ["85025", "80053", "83036", "84484", "83880"]
        )
    ]
)

Crosswalk rows: 1964

Columns:
['HCPCS Code', 'Description', 'Category', 'Subcategory', 'Clinical Domain', 'Category_ID', 'Relevance', 'hpcps_prefix']

Category distribution:
Category
IMAGING                 582
LAB                     404
NUCLEAR_MEDICINE         50
THERAPY_REHAB            29
CONTRAST_AGENT           19
REMOTE_MONITORING        11
ONCOLOGY GENOMICS         7
PET_IMAGING               6
DRUG MONITORING           5
MRI_CONTRAST              4
PROSTATE CANCER           3
BREAST CANCER             3
DIABETES_MONITORING       2
BLOOD GROUP GENOMICS      2
DME_SUPPLY                2
LIVER DISEASE             2
LUNG CANCER               2
MERKEL CELL CANCER        2
HOME_HEALTH               2
THYROID CANCER            2
Name: count, dtype: int64

Sample codes:
    HCPCS Code                          Description Category  \
50       83880                                  BNP      LAB   
51       84484                             Troponin      LAB   
416      80053  Compreh

In [ ]:
d5 = pd.read_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD5_test_diagnosis_enhanced_score.csv"
    ),
    low_memory=False
)

print(d5["diagnosis_enhanced_tier"].value_counts(normalize=True))

print(
    d5.groupby("diagnosis_enhanced_tier")[
        "diagnosis_enhanced_risk_score"
    ].agg(["count","mean"])
)

diagnosis_enhanced_tier
ROUTINE            0.351278
CARE_MANAGEMENT    0.282908
WATCHLIST          0.204962
HIGH_PRIORITY      0.155775
URGENT             0.005076
Name: proportion, dtype: float64
                           count      mean
diagnosis_enhanced_tier                   
CARE_MANAGEMENT          1028296  0.352019
HIGH_PRIORITY             566202  0.488868
ROUTINE                  1276806  0.022888
URGENT                     18451  0.847896
WATCHLIST                 744985  0.174197


In [ ]:
d3c = pd.read_csv(
    os.path.join(
        DIAGNOSIS_OUTPUT_FOLDER,
        "stepD3C_test_pathway_exposure_features.csv"
    ),
    usecols=[
        "member_id",
        "prediction_month",
        "hospitalized_next_90d"
    ],
    low_memory=False
)

d3c["member_id"] = d3c["member_id"].astype(str)
d3c["prediction_month"] = pd.to_datetime(d3c["prediction_month"])

d5["member_id"] = d5["member_id"].astype(str)
d5["prediction_month"] = pd.to_datetime(d5["prediction_month"])

d5_eval = d5.merge(
    d3c,
    on=["member_id", "prediction_month"],
    how="left"
)

print(
    d5_eval.groupby("diagnosis_enhanced_tier")[
        "hospitalized_next_90d"
    ].agg(
        count="count",
        hospitalizations="sum",
        hospitalization_rate="mean"
    )
)

                           count  hospitalizations  hospitalization_rate
diagnosis_enhanced_tier                                                 
CARE_MANAGEMENT          1028296             58092              0.056493
HIGH_PRIORITY             566202             64352              0.113656
ROUTINE                  1276806             18201              0.014255
URGENT                     18451              3114              0.168771
WATCHLIST                 744985             24060              0.032296


In [ ]:
# ============================================================
# Step L1 - Build Lab Proxy Timeline
# Carrier A/B + Outpatient HCPCS
# ============================================================

print("\nSTEP L1 - Building lab proxy timeline")


def clean_hcpcs(code):
    if pd.isna(code):
        return ""
    code = str(code).strip().upper()
    if code in ["", "NAN", "NONE"]:
        return ""
    if code.endswith(".0"):
        code = code[:-2]
    return code


def is_lab_hcpcs(code):
    code = clean_hcpcs(code)

    if code == "":
        return False

    # Numeric CPT lab range
    if code.isdigit():
        try:
            value = int(code)
            return LAB_CPT_MIN <= value <= LAB_CPT_MAX
        except Exception:
            return False

    return False


def classify_lab_category(code):
    code = clean_hcpcs(code)

    if code == "":
        return "UNKNOWN"

    if not is_lab_hcpcs(code):
        return "NON_LAB"

    for category, prefixes in LAB_CATEGORY_PREFIX_MAP.items():
        for prefix in prefixes:
            if code.startswith(prefix):
                return category

    return "OTHER_LAB"


def carrier_files_ab(sample):

    possible_files = [
        os.path.join(
            INPUT_FOLDER,
            f"DE1_0_2008_to_2010_Carrier_Claims_Sample_{sample}A.csv"
        ),
        os.path.join(
            INPUT_FOLDER,
            f"DE1_0_2008_to_2010_Carrier_Claims_Sample_{sample}B.csv"
        ),
        os.path.join(
            INPUT_FOLDER,
            f"DEI_0_2008_to_2010_Carrier_Claims_Sample_{sample}A.csv"
        ),
        os.path.join(
            INPUT_FOLDER,
            f"DEI_0_2008_to_2010_Carrier_Claims_Sample_{sample}B.csv"
        )
    ]

    return [p for p in possible_files if os.path.exists(p)]


def outpatient_claim_file(sample):
    return os.path.join(
        INPUT_FOLDER,
        f"DE1_0_2008_to_2010_Outpatient_Claims_Sample_{sample}.csv"
    )


def load_carrier_lab_events(samples, dataset_label):

    parts = []

    for sample in samples:

        files = carrier_files_ab(sample)

        if len(files) == 0:
            print(f"No carrier files found for Sample {sample}.")

        for path in files:

            print(f"\nLoading Carrier lab events {dataset_label} Sample {sample}: {path}")

            df = pd.read_csv(path, dtype=str, low_memory=False)

            member_col = "DESYNPUF_ID"
            claim_col = "CLM_ID"

            # Carrier line-level HCPCS fields
            hcpcs_cols = [
                c for c in df.columns
                if c.startswith("HCPCS_CD_")
            ]

            if len(hcpcs_cols) == 0:
                print("No HCPCS columns found, skipping:", path)
                continue

            # Prefer matching line service dates if present
            line_date_cols = [
                c for c in df.columns
                if c.startswith("LINE_SRVC_DT_")
            ]

            # Fallback date
            fallback_date_col = "CLM_FROM_DT" if "CLM_FROM_DT" in df.columns else None

            if fallback_date_col is None and len(line_date_cols) == 0:
                print("No usable date columns found, skipping:", path)
                continue

            usecols = [member_col, claim_col] + hcpcs_cols + line_date_cols
            if fallback_date_col is not None:
                usecols.append(fallback_date_col)

            temp = df[usecols].copy()

            event_parts = []

            for hcpcs_col in hcpcs_cols:

                suffix = hcpcs_col.replace("HCPCS_CD_", "")
                line_date_col = f"LINE_SRVC_DT_{suffix}"

                if line_date_col in temp.columns:
                    date_series = temp[line_date_col]
                elif fallback_date_col is not None:
                    date_series = temp[fallback_date_col]
                else:
                    continue

                part = pd.DataFrame({
                    "member_id": temp[member_col].apply(clean_id),
                    "claim_id": temp[claim_col].apply(clean_id),
                    "lab_service_date": safe_date(date_series),
                    "hcpcs_code": temp[hcpcs_col].apply(clean_hcpcs),
                    "claim_source": "CARRIER",
                    "source_sample": sample
                })

                event_parts.append(part)

            if len(event_parts) == 0:
                continue

            lab = pd.concat(event_parts, ignore_index=True)

            lab = lab[
                (lab["member_id"] != "")
                &
                (lab["hcpcs_code"] != "")
                &
                (lab["lab_service_date"].notna())
            ].copy()

            lab["is_lab_hcpcs"] = lab["hcpcs_code"].apply(is_lab_hcpcs)

            lab = lab[lab["is_lab_hcpcs"]].copy()

            lab["lab_category"] = lab["hcpcs_code"].apply(classify_lab_category)

            print("Carrier lab rows extracted:", lab.shape[0])

            parts.append(lab)

            del df, temp, lab
            gc.collect()

    if len(parts) == 0:
        return pd.DataFrame(
            columns=[
                "member_id",
                "claim_id",
                "lab_service_date",
                "hcpcs_code",
                "claim_source",
                "source_sample",
                "is_lab_hcpcs",
                "lab_category"
            ]
        )

    out = pd.concat(parts, ignore_index=True)

    print(f"\n{dataset_label} Carrier lab events:", out.shape)
    print(out["lab_category"].value_counts().head(20))

    return out


def load_outpatient_lab_events(samples, dataset_label):

    parts = []

    for sample in samples:

        path = outpatient_claim_file(sample)

        print(f"\nLoading Outpatient lab events {dataset_label} Sample {sample}: {path}")

        if not os.path.exists(path):
            print("Outpatient file not found, skipping:", path)
            continue

        df = pd.read_csv(path, dtype=str, low_memory=False)

        member_col = "DESYNPUF_ID"
        claim_col = "CLM_ID"
        date_col = "CLM_FROM_DT"

        hcpcs_cols = [
            c for c in df.columns
            if c.startswith("HCPCS_CD_")
        ]

        if len(hcpcs_cols) == 0:
            print("No HCPCS columns found in outpatient file, skipping:", path)
            continue

        keep_cols = [member_col, claim_col, date_col] + hcpcs_cols
        keep_cols = [c for c in keep_cols if c in df.columns]

        temp = df[keep_cols].copy()

        long_df = temp.melt(
            id_vars=[member_col, claim_col, date_col],
            value_vars=hcpcs_cols,
            var_name="hcpcs_position",
            value_name="hcpcs_code"
        )

        long_df["member_id"] = long_df[member_col].apply(clean_id)
        long_df["claim_id"] = long_df[claim_col].apply(clean_id)
        long_df["lab_service_date"] = safe_date(long_df[date_col])
        long_df["hcpcs_code"] = long_df["hcpcs_code"].apply(clean_hcpcs)
        long_df["claim_source"] = "OUTPATIENT"
        long_df["source_sample"] = sample

        long_df = long_df[
            (long_df["member_id"] != "")
            &
            (long_df["hcpcs_code"] != "")
            &
            (long_df["lab_service_date"].notna())
        ].copy()

        long_df["is_lab_hcpcs"] = long_df["hcpcs_code"].apply(is_lab_hcpcs)
        long_df = long_df[long_df["is_lab_hcpcs"]].copy()
        long_df["lab_category"] = long_df["hcpcs_code"].apply(classify_lab_category)

        lab = long_df[
            [
                "member_id",
                "claim_id",
                "lab_service_date",
                "hcpcs_code",
                "claim_source",
                "source_sample",
                "is_lab_hcpcs",
                "lab_category"
            ]
        ].copy()

        print("Outpatient lab rows extracted:", lab.shape[0])

        parts.append(lab)

        del df, temp, long_df, lab
        gc.collect()

    if len(parts) == 0:
        return pd.DataFrame(
            columns=[
                "member_id",
                "claim_id",
                "lab_service_date",
                "hcpcs_code",
                "claim_source",
                "source_sample",
                "is_lab_hcpcs",
                "lab_category"
            ]
        )

    out = pd.concat(parts, ignore_index=True)

    print(f"\n{dataset_label} Outpatient lab events:", out.shape)
    print(out["lab_category"].value_counts().head(20))

    return out


carrier_train_labs = load_carrier_lab_events(
    TRAIN_SAMPLES,
    "TRAIN Sets 1+2"
)

carrier_test_labs = load_carrier_lab_events(
    [TEST_SAMPLE],
    "TEST Set 3"
)

outpatient_train_labs = load_outpatient_lab_events(
    TRAIN_SAMPLES,
    "TRAIN Sets 1+2"
)

outpatient_test_labs = load_outpatient_lab_events(
    [TEST_SAMPLE],
    "TEST Set 3"
)

lab_train_events = pd.concat(
    [carrier_train_labs, outpatient_train_labs],
    ignore_index=True
)

lab_test_events = pd.concat(
    [carrier_test_labs, outpatient_test_labs],
    ignore_index=True
)

lab_train_events = lab_train_events.drop_duplicates(
    [
        "member_id",
        "claim_id",
        "lab_service_date",
        "hcpcs_code",
        "claim_source"
    ]
)

lab_test_events = lab_test_events.drop_duplicates(
    [
        "member_id",
        "claim_id",
        "lab_service_date",
        "hcpcs_code",
        "claim_source"
    ]
)

print("\nTRAIN lab events shape:", lab_train_events.shape)
print(lab_train_events["lab_category"].value_counts().head(20))

print("\nTEST lab events shape:", lab_test_events.shape)
print(lab_test_events["lab_category"].value_counts().head(20))

lab_train_events.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL1_train_lab_events.csv"
    ),
    index=False
)

lab_test_events.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL1_test_lab_events.csv"
    ),
    index=False
)

print("\nStep L1 completed.")


STEP L1 - Building lab proxy timeline

Loading Carrier lab events TRAIN Sets 1+2 Sample 1: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Carrier_Claims_Sample_1B.csv
Carrier lab rows extracted: 963952

Loading Carrier lab events TRAIN Sets 1+2 Sample 2: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Carrier_Claims_Sample_2A.csv
Carrier lab rows extracted: 961190

Loading Carrier lab events TRAIN Sets 1+2 Sample 2: /content/drive/MyDrive/Colab Notebooks/Medicare Synthetic Model 2/DE1_0_2008_to_2010_Carrier_Claims_Sample_2B.csv
Carrier lab rows extracted: 965148

TRAIN Sets 1+2 Carrier lab events: (2890290, 8)
lab_category
OTHER_LAB               1162171
CHEMISTRY_PANEL          566795
HEMATOLOGY_CBC           551969
ANATOMIC_PATHOLOGY       192367
URINALYSIS               169039
MICROBIOLOGY_CULTURE     152404
IMMUNOLOGY                95545
Name: count, dtype: int64

Loading Carrier lab events TEST Set 3 Samp

In [ ]:
# Step L1(V) Validation and save checkpoints

print("lab_train_events exists:", "lab_train_events" in globals())
print("lab_test_events exists:", "lab_test_events" in globals())

if "lab_train_events" in globals():
    print("TRAIN lab shape:", lab_train_events.shape)
    print(lab_train_events["lab_category"].value_counts(dropna=False))

if "lab_test_events" in globals():
    print("TEST lab shape:", lab_test_events.shape)
    print(lab_test_events["lab_category"].value_counts(dropna=False))

lab_train_events.to_pickle(
    os.path.join(LAB_OUTPUT_FOLDER, "checkpoint_L1_lab_train_events.pkl")
)

lab_test_events.to_pickle(
    os.path.join(LAB_OUTPUT_FOLDER, "checkpoint_L1_lab_test_events.pkl")
)

print("L1 lab checkpoints saved.")

lab_train_events exists: True
lab_test_events exists: True
TRAIN lab shape: (5038126, 8)
lab_category
OTHER_LAB               2014292
HEMATOLOGY_CBC          1029499
CHEMISTRY_PANEL          980748
MICROBIOLOGY_CULTURE     301582
URINALYSIS               261901
ANATOMIC_PATHOLOGY       249183
IMMUNOLOGY               200921
Name: count, dtype: int64
TEST lab shape: (2961792, 8)
lab_category
OTHER_LAB               1185820
HEMATOLOGY_CBC           600296
CHEMISTRY_PANEL          575721
MICROBIOLOGY_CULTURE     172553
URINALYSIS               157588
ANATOMIC_PATHOLOGY       154258
IMMUNOLOGY               115555
GENETIC_MOLECULAR             1
Name: count, dtype: int64
L1 lab checkpoints saved.


In [ ]:
# ============================================================
# Step L2 - Lab Lookback / Escalation Features
# Adds lab trajectory features to df_train_timeline and df_test_timeline
# ============================================================

print("\nSTEP L2 - Building lab lookback and escalation features")


def build_lab_monthly(lab_events, dataset_label):

    labs = lab_events.copy()

    labs["lab_service_date"] = pd.to_datetime(
        labs["lab_service_date"],
        errors="coerce"
    )

    labs["lab_month_start"] = labs["lab_service_date"].values.astype("datetime64[M]")

    monthly = (
        labs
        .groupby(["member_id", "lab_month_start"])
        .agg(
            lab_claims_month=("claim_id", "nunique"),
            lab_events_month=("hcpcs_code", "count"),
            unique_lab_codes_month=("hcpcs_code", "nunique"),
            unique_lab_categories_month=("lab_category", "nunique")
        )
        .reset_index()
    )

    # Category-specific monthly counts
    category_counts = (
        labs
        .pivot_table(
            index=["member_id", "lab_month_start"],
            columns="lab_category",
            values="hcpcs_code",
            aggfunc="count",
            fill_value=0
        )
        .reset_index()
    )

    category_counts.columns = [
        c if isinstance(c, str) else str(c)
        for c in category_counts.columns
    ]

    category_cols = [
        c for c in category_counts.columns
        if c not in ["member_id", "lab_month_start"]
    ]

    rename_map = {
        c: "labcat_" + c.lower() + "_month"
        for c in category_cols
    }

    category_counts = category_counts.rename(columns=rename_map)

    monthly = monthly.merge(
        category_counts,
        on=["member_id", "lab_month_start"],
        how="left"
    )

    print(f"{dataset_label} lab monthly shape:", monthly.shape)

    return monthly


def add_lab_features_to_timeline(timeline, lab_events, dataset_label):

    df = timeline.copy()

    df[PREDICTION_DATE_COL] = pd.to_datetime(
        df[PREDICTION_DATE_COL],
        errors="coerce"
    )

    lab_monthly = build_lab_monthly(
        lab_events,
        dataset_label
    )

    df = df.merge(
        lab_monthly,
        left_on=["member_id", PREDICTION_DATE_COL],
        right_on=["member_id", "lab_month_start"],
        how="left"
    )

    df = df.drop(columns=["lab_month_start"], errors="ignore")

    monthly_lab_cols = [
        c for c in df.columns
        if c.startswith("lab_")
        or c.startswith("unique_lab_")
        or c.startswith("labcat_")
    ]

    for col in monthly_lab_cols:
        df[col] = safe_numeric(df[col], fill_value=0)

    df = df.sort_values(["member_id", PREDICTION_DATE_COL])

    window_map = {
        30: 1,
        90: 3,
        180: 6,
        365: 12
    }

    base_cols = [
        "lab_claims_month",
        "lab_events_month",
        "unique_lab_codes_month",
        "unique_lab_categories_month"
    ]

    category_month_cols = [
        c for c in df.columns
        if c.startswith("labcat_") and c.endswith("_month")
    ]

    rolling_base_cols = base_cols + category_month_cols

    for days, months_back in window_map.items():

        for base_col in rolling_base_cols:

            if base_col not in df.columns:
                df[base_col] = 0

            output_col = base_col.replace("_month", f"_{days}d")

            df[output_col] = (
                df
                .groupby("member_id")[base_col]
                .transform(
                    lambda x: x.shift(1).rolling(
                        window=months_back,
                        min_periods=1
                    ).sum()
                )
            )

            df[output_col] = safe_numeric(
                df[output_col],
                fill_value=0
            )

    # --------------------------------------------------------
    # Lab escalation / trajectory features
    # --------------------------------------------------------

    df["lab_growth_30_vs_90"] = (
        df["lab_events_30d"] /
        (df["lab_events_90d"] / 3).replace(0, np.nan)
    )

    df["lab_growth_90_vs_365"] = (
        df["lab_events_90d"] /
        (df["lab_events_365d"] / 4).replace(0, np.nan)
    )

    df["lab_diversity_growth_30_vs_90"] = (
        df["unique_lab_codes_30d"] /
        (df["unique_lab_codes_90d"] / 3).replace(0, np.nan)
    )

    df["lab_category_growth_30_vs_90"] = (
        df["unique_lab_categories_30d"] /
        (df["unique_lab_categories_90d"] / 3).replace(0, np.nan)
    )

    df["lab_escalation_score"] = (
        0.35 * df["lab_growth_30_vs_90"] +
        0.25 * df["lab_growth_90_vs_365"] +
        0.25 * df["lab_diversity_growth_30_vs_90"] +
        0.15 * df["lab_category_growth_30_vs_90"]
    )

    df["lab_escalation_score"] = (
        df["lab_escalation_score"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(0, 10)
    )

    df["lab_escalation_flag"] = (
        df["lab_escalation_score"] >= LAB_ESCALATION_THRESHOLD
    ).astype(int)

    df["lab_burst_flag_30d"] = (
        df["lab_events_30d"] >= LAB_BURST_THRESHOLD_30D
    ).astype(int)

    df["lab_diversity_flag_90d"] = (
        df["unique_lab_codes_90d"] >= LAB_DIVERSITY_THRESHOLD_90D
    ).astype(int)

    df["lab_monitoring_intensity_score"] = (
        0.40 * np.log1p(df["lab_events_90d"]) +
        0.30 * np.log1p(df["unique_lab_codes_90d"]) +
        0.30 * np.log1p(df["unique_lab_categories_90d"])
    )

    df["lab_monitoring_intensity_score"] = (
        df["lab_monitoring_intensity_score"]
        .replace([np.inf, -np.inf], 0)
        .fillna(0)
    )

    lab_feature_cols = [
        c for c in df.columns
        if c.startswith("lab_")
        or c.startswith("unique_lab_")
        or c.startswith("labcat_")
    ]

    for col in lab_feature_cols:
        df[col] = (
            safe_numeric(df[col], fill_value=0)
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
        )

    print(f"\n{dataset_label} lab features added.")
    print("Lab feature count:", len(lab_feature_cols))

    key_lab_cols = [
        "lab_events_30d",
        "lab_events_90d",
        "lab_events_365d",
        "unique_lab_codes_90d",
        "unique_lab_categories_90d",
        "lab_growth_30_vs_90",
        "lab_escalation_score",
        "lab_escalation_flag",
        "lab_burst_flag_30d",
        "lab_diversity_flag_90d",
        "lab_monitoring_intensity_score"
    ]

    key_lab_cols = [c for c in key_lab_cols if c in df.columns]

    print(df[key_lab_cols].describe())

    return df, lab_feature_cols


df_train_timeline, lab_feature_cols = add_lab_features_to_timeline(
    df_train_timeline,
    lab_train_events,
    "TRAIN Sets 1+2"
)

df_test_timeline, _ = add_lab_features_to_timeline(
    df_test_timeline,
    lab_test_events,
    "TEST Set 3"
)

pd.DataFrame({
    "lab_feature": lab_feature_cols
}).to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL2_lab_feature_list.csv"
    ),
    index=False
)

df_train_timeline.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL2_train_timeline_with_lab_features.csv"
    ),
    index=False
)

df_test_timeline.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL2_test_timeline_with_lab_features.csv"
    ),
    index=False
)

print("\nStep L2 completed.")


STEP L2 - Building lab lookback and escalation features
TRAIN Sets 1+2 lab monthly shape: (1719555, 13)

TRAIN Sets 1+2 lab features added.
Lab feature count: 64
       lab_events_30d  lab_events_90d  lab_events_365d  unique_lab_codes_90d  \
count    7.265664e+06    7.265664e+06     7.265664e+06          7.265664e+06   
mean     6.728799e-01    1.996311e+00     7.298229e+00          1.951039e+00   
std      1.796272e+00    3.617146e+00     1.010936e+01          3.480035e+00   
min      0.000000e+00    0.000000e+00     0.000000e+00          0.000000e+00   
25%      0.000000e+00    0.000000e+00     0.000000e+00          0.000000e+00   
50%      0.000000e+00    0.000000e+00     3.000000e+00          0.000000e+00   
75%      0.000000e+00    3.000000e+00     1.100000e+01          3.000000e+00   
max      5.200000e+01    7.600000e+01     1.930000e+02          6.600000e+01   

       unique_lab_categories_90d  lab_growth_30_vs_90  lab_escalation_score  \
count               7.265664e+06     

In [ ]:
# Step L2(V) Validation

l2_cols = [
    "lab_events_30d",
    "lab_events_90d",
    "lab_events_365d",
    "unique_lab_codes_90d",
    "unique_lab_categories_90d",
    "lab_escalation_score",
    "lab_monitoring_intensity_score"
]

print(df_train_timeline[l2_cols].describe())
print(df_test_timeline[l2_cols].describe())

       lab_events_30d  lab_events_90d  lab_events_365d  unique_lab_codes_90d  \
count    7.265664e+06    7.265664e+06     7.265664e+06          7.265664e+06   
mean     6.728799e-01    1.996311e+00     7.298229e+00          1.951039e+00   
std      1.796272e+00    3.617146e+00     1.010936e+01          3.480035e+00   
min      0.000000e+00    0.000000e+00     0.000000e+00          0.000000e+00   
25%      0.000000e+00    0.000000e+00     0.000000e+00          0.000000e+00   
50%      0.000000e+00    0.000000e+00     3.000000e+00          0.000000e+00   
75%      0.000000e+00    3.000000e+00     1.100000e+01          3.000000e+00   
max      5.200000e+01    7.600000e+01     1.930000e+02          6.600000e+01   

       unique_lab_categories_90d  lab_escalation_score  \
count               7.265664e+06          7.265664e+06   
mean                1.311301e+00          5.334179e-01   
std                 2.006263e+00          8.841180e-01   
min                 0.000000e+00          0.000

In [ ]:
# ============================================================
# Step L3 - Lab Workup Pattern Flags
# Creates clinical workup proxy patterns from lab categories
# ============================================================

print("\nSTEP L3 - Building lab workup pattern flags")


def add_lab_workup_patterns(df, dataset_label):

    df = df.copy()

    lab_cols = [c for c in df.columns if c.startswith("labcat_")]

    for col in lab_cols:
        df[col] = safe_numeric(df[col], fill_value=0)

    # Helper to safely get columns
    def get_col(col):
        if col not in df.columns:
            df[col] = 0
        return safe_numeric(df[col], fill_value=0)

    chemistry_90 = get_col("labcat_chemistry_panel_90d")
    cbc_90 = get_col("labcat_hematology_cbc_90d")
    coag_90 = get_col("labcat_coagulation_90d")
    micro_90 = get_col("labcat_microbiology_culture_90d")
    urine_90 = get_col("labcat_urinalysis_90d")
    immunology_90 = get_col("labcat_immunology_90d")
    drug_monitor_90 = get_col("labcat_therapeutic_drug_monitoring_90d")

    chemistry_30 = get_col("labcat_chemistry_panel_30d")
    cbc_30 = get_col("labcat_hematology_cbc_30d")
    micro_30 = get_col("labcat_microbiology_culture_30d")
    urine_30 = get_col("labcat_urinalysis_30d")

    # Infection / sepsis proxy
    df["lab_infection_workup_90d_flag"] = (
        (cbc_90 > 0) &
        ((micro_90 > 0) | (urine_90 > 0))
    ).astype(int)

    df["lab_infection_workup_30d_flag"] = (
        (cbc_30 > 0) &
        ((micro_30 > 0) | (urine_30 > 0))
    ).astype(int)

    # Renal / metabolic monitoring proxy
    df["lab_renal_metabolic_monitoring_90d_flag"] = (
        (chemistry_90 > 0) &
        (urine_90 > 0)
    ).astype(int)

    # Hematology / bleeding / anticoagulation proxy
    df["lab_bleeding_coag_workup_90d_flag"] = (
        (cbc_90 > 0) &
        (coag_90 > 0)
    ).astype(int)

    # Broad complex workup proxy
    df["lab_complex_workup_90d_flag"] = (
        (
            (cbc_90 > 0).astype(int) +
            (chemistry_90 > 0).astype(int) +
            (micro_90 > 0).astype(int) +
            (urine_90 > 0).astype(int) +
            (coag_90 > 0).astype(int) +
            (immunology_90 > 0).astype(int) +
            (drug_monitor_90 > 0).astype(int)
        ) >= 3
    ).astype(int)

    # Recent workup burst
    df["lab_recent_workup_burst_flag"] = (
        (df.get("lab_events_30d", 0) >= LAB_BURST_THRESHOLD_30D) |
        (df["lab_infection_workup_30d_flag"] == 1)
    ).astype(int)

    df["lab_workup_pattern_score"] = (
        0.25 * df["lab_infection_workup_90d_flag"] +
        0.20 * df["lab_renal_metabolic_monitoring_90d_flag"] +
        0.15 * df["lab_bleeding_coag_workup_90d_flag"] +
        0.25 * df["lab_complex_workup_90d_flag"] +
        0.15 * df["lab_recent_workup_burst_flag"]
    )

    pattern_cols = [
        "lab_infection_workup_90d_flag",
        "lab_infection_workup_30d_flag",
        "lab_renal_metabolic_monitoring_90d_flag",
        "lab_bleeding_coag_workup_90d_flag",
        "lab_complex_workup_90d_flag",
        "lab_recent_workup_burst_flag",
        "lab_workup_pattern_score"
    ]

    print(f"\n{dataset_label} lab workup pattern summary:")
    print(df[pattern_cols].describe())

    return df, pattern_cols


df_train_timeline, lab_pattern_cols = add_lab_workup_patterns(
    df_train_timeline,
    "TRAIN Sets 1+2"
)

df_test_timeline, _ = add_lab_workup_patterns(
    df_test_timeline,
    "TEST Set 3"
)

pd.DataFrame({"lab_pattern_feature": lab_pattern_cols}).to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL3_lab_workup_pattern_features.csv"
    ),
    index=False
)

print("\nStep L3 completed.")


STEP L3 - Building lab workup pattern flags

TRAIN Sets 1+2 lab workup pattern summary:
       lab_infection_workup_90d_flag  lab_infection_workup_30d_flag  \
count                   7.265664e+06                   7.265664e+06   
mean                    9.051795e-02                   2.442007e-02   
std                     2.869224e-01                   1.543494e-01   
min                     0.000000e+00                   0.000000e+00   
25%                     0.000000e+00                   0.000000e+00   
50%                     0.000000e+00                   0.000000e+00   
75%                     0.000000e+00                   0.000000e+00   
max                     1.000000e+00                   1.000000e+00   

       lab_renal_metabolic_monitoring_90d_flag  \
count                             7.265664e+06   
mean                              5.332823e-02   
std                               2.246872e-01   
min                               0.000000e+00   
25%                  

In [ ]:
# Step L3(V) Validation

l3_cols = [
    "lab_infection_workup_90d_flag",
    "lab_infection_workup_30d_flag",
    "lab_renal_metabolic_monitoring_90d_flag",
    "lab_complex_workup_90d_flag",
    "lab_recent_workup_burst_flag",
    "lab_workup_pattern_score"
]

print(df_train_timeline[l3_cols].describe())
print(df_test_timeline[l3_cols].describe())

       lab_infection_workup_90d_flag  lab_infection_workup_30d_flag  \
count                   7.265664e+06                   7.265664e+06   
mean                    9.051795e-02                   2.442007e-02   
std                     2.869224e-01                   1.543494e-01   
min                     0.000000e+00                   0.000000e+00   
25%                     0.000000e+00                   0.000000e+00   
50%                     0.000000e+00                   0.000000e+00   
75%                     0.000000e+00                   0.000000e+00   
max                     1.000000e+00                   1.000000e+00   

       lab_renal_metabolic_monitoring_90d_flag  lab_complex_workup_90d_flag  \
count                             7.265664e+06                 7.265664e+06   
mean                              5.332823e-02                 9.340523e-02   
std                               2.246872e-01                 2.909995e-01   
min                               0.000000e+

In [ ]:
print(
    df_hcpcs[
        df_hcpcs["HCPCS Code"].isin(
            [
                "85610",
                "85730",
                "85732",
                "85736",
                "85379"
            ]
        )
    ][
        [
            "HCPCS Code",
            "Category",
            "Subcategory"
        ]
    ]
)

    HCPCS Code Category           Subcategory
46       85379      LAB  BLEEDING_COAGULATION
47       85610      LAB  BLEEDING_COAGULATION
48       85730      LAB  BLEEDING_COAGULATION
429      85610      LAB        HEMATOLOGY_CBC
430      85730      LAB        HEMATOLOGY_CBC
453      85379      LAB  BLEEDING_COAGULATION


In [ ]:
# ============================================================
# Step L3.5 Merge D5 Diagnosis Outputs into Lab Timeline
# ============================================================

import pandas as pd
import numpy as np
import os

TRAIN_D5_FILE = "/content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Output/Diagnosis/stepD5_train_diagnosis_scored.csv"
TEST_D5_FILE  = "/content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Output/Diagnosis/stepD5_test_diagnosis_scored.csv"

train_d5 = pd.read_csv(TRAIN_D5_FILE)
test_d5 = pd.read_csv(TEST_D5_FILE)

# Make sure dates match
train_d5[PREDICTION_DATE_COL] = pd.to_datetime(train_d5[PREDICTION_DATE_COL], errors="coerce")
test_d5[PREDICTION_DATE_COL] = pd.to_datetime(test_d5[PREDICTION_DATE_COL], errors="coerce")

df_train_timeline[PREDICTION_DATE_COL] = pd.to_datetime(df_train_timeline[PREDICTION_DATE_COL], errors="coerce")
df_test_timeline[PREDICTION_DATE_COL] = pd.to_datetime(df_test_timeline[PREDICTION_DATE_COL], errors="coerce")

d5_cols_to_merge = [
    "member_id",
    PREDICTION_DATE_COL,
    "diagnosis_enhanced_risk_score",
    "population_diagnosis_pathway_lift",
    "population_pathway_count",
    "population_diagnosis_pathway_rate",
    "population_high_risk_pathway_count",
    "diagnosis_enhanced_tier"
]

train_d5_merge = train_d5[[c for c in d5_cols_to_merge if c in train_d5.columns]].copy()
test_d5_merge = test_d5[[c for c in d5_cols_to_merge if c in test_d5.columns]].copy()

# Drop old placeholder columns from lab timelines
drop_cols = [
    c for c in d5_cols_to_merge
    if c not in ["member_id", PREDICTION_DATE_COL]
    and c in df_train_timeline.columns
]

df_train_timeline = df_train_timeline.drop(columns=drop_cols, errors="ignore")
df_test_timeline = df_test_timeline.drop(columns=drop_cols, errors="ignore")

# Merge real D5 features
df_train_timeline = df_train_timeline.merge(
    train_d5_merge,
    on=["member_id", PREDICTION_DATE_COL],
    how="left"
)

df_test_timeline = df_test_timeline.merge(
    test_d5_merge,
    on=["member_id", PREDICTION_DATE_COL],
    how="left"
)

# Fill safe defaults
defaults = {
    "diagnosis_enhanced_risk_score": 0,
    "population_diagnosis_pathway_lift": 1,
    "population_pathway_count": 0,
    "population_diagnosis_pathway_rate": 0,
    "population_high_risk_pathway_count": 0
}

for df in [df_train_timeline, df_test_timeline]:
    for col, default in defaults.items():
        if col not in df.columns:
            df[col] = default
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default)

print("D5 diagnosis outputs merged into lab timelines.")

print("\nTRAIN merged D5 check:")
print(
    df_train_timeline[
        [
            "diagnosis_enhanced_risk_score",
            "population_diagnosis_pathway_lift",
            "population_pathway_count",
            "population_diagnosis_pathway_rate",
            "population_high_risk_pathway_count"
        ]
    ].describe()
)

print("\nTEST merged D5 check:")
print(
    df_test_timeline[
        [
            "diagnosis_enhanced_risk_score",
            "population_diagnosis_pathway_lift",
            "population_pathway_count",
            "population_diagnosis_pathway_rate",
            "population_high_risk_pathway_count"
        ]
    ].describe()
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Hospitalization_Forecasting_Model/Output/Diagnosis/stepD5_train_diagnosis_scored.csv'

In [ ]:
# ============================================================
# Step L4 - Diagnosis + Lab Interaction Features
# UPDATED FULL VERSION
# Combines lab signals with chronic/diagnosis/pathway risk signals
# ============================================================

print("\nSTEP L4 - Building diagnosis + lab interaction features")


def add_diagnosis_lab_interactions(df, dataset_label):

    df = df.copy()

    needed_defaults = {
        "lab_escalation_score": 0,
        "lab_monitoring_intensity_score": 0,
        "lab_workup_pattern_score": 0,
        "lab_infection_workup_90d_flag": 0,
        "lab_renal_metabolic_monitoring_90d_flag": 0,
        "lab_complex_workup_90d_flag": 0,
        "lab_recent_workup_burst_flag": 0,
        "chronic_count": 0,
        "high_risk_patient": 0,
        "prior_ip_claims_90d": 0,
        "prior_ip_claims_365d": 0,
        "rx_escalation_score": 0,
        "diagnosis_enhanced_risk_score": 0,
        "population_diagnosis_pathway_lift": 1,
        "diagnosis_pathway_lift_vs_global": 1,
        "population_pathway_count": 0
    }

    for col, default in needed_defaults.items():
        if col not in df.columns:
            df[col] = default

        df[col] = safe_numeric(df[col], fill_value=default)

    # --------------------------------------------------------
    # Interaction features
    # --------------------------------------------------------

    df["lab_chronic_burden_interaction"] = (
        df["lab_monitoring_intensity_score"] *
        np.log1p(df["chronic_count"])
    )

    df["lab_rx_escalation_interaction"] = (
        df["lab_escalation_score"] *
        df["rx_escalation_score"]
    )

    df["lab_prior_ip_interaction"] = (
        df["lab_workup_pattern_score"] *
        np.log1p(df["prior_ip_claims_365d"])
    )

    df["lab_diagnosis_pathway_interaction"] = (
        df["lab_workup_pattern_score"] *
        np.maximum(
            df["population_diagnosis_pathway_lift"],
            df["diagnosis_pathway_lift_vs_global"]
        )
    )

    df["lab_complex_deterioration_flag"] = (
        (
            (df["lab_complex_workup_90d_flag"] == 1) |
            (df["lab_recent_workup_burst_flag"] == 1)
        )
        &
        (
            (df["chronic_count"] >= 5) |
            (df["population_pathway_count"] >= 10) |
            (df["prior_ip_claims_365d"] >= 1)
        )
    ).astype(int)

    df["lab_infection_plus_pathway_flag"] = (
        (df["lab_infection_workup_90d_flag"] == 1)
        &
        (
            (df["population_diagnosis_pathway_lift"] >= 1.5)
            |
            (df["diagnosis_pathway_lift_vs_global"] >= 1.5)
        )
    ).astype(int)

    df["lab_renal_plus_chronic_flag"] = (
        (df["lab_renal_metabolic_monitoring_90d_flag"] == 1)
        &
        (df["chronic_count"] >= 5)
    ).astype(int)

    # --------------------------------------------------------
    # Composite score
    # --------------------------------------------------------

    df["pathway_lift_for_lab"] = np.maximum(
        df["population_diagnosis_pathway_lift"],
        df["diagnosis_pathway_lift_vs_global"]
    )

    df["lab_monitoring_norm"] = (
        df["lab_monitoring_intensity_score"] /
        max(df["lab_monitoring_intensity_score"].quantile(0.99), 1)
    ).clip(0, 1)

    df["lab_escalation_norm"] = (
        df["lab_escalation_score"] /
        max(df["lab_escalation_score"].quantile(0.99), 1)
    ).clip(0, 1)

    df["pathway_lift_for_lab_norm"] = (
        df["pathway_lift_for_lab"] /
        max(df["pathway_lift_for_lab"].quantile(0.99), 1)
    ).clip(0, 1)

    df["lab_diagnosis_composite_score"] = (
        0.25 * df["lab_monitoring_norm"] +
        0.20 * df["lab_escalation_norm"] +
        0.20 * df["lab_workup_pattern_score"].clip(0, 1) +
        0.15 * df["pathway_lift_for_lab_norm"] +
        0.10 * df["lab_complex_deterioration_flag"] +
        0.10 * df["lab_infection_plus_pathway_flag"]
    )

    df["lab_diagnosis_composite_score"] = (
        df["lab_diagnosis_composite_score"]
        .replace([np.inf, -np.inf], 0)
        .fillna(0)
        .clip(0, 1)
    )

    interaction_cols = [
        "lab_chronic_burden_interaction",
        "lab_rx_escalation_interaction",
        "lab_prior_ip_interaction",
        "lab_diagnosis_pathway_interaction",
        "lab_complex_deterioration_flag",
        "lab_infection_plus_pathway_flag",
        "lab_renal_plus_chronic_flag",
        "pathway_lift_for_lab",
        "lab_diagnosis_composite_score"
    ]

    print(f"\n{dataset_label} diagnosis + lab interaction summary:")
    print(df[interaction_cols].describe())

    return df, interaction_cols


df_train_timeline, lab_interaction_cols = add_diagnosis_lab_interactions(
    df_train_timeline,
    "TRAIN Sets 1+2"
)

df_test_timeline, _ = add_diagnosis_lab_interactions(
    df_test_timeline,
    "TEST Set 3"
)

pd.DataFrame(
    {"lab_interaction_feature": lab_interaction_cols}
).to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL4_lab_diagnosis_interaction_features.csv"
    ),
    index=False
)

print("\nStep L4 completed.")

In [ ]:
# Step L4(V) Validation

l4_cols = [
    "lab_chronic_burden_interaction",
    "lab_rx_escalation_interaction",
    "lab_prior_ip_interaction",
    "lab_diagnosis_pathway_interaction",
    "lab_complex_deterioration_flag",
    "lab_infection_plus_pathway_flag",
    "lab_renal_plus_chronic_flag",
    "lab_diagnosis_composite_score"
]

print(df_train_timeline[l4_cols].describe())
print(df_test_timeline[l4_cols].describe())

In [ ]:
# ============================================================
# Step L5 - Lab Signal Validation Summary
# Validates whether lab features stratify hospitalization risk
# ============================================================

print("\nSTEP L5 - Validating lab signal stratification")


def build_lab_validation_summary(df, dataset_label):

    df = df.copy()

    required_cols = {
        "lab_escalation_score": 0,
        "lab_monitoring_intensity_score": 0,
        "lab_workup_pattern_score": 0,
        "lab_diagnosis_composite_score": 0,
        "lab_complex_deterioration_flag": 0,
        "lab_infection_plus_pathway_flag": 0,
        "lab_renal_plus_chronic_flag": 0
    }

    for col, default in required_cols.items():
        if col not in df.columns:
            df[col] = default
        df[col] = safe_numeric(df[col], fill_value=default)

    df["lab_signal_tier"] = pd.qcut(
        df["lab_diagnosis_composite_score"].rank(method="first"),
        q=5,
        labels=[
            "Q1_LOW",
            "Q2_LOW_MODERATE",
            "Q3_MODERATE",
            "Q4_HIGH",
            "Q5_HIGHEST"
        ]
    )

    tier_summary = (
        df.groupby("lab_signal_tier", observed=True)
        .agg(
            patient_month_count=("member_id", "count"),
            unique_members=("member_id", "nunique"),
            hospitalization_rate=(TARGET_LABEL_NAME, "mean"),
            avg_lab_composite_score=("lab_diagnosis_composite_score", "mean"),
            avg_lab_escalation_score=("lab_escalation_score", "mean"),
            avg_lab_monitoring_intensity=("lab_monitoring_intensity_score", "mean"),
            avg_lab_workup_score=("lab_workup_pattern_score", "mean"),
            complex_deterioration_rate=("lab_complex_deterioration_flag", "mean"),
            infection_pathway_rate=("lab_infection_plus_pathway_flag", "mean"),
            renal_chronic_rate=("lab_renal_plus_chronic_flag", "mean")
        )
        .reset_index()
    )

    flag_summary_rows = []

    for flag in [
        "lab_complex_deterioration_flag",
        "lab_infection_plus_pathway_flag",
        "lab_renal_plus_chronic_flag"
    ]:
        temp = (
            df.groupby(flag)
            .agg(
                patient_month_count=("member_id", "count"),
                unique_members=("member_id", "nunique"),
                hospitalization_rate=(TARGET_LABEL_NAME, "mean"),
                avg_lab_composite_score=("lab_diagnosis_composite_score", "mean")
            )
            .reset_index()
        )

        temp["flag_name"] = flag
        temp = temp.rename(columns={flag: "flag_value"})
        flag_summary_rows.append(temp)

    flag_summary = pd.concat(flag_summary_rows, ignore_index=True)

    print(f"\n{dataset_label} lab signal tier summary:")
    print(tier_summary)

    print(f"\n{dataset_label} lab flag summary:")
    print(flag_summary)

    return tier_summary, flag_summary


train_lab_tier_summary, train_lab_flag_summary = build_lab_validation_summary(
    df_train_timeline,
    "TRAIN Sets 1+2"
)

test_lab_tier_summary, test_lab_flag_summary = build_lab_validation_summary(
    df_test_timeline,
    "TEST Set 3"
)

train_lab_tier_summary.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL5_train_lab_signal_tier_summary.csv"
    ),
    index=False
)

test_lab_tier_summary.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL5_test_lab_signal_tier_summary.csv"
    ),
    index=False
)

train_lab_flag_summary.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL5_train_lab_flag_summary.csv"
    ),
    index=False
)

test_lab_flag_summary.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL5_test_lab_flag_summary.csv"
    ),
    index=False
)

print("\nStep L5 completed.")

In [ ]:
# ============================================================
# HCPCS Crosswalk Cleanup Before L6
# ============================================================

import pandas as pd
import numpy as np

df_hcpcs["HCPCS Code"] = (
    df_hcpcs["HCPCS Code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Standardize text columns
for col in ["Category", "Subcategory", "Clinical Domain", "Category_ID", "Relevance", "HCPCS_PREFIX"]:
    if col not in df_hcpcs.columns:
        df_hcpcs[col] = np.nan

# Fill missing Category from Subcategory logic
def infer_category(row):
    cat = row["Category"]
    sub = str(row["Subcategory"]).upper().strip()
    code = str(row["HCPCS Code"]).upper().strip()

    if pd.notna(cat) and str(cat).strip() != "":
        return str(cat).upper().strip()

    if sub in ["CHEMISTRY PANEL", "CHEMISTRY", "METABOLIC CHEMISTRY"]:
        return "LAB"
    if sub in ["HEMATOLOGY_CBC", "HEMATOLOGY", "CBC"]:
        return "LAB"
    if sub in ["URINALYSIS"]:
        return "LAB"
    if sub in ["MICROBIOLOGY", "MICROBIOLOGY_CULTURE"]:
        return "LAB"
    if sub in ["IMMUNOLOGY", "AUTOIMMUNE_PANEL"]:
        return "LAB"
    if sub in ["PATHOLOGY", "ANATOMIC_PATHOLOGY"]:
        return "LAB"
    if sub in ["CARDIAC_DETERIORATION"]:
        return "LAB"
    if sub in ["RENAL_MONITORING"]:
        return "LAB"
    if sub in ["BLEEDING_COAGULATION"]:
        return "LAB"

    if code.startswith("A95") or code.startswith("A96") or code.startswith("A97"):
        return "NUCLEAR_MEDICINE"
    if code.startswith("A42"):
        return "DIABETES_MONITORING"
    if code.startswith("0") and code.endswith(("U", "M")):
        return "MOLECULAR_DIAGNOSTICS"

    return "OTHER_DIAGNOSTIC"

df_hcpcs["Category"] = df_hcpcs.apply(infer_category, axis=1)

# Add lab family if missing
if "LAB_FAMILY" not in df_hcpcs.columns:
    df_hcpcs["LAB_FAMILY"] = np.nan

def infer_lab_family(row):
    fam = row.get("LAB_FAMILY", np.nan)
    if pd.notna(fam) and str(fam).strip() != "":
        return str(fam).upper().strip()

    code = str(row["HCPCS Code"]).upper().strip()
    sub = str(row["Subcategory"]).upper().strip()

    specific = {
        "80048": "BMP",
        "80053": "CMP",
        "80061": "LIPID_PANEL",
        "83036": "A1C",
        "85025": "CBC",
        "85027": "CBC",
        "85610": "INR",
        "85730": "PTT",
        "84484": "TROPONIN",
        "83880": "BNP",
        "87086": "URINE_CULTURE",
        "81001": "URINALYSIS",
        "81003": "URINALYSIS"
    }

    if code in specific:
        return specific[code]

    if "CHEMISTRY" in sub:
        return "CHEMISTRY"
    if "HEMATOLOGY" in sub or "CBC" in sub:
        return "CBC"
    if "MICROBIOLOGY" in sub:
        return "MICROBIOLOGY"
    if "IMMUNOLOGY" in sub:
        return "IMMUNOLOGY"
    if "PATHOLOGY" in sub:
        return "PATHOLOGY"
    if "CARDIAC" in sub:
        return "CARDIAC_BIOMARKER"
    if "COAG" in sub:
        return "COAGULATION"

    return "GENERAL"

df_hcpcs["LAB_FAMILY"] = df_hcpcs.apply(infer_lab_family, axis=1)

# Add acuity if missing
if "ACUITY_LEVEL" not in df_hcpcs.columns:
    df_hcpcs["ACUITY_LEVEL"] = np.nan

def infer_acuity(row):
    acuity = row.get("ACUITY_LEVEL", np.nan)
    if pd.notna(acuity) and str(acuity).strip() != "":
        return str(acuity).upper().strip()

    code = str(row["HCPCS Code"]).upper().strip()
    sub = str(row["Subcategory"]).upper().strip()

    if code in ["84484", "83880", "83605", "87040", "85610", "85730", "85379"]:
        return "HIGH"
    if "CARDIAC" in sub or "COAG" in sub or "MICROBIOLOGY" in sub:
        return "HIGH"
    if "CHEMISTRY" in sub or "HEMATOLOGY" in sub or "CBC" in sub:
        return "MEDIUM"

    return "LOW"

df_hcpcs["ACUITY_LEVEL"] = df_hcpcs.apply(infer_acuity, axis=1)

# Add care pathway if missing
if "CARE_PATHWAY" not in df_hcpcs.columns:
    df_hcpcs["CARE_PATHWAY"] = np.nan

def infer_care_pathway(row):
    cp = row.get("CARE_PATHWAY", np.nan)
    if pd.notna(cp) and str(cp).strip() != "":
        return str(cp).upper().strip()

    code = str(row["HCPCS Code"]).upper().strip()
    sub = str(row["Subcategory"]).upper().strip()

    if code in ["84484", "83880", "83874"]:
        return "CARDIAC"
    if code in ["85610", "85730", "85379"]:
        return "ANTICOAGULATION"
    if code in ["87086", "87040", "87070", "87186"]:
        return "INFECTION"
    if code in ["80048", "80053", "82565", "82043"]:
        return "METABOLIC_RENAL"
    if code == "83036":
        return "DIABETES"
    if "PATHOLOGY" in sub:
        return "ONCOLOGY"
    if "MICROBIOLOGY" in sub:
        return "INFECTION"
    if "CARDIAC" in sub:
        return "CARDIAC"

    return "GENERAL"

df_hcpcs["CARE_PATHWAY"] = df_hcpcs.apply(infer_care_pathway, axis=1)

print("Crosswalk rows:", len(df_hcpcs))
print("\nCategory distribution after cleanup:")
print(df_hcpcs["Category"].value_counts().head(20))

print("\nSample cleaned codes:")
print(
    df_hcpcs[
        df_hcpcs["HCPCS Code"].isin(
            ["85025", "80053", "83036", "84484", "83880", "85610", "85730", "87086"]
        )
    ][
        ["HCPCS Code", "Description", "Category", "Subcategory", "LAB_FAMILY", "ACUITY_LEVEL", "CARE_PATHWAY"]
    ]
)

In [ ]:
# ============================================================
# Step L6 - HCPCS Clinical Dictionary Integration
# Uses HCPCS_List.xlsx to enrich lab events and evaluate lab-code risk
# ============================================================

print("\nSTEP L6 - HCPCS clinical dictionary integration")

# ------------------------------------------------------------
# Crosswalk file
# ------------------------------------------------------------

HCPCS_CROSSWALK_FILE = "/content/drive/MyDrive/Colab Notebooks/HCPCS_List.xlsx"

if not os.path.exists(HCPCS_CROSSWALK_FILE):
    raise FileNotFoundError(
        f"HCPCS crosswalk file not found: {HCPCS_CROSSWALK_FILE}"
    )

print("HCPCS crosswalk found:", HCPCS_CROSSWALK_FILE)


# ------------------------------------------------------------
# Load and standardize HCPCS crosswalk
# ------------------------------------------------------------

def standardize_hcpcs_crosswalk_columns(df):

    df = df.copy()

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    rename_candidates = {
        "hcpcs": "hcpcs_code",
        "hcpcs_code": "hcpcs_code",
        "code": "hcpcs_code",
        "description": "description",
        "category": "category",
        "subcategory": "subcategory",
        "clinical_domain": "clinical_domain",
        "hospitalization_relevance": "hospitalization_relevance",
        "hcpcs_prefix": "hcpcs_prefix"
    }

    rename_map = {}

    for col in df.columns:
        if col in rename_candidates:
            rename_map[col] = rename_candidates[col]

    df = df.rename(columns=rename_map)

    required = [
        "hcpcs_code",
        "description",
        "category",
        "subcategory",
        "clinical_domain"
    ]

    missing = [c for c in required if c not in df.columns]

    if len(missing) > 0:
        raise ValueError(f"Missing required HCPCS crosswalk columns: {missing}")

    df["hcpcs_code"] = df["hcpcs_code"].apply(clean_hcpcs)
    df["description"] = df["description"].fillna("").astype(str).str.strip()
    df["category"] = df["category"].fillna("UNKNOWN").astype(str).str.strip().str.upper()
    df["subcategory"] = df["subcategory"].fillna("UNKNOWN").astype(str).str.strip().str.upper()
    df["clinical_domain"] = df["clinical_domain"].fillna("UNKNOWN").astype(str).str.strip().str.upper()

    if "hospitalization_relevance" not in df.columns:
        df["hospitalization_relevance"] = "UNKNOWN"
    else:
        df["hospitalization_relevance"] = (
            df["hospitalization_relevance"]
            .fillna("UNKNOWN")
            .astype(str)
            .str.strip()
            .str.upper()
        )

    if "hcpcs_prefix" not in df.columns:
        df["hcpcs_prefix"] = df["hcpcs_code"].str[0]
    else:
        df["hcpcs_prefix"] = (
            df["hcpcs_prefix"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.upper()
        )

    df = df[df["hcpcs_code"] != ""].copy()

    df = df.drop_duplicates("hcpcs_code").reset_index(drop=True)

    return df


hcpcs_crosswalk_raw = pd.read_excel(
    HCPCS_CROSSWALK_FILE,
    dtype=str
)

hcpcs_crosswalk = standardize_hcpcs_crosswalk_columns(
    hcpcs_crosswalk_raw
)

print("\nHCPCS crosswalk loaded.")
print("Shape:", hcpcs_crosswalk.shape)
print("\nSubcategory distribution:")
print(hcpcs_crosswalk["subcategory"].value_counts(dropna=False))
print("\nClinical domain distribution:")
print(hcpcs_crosswalk["clinical_domain"].value_counts(dropna=False))


# ------------------------------------------------------------
# Enrich lab events with crosswalk
# ------------------------------------------------------------

def enrich_lab_events_with_crosswalk(lab_events, crosswalk, dataset_label):

    labs = lab_events.copy()

    labs["hcpcs_code"] = labs["hcpcs_code"].apply(clean_hcpcs)

    labs = labs.merge(
        crosswalk[
            [
                "hcpcs_code",
                "description",
                "category",
                "subcategory",
                "clinical_domain",
                "hospitalization_relevance",
                "hcpcs_prefix"
            ]
        ],
        on="hcpcs_code",
        how="left",
        suffixes=("", "_xwalk")
    )

    labs["crosswalk_match_flag"] = labs["subcategory"].notna().astype(int)

    labs["description"] = labs["description"].fillna("")
    labs["category"] = labs["category"].fillna("UNMAPPED")
    labs["subcategory"] = labs["subcategory"].fillna("UNMAPPED")
    labs["clinical_domain"] = labs["clinical_domain"].fillna("UNMAPPED")
    labs["hospitalization_relevance"] = labs["hospitalization_relevance"].fillna("UNMAPPED")
    labs["hcpcs_prefix"] = labs["hcpcs_prefix"].fillna(labs["hcpcs_code"].str[0])

    match_rate = labs["crosswalk_match_flag"].mean()

    print(f"\n{dataset_label} lab crosswalk enrichment complete.")
    print("Rows:", labs.shape[0])
    print("Crosswalk match rate:", round(match_rate, 4))

    print("\nMapped / unmapped distribution:")
    print(labs["crosswalk_match_flag"].value_counts(dropna=False))

    print("\nTop subcategories:")
    print(labs["subcategory"].value_counts(dropna=False).head(20))

    print("\nTop clinical domains:")
    print(labs["clinical_domain"].value_counts(dropna=False).head(20))

    return labs


lab_train_enriched = enrich_lab_events_with_crosswalk(
    lab_train_events,
    hcpcs_crosswalk,
    "TRAIN Sets 1+2"
)

lab_test_enriched = enrich_lab_events_with_crosswalk(
    lab_test_events,
    hcpcs_crosswalk,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Attach future hospitalization label to lab event month
# ------------------------------------------------------------

def attach_member_month_label_to_labs(lab_events_enriched, timeline, dataset_label):

    labs = lab_events_enriched.copy()
    df = timeline.copy()

    labs["lab_service_date"] = pd.to_datetime(
        labs["lab_service_date"],
        errors="coerce"
    )

    labs["lab_month_start"] = (
        labs["lab_service_date"]
        .dt.to_period("M")
        .dt.to_timestamp()
    )

    df[PREDICTION_DATE_COL] = pd.to_datetime(
        df[PREDICTION_DATE_COL],
        errors="coerce"
    )

    label_cols = [
        "member_id",
        PREDICTION_DATE_COL,
        TARGET_LABEL_NAME
    ]

    df_label = df[label_cols].drop_duplicates().copy()

    labs = labs.merge(
        df_label,
        left_on=["member_id", "lab_month_start"],
        right_on=["member_id", PREDICTION_DATE_COL],
        how="left"
    )

    labs[TARGET_LABEL_NAME] = safe_numeric(
        labs[TARGET_LABEL_NAME],
        fill_value=0
    )

    labs = labs.drop(columns=[PREDICTION_DATE_COL], errors="ignore")

    print(f"\n{dataset_label} labs with member-month labels:")
    print(labs.shape)
    print("Label mean:", labs[TARGET_LABEL_NAME].mean())

    return labs


lab_train_enriched_labeled = attach_member_month_label_to_labs(
    lab_train_enriched,
    df_train_timeline,
    "TRAIN Sets 1+2"
)

lab_test_enriched_labeled = attach_member_month_label_to_labs(
    lab_test_enriched,
    df_test_timeline,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Build HCPCS-level risk table
# ------------------------------------------------------------

def build_hcpcs_risk_table(labeled_labs, dataset_label):

    global_rate = labeled_labs[TARGET_LABEL_NAME].mean()

    risk = (
        labeled_labs
        .groupby(
            [
                "hcpcs_code",
                "description",
                "subcategory",
                "clinical_domain",
                "hospitalization_relevance",
                "crosswalk_match_flag"
            ],
            dropna=False
        )
        .agg(
            lab_event_count=("member_id", "count"),
            unique_members=("member_id", "nunique"),
            hospitalization_rate=(TARGET_LABEL_NAME, "mean")
        )
        .reset_index()
    )

    risk["lift_vs_lab_global"] = (
        risk["hospitalization_rate"] /
        max(global_rate, 0.000001)
    )

    risk = risk.sort_values(
        ["hospitalization_rate", "lab_event_count"],
        ascending=[False, False]
    )

    print(f"\n{dataset_label} HCPCS-level risk table:")
    print(risk.head(50))

    return risk


train_hcpcs_risk = build_hcpcs_risk_table(
    lab_train_enriched_labeled,
    "TRAIN Sets 1+2"
)

test_hcpcs_risk = build_hcpcs_risk_table(
    lab_test_enriched_labeled,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Build subcategory/domain risk tables
# ------------------------------------------------------------

def build_lab_group_risk_tables(labeled_labs, dataset_label):

    global_rate = labeled_labs[TARGET_LABEL_NAME].mean()

    subcategory_risk = (
        labeled_labs
        .groupby("subcategory", dropna=False)
        .agg(
            lab_event_count=("member_id", "count"),
            unique_members=("member_id", "nunique"),
            hospitalization_rate=(TARGET_LABEL_NAME, "mean"),
            unique_hcpcs_codes=("hcpcs_code", "nunique")
        )
        .reset_index()
    )

    subcategory_risk["lift_vs_lab_global"] = (
        subcategory_risk["hospitalization_rate"] /
        max(global_rate, 0.000001)
    )

    subcategory_risk = subcategory_risk.sort_values(
        ["hospitalization_rate", "lab_event_count"],
        ascending=[False, False]
    )

    domain_risk = (
        labeled_labs
        .groupby("clinical_domain", dropna=False)
        .agg(
            lab_event_count=("member_id", "count"),
            unique_members=("member_id", "nunique"),
            hospitalization_rate=(TARGET_LABEL_NAME, "mean"),
            unique_hcpcs_codes=("hcpcs_code", "nunique")
        )
        .reset_index()
    )

    domain_risk["lift_vs_lab_global"] = (
        domain_risk["hospitalization_rate"] /
        max(global_rate, 0.000001)
    )

    domain_risk = domain_risk.sort_values(
        ["hospitalization_rate", "lab_event_count"],
        ascending=[False, False]
    )

    print(f"\n{dataset_label} subcategory risk:")
    print(subcategory_risk)

    print(f"\n{dataset_label} clinical domain risk:")
    print(domain_risk)

    return subcategory_risk, domain_risk


train_lab_subcategory_risk, train_lab_domain_risk = build_lab_group_risk_tables(
    lab_train_enriched_labeled,
    "TRAIN Sets 1+2"
)

test_lab_subcategory_risk, test_lab_domain_risk = build_lab_group_risk_tables(
    lab_test_enriched_labeled,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Coverage / unmapped HCPCS analysis
# ------------------------------------------------------------

def build_unmapped_hcpcs_table(labeled_labs, dataset_label):

    unmapped = labeled_labs[
        labeled_labs["crosswalk_match_flag"] == 0
    ].copy()

    if len(unmapped) == 0:
        return pd.DataFrame()

    unmapped_summary = (
        unmapped
        .groupby("hcpcs_code")
        .agg(
            lab_event_count=("member_id", "count"),
            unique_members=("member_id", "nunique"),
            hospitalization_rate=(TARGET_LABEL_NAME, "mean")
        )
        .reset_index()
        .sort_values("lab_event_count", ascending=False)
    )

    print(f"\n{dataset_label} top unmapped HCPCS codes:")
    print(unmapped_summary.head(50))

    return unmapped_summary


train_unmapped_hcpcs = build_unmapped_hcpcs_table(
    lab_train_enriched_labeled,
    "TRAIN Sets 1+2"
)

test_unmapped_hcpcs = build_unmapped_hcpcs_table(
    lab_test_enriched_labeled,
    "TEST Set 3"
)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

hcpcs_crosswalk.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_hcpcs_crosswalk_standardized.csv"
    ),
    index=False
)

lab_train_enriched_labeled.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_train_lab_events_enriched_labeled.csv"
    ),
    index=False
)

lab_test_enriched_labeled.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_test_lab_events_enriched_labeled.csv"
    ),
    index=False
)

train_hcpcs_risk.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_train_hcpcs_risk.csv"
    ),
    index=False
)

test_hcpcs_risk.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_test_hcpcs_risk.csv"
    ),
    index=False
)

train_lab_subcategory_risk.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_train_lab_subcategory_risk.csv"
    ),
    index=False
)

test_lab_subcategory_risk.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_test_lab_subcategory_risk.csv"
    ),
    index=False
)

train_lab_domain_risk.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_train_lab_domain_risk.csv"
    ),
    index=False
)

test_lab_domain_risk.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_test_lab_domain_risk.csv"
    ),
    index=False
)

train_unmapped_hcpcs.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_train_unmapped_hcpcs.csv"
    ),
    index=False
)

test_unmapped_hcpcs.to_csv(
    os.path.join(
        LAB_OUTPUT_FOLDER,
        "stepL6_test_unmapped_hcpcs.csv"
    ),
    index=False
)

print("\nStep L6 completed.")
print("Saved HCPCS crosswalk integration and risk outputs.")

In [ ]:
# Step L6(V)

df_train_timeline.to_pickle(
    os.path.join(LAB_OUTPUT_FOLDER, "checkpoint_after_L6_train_timeline.pkl")
)

df_test_timeline.to_pickle(
    os.path.join(LAB_OUTPUT_FOLDER, "checkpoint_after_L6_test_timeline.pkl")
)

print("Fix #2 lab checkpoint saved.")